# Pipeline OCR/VLM — Domiciliation des salaires étrangers

**Modèle : Qwen3.6-VL-27B-FP8 sur GPU Domino — version V7.3**

La V7.3 poursuit un seul objectif : **la qualité de la donnée extraite et
la fiabilité de son format de sortie**. Les gains de temps qu'elle apporte
sont un effet de bord du batching, pas un compromis sur l'exactitude.

## 1. Lecture du permis de travail — renforcée, pas allégée

Les versions antérieures à la 7.2 ne parvenaient pas à lire la planche
« permis de travail ». La V7.2 avait ajouté une escalade en cascade de
quatre recadrages avec arrêt anticipé — ce qui posait deux problèmes :
la cascade pouvait **sauter la vue portant le champ manquant**, et le
plafond `MAX_PIXELS` du processor réduisait silencieusement les
recadrages « haute définition ».

La V7.3 corrige les deux :

- **Vues systématiques, en un seul appel GPU.** Entête (numéro de permis),
  colonne identité, colonne poste et planche entière sont toutes lues,
  puis fusionnées. Plus d'arrêt anticipé.
- **Prompts spécialisés par vue.** Chaque vue ne demande que ses propres
  champs, au lieu des 21 champs du prompt global : moins d'invitation à
  combler ce qui n'est pas visible, et trois fois moins de tokens.
- **Vue « entête » dédiée** au numéro de permis, champ le plus critique
  et le plus souvent mal lu.
- **Résolution réellement augmentée** : zoom 5 au lieu de 4, côté maximum
  2400 px au lieu de 1800, et un **second processor** (`processor_hd`)
  au plafond de pixels relevé pour que ces images ne soient plus rabotées.
- **Fusion tracée** : pour chaque champ, la vue d'origine est enregistrée,
  et tout désaccord entre deux vues est signalé au lieu d'être masqué.

## 2. Typage et validation des valeurs

Chaque champ extrait produit désormais trois informations :

| Sortie | Contenu |
|---|---|
| `<CHAMP>_RAW` | la chaîne exactement telle que lue, jamais modifiée |
| `<CHAMP>` | la valeur typée : montant `float`, date ISO `AAAA-MM-JJ`, entier |
| `<CHAMP>_STATUT` | `OK` / `AMBIGU` / `ILLISIBLE` / `HORS_BORNES` / `ABSENT` |

- **Montants** : les trois conventions rencontrées (`506,471.38`,
  `506 471,38`, `466300.88`) sont lues à partir du nombre de chiffres
  suivant le dernier séparateur. Les cas réellement indécidables ne sont
  plus tranchés au hasard, ils sont marqués `AMBIGU`.
- **Dates** : parseur à motifs explicites, sortie ISO. `pd.to_datetime`
  acceptait `2026.1` comme une date ; ce n'est plus le cas.
- **Numéros de compte** : compactés, contrôlés en longueur, en code
  banque et par **clé de contrôle** — le seul test qui détecte une erreur
  sur un seul chiffre.

## 3. Contrôles croisés et arithmétiques

Le dossier est massivement redondant : le salaire net figure sur trois
documents, les dates sur quatre. La V7.3 exploite cette redondance et les
identités arithmétiques qui lient les montants :

- part transférable = salaire net × taux ;
- part payable en dinars = net − part transférable ;
- montant domicilié = part transférable × durée ;
- date de fin = date de début + durée − 1 jour ;
- validité du permis couvrant la période du contrat ;
- clé de contrôle du compte domiciliataire ;
- concordance de chaque donnée entre les documents qui la portent.

Chaque contrôle produit un verdict daté et chiffré, exporté dans les
onglets `CONTROLES` et `ANOMALIES`. Une erreur d'un seul chiffre sur la
part transférable déclenche trois contrôles simultanément.

## 4. Relecture ciblée au lieu d'escalade aveugle

Quand un contrôle bloquant échoue, le pipeline relit **le bloc précis mis
en cause**, avec un prompt réduit à ces seuls champs. La valeur relue ne
remplace la valeur initiale que si elle rétablit la cohérence ; sinon
l'anomalie reste visible. Les deux lectures sont conservées.

## 5. Performance — conséquence du batching, pas d'un compromis

- `ask_batch_multi` : un prompt différent par image, **un seul appel à
  `generate`**. La V7.2 déclarait `GPU_BATCH_SIZE_EXTRACTION` sans jamais
  l'utiliser et enchaînait 7 à 9 générations séquentielles par dossier.
- Rendu haute définition **mis en cache** : une page n'est plus rendue
  qu'une fois au lieu de quatre (jusqu'à 2,5 s par rendu sur un scan lourd).
- Classification en basse résolution, avec **reclassement systématique en
  pleine résolution** de toute page classée `AUTRE` ou
  `PERMIS_TRAVAIL_COUVERTURE`. On n'économise jamais sur la planche.
- `attn_implementation="sdpa"` — disponible nativement avec torch, sans
  installer flash-attn. Repli automatique sur `eager` si refusé.
- Lecture JSON par comptage d'accolades, avec réparation des réponses
  tronquées : une troncature ne provoque plus une escalade inutile.
- `max_new_tokens` d'extraction ramené de 1700 à 900.

Le pipeline **n'effectue toujours pas les contrôles réglementaires
finaux**. Les règles métier et les décisions restent dans Alteryx.


## 1. Dépendances

In [ ]:

# Installation du runtime minimal compatible Qwen3.6-VL-FP8 (cf. bilans V13c)
# %pip install -q -U 'transformers>=4.57.0' accelerate

import sys
from importlib import metadata

REQUIRED_PACKAGES = {
    "torch": "2.0",
    "transformers": "4.57",
    "accelerate": "0.30",
    "PyMuPDF": "1.23",
    "Pillow": "9.0",
    "openpyxl": "3.1",
    "pandas": "1.5",
    "psutil": "5.9",
}

print("Python :", sys.version.replace("\n", " "))
print("\nPackages détectés :")
missing = []
for package_name, minimum in REQUIRED_PACKAGES.items():
    try:
        version = metadata.version(package_name)
        print(f"  {package_name:15s} {version:12s} | minimum conseillé {minimum}")
    except metadata.PackageNotFoundError:
        missing.append(package_name)
        print(f"  {package_name:15s} ABSENT")

if missing:
    raise RuntimeError(
        "Packages manquants : " + ", ".join(missing) +
        ". Installer uniquement ces packages dans l'environnement Domino."
    )

print("\n✅ Vérification des packages terminée sans modification de l'environnement")


## 2. Imports

In [ ]:

import gc
import hashlib
import json
import math
import re
import sys
import time
import calendar
from collections import defaultdict
from datetime import date, datetime, timedelta
from pathlib import Path

import fitz
import numpy as np
import pandas as pd
import psutil
import torch
from PIL import Image
from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
from transformers import AutoProcessor, AutoModelForImageTextToText

print("✅ Imports OK")
print("Python       :", sys.version.split()[0])
print("PyMuPDF     :", fitz.__doc__.splitlines()[0] if fitz.__doc__ else "chargé")
print("Torch       :", torch.__version__)
print("CUDA dispo  :", torch.cuda.is_available())
print("GPU         :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Aucun")


## 3. Configuration

In [ ]:
MODEL_PATH = '/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.6-27B-FP8/main'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# =====================================================================
# V7.3 — Rendu : trois niveaux au lieu de deux
# =====================================================================
# 1. CLASSIFICATION : la reconnaissance d'un titre et d'une mise en page
#    n'a pas besoin de 1400 px. On réutilise l'image standard, réduite en
#    mémoire : aucun rendu PDF supplémentaire, ~4x moins de tokens image.
IMAGE_MAX_SIZE_CLASSIFICATION = 900
CLASSIFICATION_BASSE_RESOLUTION = True
# Filet de sécurité : toute page classée AUTRE ou PERMIS_TRAVAIL_COUVERTURE
# est reclassée en pleine résolution. On n'économise jamais sur la planche.
RECLASSEMENT_PLEINE_RESOLUTION = True

# 2. STANDARD : documents dactylographiés (engagement, contrats).
PDF_ZOOM = 2.0
IMAGE_MAX_SIZE = 1400

# 3. HAUTE DÉFINITION : planche « permis de travail ».
#    V7.3 monte le zoom de 4 à 5 et le côté max de 1800 à 2400 px.
#    C'est possible sans exploser le temps parce que la page n'est plus
#    rendue qu'une seule fois (cache), et sans être raboté par le
#    processor parce qu'un second processor lui est dédié (MAX_PIXELS_HD).
PDF_ZOOM_HAUTE_DEF = 5
IMAGE_MAX_SIZE_HAUTE_DEF = 2400

MIN_PIXELS = 4 * 32 * 32
# Plafond du processor standard.
MAX_PIXELS = 1800 * 32 * 32
# Plafond du processor dédié à la planche permis.
# V7.2 : MAX_PIXELS = 2200*32*32 = 2,25 Mpx, alors qu'un recadrage HD
# atteint 2,6 Mpx. Le processor le réduisait donc en silence : la "haute
# définition" n'en était pas une. Ce plafond corrige le problème.
MAX_PIXELS_HD = 4200 * 32 * 32

BLANK_THRESHOLD = 0.985          # V7.3 : les pages blanches sont désormais
IGNORER_PAGES_BLANCHES = True    # réellement écartées (is_blank était inutilisé)

GPU_BATCH_SIZE_CLASSIFICATION = 16
GPU_BATCH_SIZE_EXTRACTION = 6    # V7.3 : enfin utilisé (V7.2 le déclarait sans l'appeler)
GPU_BATCH_SIZE_VUES = 4          # vues de la planche, envoyées en un seul generate

MAX_NEW_TOKENS_CLASSIFICATION = 100
MAX_NEW_TOKENS_EXTRACTION = 900  # V7.2 : 1700, soit ~3x la taille réelle des JSON.
                                 # Une troncature produit un JSON invalide -> {} -> escalade inutile.
MAX_NEW_TOKENS_VUE = 500         # vues spécialisées : moins de champs demandés
MAX_NEW_TOKENS_RELECTURE = 400   # relecture ciblée d'un bloc de montants

# =====================================================================
# V7.3 — Stratégie de lecture de la planche permis
# =====================================================================
TYPES_HAUTE_DEFINITION = {"TITRE_TRAVAIL", "PERMIS_TRAVAIL_COUVERTURE"}
# Les vues de la planche ne sont plus une escalade conditionnelle : elles
# sont systématiques et envoyées en un seul batch. Une cascade avec arrêt
# anticipé pouvait sauter la vue qui portait le champ manquant.
LECTURE_PLANCHE_MULTI_VUES = True
SEUIL_REMPLISSAGE_OK = 0.55      # conservé pour les types dactylographiés

# =====================================================================
# V7.3 — Qualité et typage des données de sortie
# =====================================================================
# Tolérance sur les contrôles arithmétiques, en dinars.
TOLERANCE_MONTANT_DZD = 0.05
# Bornes de plausibilité (salaires expatriés en DZD).
MONTANT_MIN_PLAUSIBLE = 0.0
MONTANT_MAX_PLAUSIBLE = 1e11
ANNEE_MIN_PLAUSIBLE = 1950
ANNEE_MAX_PLAUSIBLE = 2100
# Format canonique des dates en sortie JSON.
FORMAT_DATE_SORTIE = "%Y-%m-%d"

# Contrôle de la clé RIB du compte domiciliataire.
# ATTENTION : la constante a été calibrée sur les comptes BNPPA observés
# (027 00731 ...), pour lesquels cle = 98 - ((89*banque + 15*agence +
# 3*compte) mod 97). Lancer calibrer_cle_rib() sur un échantillon de
# comptes certains AVANT de traiter un lot : si le taux de réussite n'est
# pas de 100 %, laisser CONTROLE_CLE_RIB = False.
CONTROLE_CLE_RIB = True
CLE_RIB_CONSTANTE = 98
LONGUEUR_COMPTE_ATTENDUE = 20
CODE_BANQUE_BNPPA = "027"

# Relecture ciblée : si un contrôle arithmétique échoue, le bloc de
# montants concerné est relu une fois, en haute définition recadrée.
# C'est une escalade motivée par une anomalie, pas une escalade aveugle.
RELECTURE_CIBLEE_ACTIVE = True
MAX_RELECTURES_PAR_DOSSIER = 2

INPUT_DIR = Path('/mnt/data/domiciliations_in')
OUTPUT_DIR = Path('/mnt/data/domiciliations_out')
JSON_DIR = OUTPUT_DIR / 'json_dossiers'
LOG_PATH = OUTPUT_DIR / 'pipeline_domiciliations.log'
EXCEL_PATH = OUTPUT_DIR / f"domiciliations_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"
MASTER_JSON_PATH = OUTPUT_DIR / 'domiciliations_master.json'
PRORATA_MODE = 'CALENDAR_DAYS'
GENERER_MOIS_COMPLETS = True
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JSON_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DIR.mkdir(parents=True, exist_ok=True)
pdfs = sorted(INPUT_DIR.glob('*.pdf'))
CLASSIFICATION_THRESHOLD = 0.80

PIPELINE_VERSION = "DOM_V7_3_QWEN3_6_VL_27B_FP8"
DOM_REFERENCE_EXCEL = Path("/mnt/data/fichier_domiciliations.xlsx")
PREDOM_REFERENCE_EXCEL = Path("/mnt/data/fichier_predomiciliations.xlsx")
NAME_MATCH_THRESHOLD = 0.86
DATE_TOLERANCE_DAYS = 5
EXCEL_AMOUNT_FORMAT = "0.00"
AMOUNT_FIELDS = {
    "DOM_SALAIRE_NET_MENSUEL", "DOM_PART_TRANSFERABLE",
    "DOM_MONTANT_TOTAL_DOMICILIE", "CTR_SALAIRE_BRUT",
    "CTR_SALAIRE_NET", "CTS_SALAIRE_NET",
    "CTS_SALAIRE_NET_ANCIEN",
    "CTS_PART_TRANSFERABLE", "CTS_PART_PAYABLE_DZD",
}

print(f'Device            : {DEVICE}')
print(f'PDFs détectés     : {len(pdfs)}')
print(f'Entrée            : {INPUT_DIR}')
print(f'Sortie            : {OUTPUT_DIR}')
print(f'Version pipeline  : {PIPELINE_VERSION}')
print(f'Classification    : {"basse résolution + reclassement" if CLASSIFICATION_BASSE_RESOLUTION else "pleine résolution"}')
print(f'Planche permis    : zoom {PDF_ZOOM_HAUTE_DEF}, max {IMAGE_MAX_SIZE_HAUTE_DEF}px, plafond {MAX_PIXELS_HD // 1024} blocs')


## 4. Chargement du modèle Qwen3.6-VL-27B-FP8

In [ ]:
if DEVICE != "cuda":
    raise RuntimeError("Ce pipeline nécessite un GPU CUDA.")

torch.backends.cuda.matmul.allow_tf32 = True

# ---------------------------------------------------------------------
# V7.3 — Implémentation d'attention
# ---------------------------------------------------------------------
# flash-attn n'est pas installable sur cet environnement Domino.
# "sdpa" est fourni nativement par torch (>= 2.0), sans aucune
# installation, et couvre l'essentiel du gain sur nos longueurs de
# séquence. On ne bascule en "eager" que s'il n'est pas disponible.
ATTN_IMPLEMENTATION = "sdpa"

# FP8 : dequantize=True fait tourner le modèle en bfloat16 et annule le
# bénéfice des kernels FP8 sur H100. Passer à False pour tester le FP8
# natif, après validation de l'exactitude sur un échantillon connu.
FP8_DEQUANTIZE = True

print("Chargement des processors...")
t0 = time.time()

def _charger_processor(max_pixels):
    proc = AutoProcessor.from_pretrained(
        MODEL_PATH,
        trust_remote_code=True,
        min_pixels=MIN_PIXELS,
        max_pixels=max_pixels,
    )
    proc.tokenizer.padding_side = "left"
    return proc

# Processor standard : pages dactylographiées et classification.
processor = _charger_processor(MAX_PIXELS)

# Processor dédié à la planche « permis de travail ».
# Même modèle, même tokenizer : seul le plafond de pixels change, pour
# que les recadrages haute définition ne soient plus réduits en silence.
processor_hd = _charger_processor(MAX_PIXELS_HD)

print("Chargement du modèle FP8...")
try:
    from transformers.integrations.finegrained_fp8 import FineGrainedFP8Config as FP8Config
except ImportError:
    from transformers import FineGrainedFP8Config as FP8Config

try:
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_PATH,
        dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        attn_implementation=ATTN_IMPLEMENTATION,
        quantization_config=FP8Config(dequantize=FP8_DEQUANTIZE),
    )
except (ValueError, ImportError) as exc:
    print(f"⚠️ attn_implementation={ATTN_IMPLEMENTATION} refusé ({exc}) — repli sur eager")
    ATTN_IMPLEMENTATION = "eager"
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_PATH,
        dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        attn_implementation=ATTN_IMPLEMENTATION,
        quantization_config=FP8Config(dequantize=FP8_DEQUANTIZE),
    )

model.eval()

print(f"✅ Modèle chargé en {time.time() - t0:.1f}s")
print(f"   Attention        : {getattr(model.config, '_attn_implementation', ATTN_IMPLEMENTATION)}")
print(f"   FP8 déquantifié  : {FP8_DEQUANTIZE}")
print(f"   VRAM allouée     : {torch.cuda.memory_allocated() / 1e9:.2f} GB")


## 5. Utilitaires PDF, image et JSON

In [ ]:
def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def resize_image(img, max_side=IMAGE_MAX_SIZE):
    w, h = img.size
    if max(w, h) <= max_side:
        return img
    ratio = max_side / max(w, h)
    return img.resize((int(w * ratio), int(h * ratio)), Image.LANCZOS)


def white_ratio(image):
    arr = np.array(image.convert("L"))
    return float((arr > 245).sum() / arr.size)


def is_blank(image, threshold=BLANK_THRESHOLD):
    return white_ratio(image) >= threshold


def pdf_to_pages(path, zoom=PDF_ZOOM):
    """
    Rend chaque page une seule fois en résolution standard.

    V7.3 : l'image de classification est dérivée en mémoire de l'image
    standard (pas de second rendu PDF) et le drapeau `blanche` est
    calculé ici pour que les pages vides ne partent pas au GPU.
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"PDF introuvable : {path}")
    if path.stat().st_size == 0:
        raise ValueError(f"PDF vide : {path}")

    pages = []
    doc = fitz.open(str(path))
    try:
        page_count = int(doc.page_count)
        if page_count <= 0:
            raise ValueError(f"PyMuPDF ne détecte aucune page dans : {path.name}")

        matrix = fitz.Matrix(zoom, zoom)
        for i in range(page_count):
            page = doc.load_page(i)
            pix = page.get_pixmap(matrix=matrix, alpha=False)

            if pix.width <= 0 or pix.height <= 0 or not pix.samples:
                raise ValueError(f"Rendu image vide : {path.name}, page {i + 1}")

            img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
            img = resize_image(img)
            blanc = white_ratio(img)

            pages.append({
                "index": i,
                "page_num": i + 1,
                "image": img,
                "image_classification": (
                    resize_image(img, IMAGE_MAX_SIZE_CLASSIFICATION)
                    if CLASSIFICATION_BASSE_RESOLUTION else img
                ),
                "width": img.width,
                "height": img.height,
                "white_ratio": round(blanc, 6),
                "blanche": bool(IGNORER_PAGES_BLANCHES and blanc >= BLANK_THRESHOLD),
            })
    finally:
        doc.close()

    if len(pages) != page_count:
        raise RuntimeError(
            f"Conversion incomplète de {path.name}: "
            f"{len(pages)} image(s) pour {page_count} page(s)"
        )
    return pages


# ---------------------------------------------------------------------
# V7.3 — Lecture JSON robuste
# ---------------------------------------------------------------------
# V7.2 utilisait re.search(r"\{.*\}") en mode glouton : deux objets JSON
# dans la réponse, ou du texte contenant une accolade, et la capture
# allait de la première à la dernière accolade -> parse impossible ->
# dictionnaire vide -> escalade coûteuse déclenchée pour rien.
# V7.3 parcourt la chaîne en comptant les accolades (en ignorant celles
# situées dans une chaîne de caractères) et sait réparer un JSON tronqué
# par max_new_tokens.

def _extraire_objets_json(texte):
    """Retourne (objets, tronque) — tronque=True si le dernier objet a dû
    être refermé artificiellement (réponse coupée par max_new_tokens)."""
    objets, profondeur, debut = [], 0, None
    tronque = False
    dans_chaine, echappe = False, False
    for i, c in enumerate(texte):
        if dans_chaine:
            if echappe:
                echappe = False
            elif c == "\\":
                echappe = True
            elif c == '"':
                dans_chaine = False
            continue
        if c == '"':
            dans_chaine = True
        elif c == "{":
            if profondeur == 0:
                debut = i
            profondeur += 1
        elif c == "}":
            if profondeur > 0:
                profondeur -= 1
                if profondeur == 0 and debut is not None:
                    objets.append(texte[debut:i + 1])
                    debut = None
    # Objet non refermé : réponse tronquée par max_new_tokens.
    # On referme la chaîne et les accolades manquantes pour récupérer les
    # champs déjà écrits, plutôt que de tout perdre.
    if profondeur > 0 and debut is not None:
        partiel = texte[debut:]
        if dans_chaine:
            partiel += '"'
        partiel = re.sub(r",\s*$", "", partiel.rstrip())
        objets.append(partiel + "}" * profondeur)
        tronque = True
    return objets, tronque


def parse_json_response(text, tracer=False):
    """
    Retourne le dictionnaire extrait de la réponse du modèle.

    tracer=True renvoie (dict, statut) où statut vaut :
      OK, REPARE_VIRGULE, REPARE_TRONQUE, FUSION_MULTI_OBJETS, VIDE.
    """
    vide = ({}, "VIDE") if tracer else {}
    if not text:
        return vide

    clean = str(text).strip()
    clean = re.sub(r"^```(?:json)?", "", clean, flags=re.I).strip()
    clean = re.sub(r"```$", "", clean).strip()

    objets, tronque = _extraire_objets_json(clean)
    if not objets:
        return vide

    fusion, statut = {}, ("REPARE_TRONQUE" if tronque else "OK")
    for brut in objets:
        parsed, essai_statut = None, "OK"
        for tentative, nom in (
            (brut, "OK"),
            (re.sub(r",\s*([}\]])", r"\1", brut), "REPARE_VIRGULE"),
            (re.sub(r",\s*$", "", brut.rstrip("}")) + "}", "REPARE_TRONQUE"),
        ):
            try:
                candidat = json.loads(tentative)
                if isinstance(candidat, dict):
                    parsed, essai_statut = candidat, nom
                    break
            except Exception:
                continue
        if parsed is None:
            continue
        if essai_statut != "OK":
            statut = essai_statut
        for cle, valeur in parsed.items():
            if fusion.get(cle) in (None, "") and valeur not in (None, ""):
                fusion[cle] = valeur

    if len(objets) > 1 and statut == "OK":
        statut = "FUSION_MULTI_OBJETS"
    if not fusion:
        return vide
    return (fusion, statut) if tracer else fusion


def log(message):
    line = f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - {message}"
    print(line)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")


def format_bytes(n):
    return f"{n / 1e6:.1f} Mo"


print("✅ Utilitaires PDF/JSON V7.3 OK (lecture JSON par comptage d'accolades)")


# ---------------------------------------------------------------------
# V7.3 — Rendu haute définition mis en cache
# ---------------------------------------------------------------------
# V7.2 rouvrait le PDF et re-rendait la page à CHAQUE recadrage : mesuré
# jusqu'à 2,5 s par rendu sur un scan lourd, soit ~7 s de CPU par dossier
# pendant lesquelles le GPU attend. V7.3 rend la page une fois, garde le
# pixmap en mémoire et découpe dedans. Le recadrage reste appliqué AVANT
# le redimensionnement : la zone utile conserve toute sa résolution.

_CACHE_HD = {}


def render_page_full_hd(pdf_path, page_index, zoom=PDF_ZOOM_HAUTE_DEF):
    cle = (str(pdf_path), int(page_index), float(zoom))
    if cle in _CACHE_HD:
        return _CACHE_HD[cle]

    doc = fitz.open(str(pdf_path))
    try:
        page = doc.load_page(int(page_index))
        pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom), alpha=False)
        if pix.width <= 0 or pix.height <= 0 or not pix.samples:
            raise ValueError(
                f"Rendu haute définition vide : {Path(pdf_path).name}, "
                f"page {int(page_index) + 1}"
            )
        img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
    finally:
        doc.close()

    _CACHE_HD[cle] = img
    return img


def vider_cache_hd():
    _CACHE_HD.clear()


def crop_region(image, haut=0.0, bas=1.0, gauche=0.0, droite=1.0):
    """Recadre une image par fractions de sa hauteur et de sa largeur."""
    largeur, hauteur = image.size
    x0 = int(max(0.0, min(1.0, gauche)) * largeur)
    x1 = int(max(0.0, min(1.0, droite)) * largeur)
    y0 = int(max(0.0, min(1.0, haut)) * hauteur)
    y1 = int(max(0.0, min(1.0, bas)) * hauteur)
    if x1 <= x0 or y1 <= y0:
        return image
    return image.crop((x0, y0, x1, y1))


def render_page_region(pdf_path, page_index, zoom=PDF_ZOOM_HAUTE_DEF,
                       max_side=IMAGE_MAX_SIZE_HAUTE_DEF, crop=None):
    """Recadrage haute définition, servi depuis le cache de rendu."""
    img = render_page_full_hd(pdf_path, page_index, zoom=zoom)
    if crop:
        img = crop_region(img, *crop)
    return resize_image(img, max_side=max_side)


def taux_remplissage(data, champs_attendus):
    if not champs_attendus:
        return 1.0
    remplis = sum(
        1 for champ in champs_attendus
        if (data or {}).get(champ) not in (None, "")
    )
    return round(remplis / len(champs_attendus), 4)


print("✅ Rendu haute définition mis en cache OK (1 rendu par page au lieu de 4)")


# ---------------------------------------------------------------------
# Détection automatique de la frontière sur la planche permis
# ---------------------------------------------------------------------
FRONTIERE_ZONE_RECHERCHE = (0.28, 0.66)
FRONTIERE_SEUIL_ENCRE = 0.004
FRONTIERE_DEFAUT = 0.47


def detecter_frontiere_documents(image, zone=FRONTIERE_ZONE_RECHERCHE,
                                 seuil_encre=FRONTIERE_SEUIL_ENCRE,
                                 defaut=FRONTIERE_DEFAUT):
    """Fraction de hauteur séparant les deux documents d'une planche."""
    try:
        arr = np.array(image.convert("L"))
    except Exception:
        return defaut

    hauteur = arr.shape[0]
    if hauteur < 10:
        return defaut

    densite_encre = (arr < 200).sum(axis=1) / max(arr.shape[1], 1)
    y_min = int(max(0.0, zone[0]) * hauteur)
    y_max = int(min(1.0, zone[1]) * hauteur)
    if y_max <= y_min:
        return defaut

    bandes, debut = [], None
    for y in range(y_min, y_max):
        vide = densite_encre[y] < seuil_encre
        if vide and debut is None:
            debut = y
        elif not vide and debut is not None:
            bandes.append((debut, y))
            debut = None
    if debut is not None:
        bandes.append((debut, y_max))

    if not bandes:
        return defaut

    debut, fin = max(bandes, key=lambda b: b[1] - b[0])
    if (fin - debut) / hauteur < 0.015:
        return defaut
    return round((debut + fin) / 2 / hauteur, 4)


def crops_planche_permis(image, marge=0.02):
    """
    Recadrages de la planche, construits sur la frontière détectée.

    V7.3 ajoute `entete` : le cadre en haut à gauche qui porte le numéro
    de permis. C'est le champ le plus critique de la page et le plus
    souvent mal lu ; il mérite sa propre vue, très serrée.
    """
    frontiere = detecter_frontiere_documents(image)
    bas_titre = min(1.0, frontiere + marge)
    haut_couverture = max(0.0, frontiere - marge)

    return {
        "frontiere": frontiere,
        "entete": (0.00, min(0.22, bas_titre), 0.00, 0.62),
        "titre": (0.00, bas_titre, 0.00, 1.00),
        "colonne_identite": (0.00, bas_titre, 0.44, 1.00),
        "colonne_poste": (0.00, bas_titre, 0.00, 0.56),
        "couverture": (haut_couverture, 1.00, 0.00, 1.00),
    }


print("✅ Détection de frontière + vue entête (numéro de permis) OK")


In [ ]:

def canonical_checkpoint_path(pdf_path):
    """
    Un seul fichier JSON par PDF :
    <nom_du_pdf_sans_extension>.json
    """
    return JSON_DIR / f"{pdf_path.stem}.json"


def checkpoint_is_complete(dossier, pdf_path):
    """
    Un checkpoint est réutilisable seulement s'il correspond au PDF
    et contient une extraction complète avec au moins une page.
    """
    if not isinstance(dossier, dict):
        return False

    stats = dossier.get("stats") or {}
    page_records = dossier.get("page_records") or []

    if dossier.get("source_file") != pdf_path.name:
        return False
    if int(stats.get("pages", 0) or 0) <= 0:
        return False
    if not page_records:
        return False

    # Vérification forte par empreinte SHA-256.
    stored_hash = dossier.get("source_sha256")
    if not stored_hash:
        return False

    try:
        return stored_hash == sha256_file(pdf_path)
    except Exception:
        return False


def load_existing_checkpoint(pdf_path):
    """
    Cherche d'abord le JSON canonique, puis les anciens JSON suffixés
    par l'empreinte. Si un ancien checkpoint valide est trouvé, il est
    migré vers le nom canonique afin de conserver un seul JSON par PDF.
    """
    canonical = canonical_checkpoint_path(pdf_path)
    candidates = [canonical] + sorted(
        JSON_DIR.glob(f"{pdf_path.stem}__*.json"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )

    seen = set()
    for candidate in candidates:
        if candidate in seen or not candidate.exists():
            continue
        seen.add(candidate)

        try:
            dossier = json.loads(candidate.read_text(encoding="utf-8"))
        except Exception:
            continue

        if not checkpoint_is_complete(dossier, pdf_path):
            continue

        # Migration vers un seul JSON canonique.
        if candidate != canonical:
            canonical.write_text(
                json.dumps(dossier, ensure_ascii=False, indent=2, default=str),
                encoding="utf-8",
            )

        # Suppression des anciens doublons JSON du même PDF.
        for duplicate in JSON_DIR.glob(f"{pdf_path.stem}__*.json"):
            if duplicate.exists():
                duplicate.unlink()

        return dossier

    return None


def enrich_dossier_row_with_stats(dossier, statut_traitement):
    """
    Ajoute au niveau dossier les indicateurs visibles dans Excel.
    """
    row = dossier.get("dossier_row") or {}
    stats = dossier.get("stats") or {}

    row.update({
        "STATUT_TRAITEMENT_PIPELINE": statut_traitement,
        "TEMPS_ECOULE_DOSSIER_S": round(float(stats.get("elapsed_s", 0) or 0), 2),
        "TOKENS_IN_DOSSIER": int(stats.get("tokens_in", 0) or 0),
        "TOKENS_OUT_DOSSIER": int(stats.get("tokens_out", 0) or 0),
        "TOKENS_TOTAL_DOSSIER": int(stats.get("tokens_total", 0) or 0),
    })

    dossier["dossier_row"] = row
    return dossier


In [ ]:

def format_duration(seconds):
    seconds = max(0, int(round(float(seconds or 0))))
    hours, rem = divmod(seconds, 3600)
    minutes, secs = divmod(rem, 60)
    if hours:
        return f"{hours:02d}:{minutes:02d}:{secs:02d}"
    return f"{minutes:02d}:{secs:02d}"


def print_pipeline_header(total_pdfs):
    print(
        f"\n{PIPELINE_VERSION} | {total_pdfs} PDF détecté(s)\n",
        flush=True,
    )


def print_compact_progress(
    position,
    total,
    pdf_name,
    pages,
    skipped,
    tokens_in,
    tokens_out,
    elapsed_s,
    pipeline_start,
):
    elapsed_global = time.time() - pipeline_start
    avg = elapsed_global / max(position, 1)
    eta = avg * max(total - position, 0)
    status = "SKIP" if skipped else "TRAITÉ"

    print(
        f"[{position}/{total}] "
        f"{pdf_name} | "
        f"pages={int(pages or 0)} | "
        f"{status} | "
        f"IN={int(tokens_in or 0):,} | "
        f"OUT={int(tokens_out or 0):,} | "
        f"{float(elapsed_s or 0):.2f}s | "
        f"ETA={format_duration(eta)}",
        flush=True,
    )


## 6. Inférence GPU batch

In [ ]:
def apply_template(messages):
    """Qwen3 : désactive le mode 'thinking' si supporté."""
    try:
        return processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )


def _message(prompt, image):
    return [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
        ],
    }]


def ask_batch_multi(prompts, images, max_new_tokens, proc=None):
    """
    V7.3 — Un prompt DIFFÉRENT par image, un seul appel à generate.

    C'est le cœur du gain de temps. V7.2 déclarait GPU_BATCH_SIZE_EXTRACTION
    mais appelait ask_single page par page : sur un dossier de 4 pages, 4
    générations séquentielles de ~300 tokens chacune, soit l'essentiel des
    130 s. Le processor accepte déjà une liste de textes ; il suffit de la
    lui donner.

    `proc` permet d'utiliser le processor haute définition sur les vues de
    la planche sans changer le plafond de pixels des pages courantes.
    """
    proc = proc or processor
    if not images:
        return []
    if len(prompts) != len(images):
        raise ValueError(
            f"{len(prompts)} prompt(s) pour {len(images)} image(s)"
        )

    texts_in = [apply_template(_message(p, im)) for p, im in zip(prompts, images)]

    inputs = proc(
        text=texts_in,
        images=list(images),
        return_tensors="pt",
        padding=True,
    ).to(DEVICE)

    t0 = time.time()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.0,
            pad_token_id=proc.tokenizer.eos_token_id,
        )
    torch.cuda.synchronize()

    if out.shape[0] != len(images):
        raise RuntimeError(
            f"Réponses VLM incohérentes : {out.shape[0]} sortie(s) "
            f"pour {len(images)} image(s)"
        )

    elapsed = time.time() - t0
    input_width = inputs["input_ids"].shape[1]
    attention_mask = inputs.get("attention_mask")
    results = []

    # Padding à gauche : toutes les générations commencent au même index.
    for i in range(len(images)):
        generated = out[i][input_width:]
        text = proc.decode(
            generated, skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )
        tokens_in = (
            int(attention_mask[i].sum().item())
            if attention_mask is not None else int(input_width)
        )
        results.append({
            "text": text,
            "tokens_in": tokens_in,
            "tokens_out": int(len(generated)),
            # Temps du batch réparti : utile pour le suivi, à ne pas lire
            # comme un temps de page réel.
            "elapsed_s": round(elapsed / len(images), 3),
            "elapsed_batch_s": round(elapsed, 3),
            "taille_batch": len(images),
        })
    return results


def ask_single(prompt, image, max_new_tokens, proc=None):
    return ask_batch_multi([prompt], [image], max_new_tokens, proc=proc)[0]


def ask_batch(prompt, images, max_new_tokens, proc=None):
    """Même prompt pour toutes les images (classification)."""
    return ask_batch_multi([prompt] * len(images), images, max_new_tokens, proc=proc)


print("✅ Inférence V7.3 OK : ask_batch_multi (un prompt par image, un seul generate)")


## 7. Prompts de classification et d’extraction

In [ ]:

PROMPT_CLASSIFICATION = """
Analyse le titre, les en-têtes, la mise en page et les blocs visuels
de cette page.

Classe la page dans exactement une seule catégorie :

- ENGAGEMENT_DOMICILIATION
- CONTRAT_TRAVAIL
- CONTRAT_SPECIFIQUE
- TITRE_TRAVAIL
- PERMIS_TRAVAIL_COUVERTURE
- AUTRE

RÈGLE DE PRIORITÉ ABSOLUE :
Certaines pages contiennent DEUX documents superposés : un titre de
travail bilingue dans la moitié haute et une couverture de permis de
travail dans la moitié basse.
Dans ce cas, classer TOUJOURS la page en TITRE_TRAVAIL.
Le grand titre « جواز العمل / Permis de Travail » de la moitié basse ne
doit JAMAIS l'emporter sur un bloc d'identité présent dans la moitié
haute.

Règles de classification :

- ENGAGEMENT_DOMICILIATION :
  le titre contient « ENGAGEMENT DE DOMICILIATION »
  ou « CONTRAT DES SALARIES ETRANGERS ».

- CONTRAT_TRAVAIL :
  le titre contient « CONTRAT DE TRAVAIL A DUREE DETERMINEE ».

- CONTRAT_SPECIFIQUE :
  le titre contient « CONTRAT DE TRAVAIL SPECIFIQUE
  A LA MAIN D'OEUVRE ETRANGERE ».

- TITRE_TRAVAIL :
  la page contient un bloc d'identité du travailleur, reconnaissable à
  AU MOINS DEUX des éléments suivants :
    - une photographie d'identité ;
    - les libellés « Nom » et « Prénom » suivis de valeurs ;
    - les libellés « Date de naissance » / « Lieu de naissance » ;
    - le libellé « Date d'entrée en Algérie » ;
    - des libellés arabes d'identité (اللقب، الإسم، تاريخ الإزدياد).
  Cette catégorie s'applique même si la page comporte aussi des cachets,
  un QR code, du texte de loi ou un second document en dessous.

- PERMIS_TRAVAIL_COUVERTURE :
  UNIQUEMENT si la page ne contient AUCUN bloc d'identité du travailleur
  et se limite au titre « Permis de Travail », au numéro de série et aux
  extraits de loi.

- AUTRE :
  aucun type ne correspond clairement.

Ne te base jamais uniquement sur le numéro de page.

Retourne uniquement ce JSON :
{
  "type_document": "TYPE",
  "confidence": 0.00,
  "titre_detecte": "TITRE BRUT OU null",
  "bloc_identite_present": true
}
"""

COMMON_RAW_RULES = """
Tu analyses une seule image, qui peut être une page entière ou un
recadrage d'une page.

RÈGLES OBLIGATOIRES :
1. Extraire uniquement les champs demandés.
2. Pour chaque champ, rechercher le libellé indiqué.
3. Recopier uniquement la valeur située juste après le libellé :
   - sur la même ligne ;
   - ou immédiatement sur la ligne suivante si la valeur continue.
4. Conserver la valeur exactement comme elle apparaît :
   espaces, ponctuation, séparateurs, format de date et format de montant.
5. Ne corrige pas l'orthographe.
6. Ne normalise pas les dates.
7. Ne normalise pas les montants.
8. Ne sépare pas automatiquement le nom et le prénom.
9. Ne complète pas une valeur partiellement lisible.
10. N'utilise aucune valeur provenant d'une autre page.
11. Si le libellé est absent ou la valeur illisible, retourne null.
12. N'invente jamais une valeur.
13. Retourne uniquement un objet JSON valide, sans commentaire.
14. Un champ absent du recadrage que tu analyses doit valoir null.
    Ne devine pas ce qui se trouve hors de l'image.
"""

PROMPT_ENGAGEMENT = COMMON_RAW_RULES + """
TYPE ATTENDU : ENGAGEMENT_DOMICILIATION

Extrais exactement les clés suivantes :

{
  "DOM_NOM_RAISON_SOCIAL_CLIENT": null,
  "DOM_COMPTE_LOCAL": null,
  "DOM_ADRESSE_CLIENT": null,
  "DOM_AGENCE_DOMICILIATAIRE": null,
  "DOM_NUMERO_CONTRAT": null,
  "DOM_DUREE_CONTRAT_MOIS": null,
  "DOM_DATE_DEBUT_CONTRAT": null,
  "DOM_DATE_FIN_CONTRAT": null,
  "DOM_NOM_RAISON_SOCIAL_EMPLOYEUR": null,
  "DOM_ADRESSE_EMPLOYEUR": null,
  "DOM_SALAIRE_NET_MENSUEL": null,
  "DOM_PART_TRANSFERABLE": null,
  "DOM_TAUX_TRANSFERABLE": null,
  "DOM_MONTANT_TOTAL_DOMICILIE": null,
  "DOM_DATE_SIGNATURE": null
}

Libellés et règles :

- DOM_NOM_RAISON_SOCIAL_CLIENT :
  valeur après « Nom et raison sociale ».
  La valeur peut être répartie en deux colonnes (nom puis prénom) :
  recopier les deux, séparés par un espace.

- DOM_COMPTE_LOCAL :
  valeur après « N de compte » ou « N° de compte ».

- DOM_ADRESSE_CLIENT :
  valeur après la première occurrence de « Adresse »
  dans la section « Identification du client ».

- DOM_AGENCE_DOMICILIATAIRE :
  valeur après « Agence domiciliataire », en haut à droite.

- DOM_NUMERO_CONTRAT :
  valeur après « Numéro du contrat ».

- DOM_DUREE_CONTRAT_MOIS :
  valeur après « Durée du contrat » ou « Duré du contrat ».

- DOM_DATE_DEBUT_CONTRAT :
  valeur après « Date de début de contrat ».

- DOM_DATE_FIN_CONTRAT :
  valeur après « Date de fin de contrat ».

- DOM_NOM_RAISON_SOCIAL_EMPLOYEUR :
  valeur après « Nom et raison sociale de L'Employeur ».

- DOM_ADRESSE_EMPLOYEUR :
  valeur après « Adresse de L'Employeur ».
  Continuer sur la ligne suivante si l'adresse se poursuit.

- DOM_SALAIRE_NET_MENSUEL :
  valeur après « Salaire net mensuel ».

- DOM_PART_TRANSFERABLE :
  valeur après « Montant de la part transférable ».

- DOM_TAUX_TRANSFERABLE :
  valeur après « Pourcentage en regard du salaire net mensuel ».

- DOM_MONTANT_TOTAL_DOMICILIE :
  valeur après « Montant domicilié en DZD ».
  Ce champ est souvent laissé vide : retourner null dans ce cas.

- DOM_DATE_SIGNATURE :
  date manuscrite ou imprimée située près de la mention
  « lu et approuvé », en bas de page.
"""

PROMPT_CONTRAT = COMMON_RAW_RULES + """
TYPE ATTENDU : CONTRAT_TRAVAIL

Extrais exactement les clés suivantes :

{
  "CTR_REFERENCE_DOCUMENT": null,
  "CTR_TYPE": null,
  "CTR_EMPLOYEUR": null,
  "CTR_ACTIVITE_EMPLOYEUR": null,
  "CTR_DUREE_MOIS": null,
  "CTR_DATE_DEBUT_CONTRAT": null,
  "CTR_POSTE": null,
  "CTR_NOM_PRENOM_TRAVAILLEUR": null,
  "CTR_PERE_NOM_PRENOM": null,
  "CTR_MERE_NOM_PRENOM": null,
  "CTR_NATIONALITE": null,
  "CTR_DATE_NAISSANCE": null,
  "CTR_LIEU_PAYS_NAISSANCE": null,
  "CTR_ADRESSE_ALGERIE": null,
  "CTR_QUALIFICATION": null,
  "CTR_NUMERO_PERMIS_TRAVAIL": null,
  "CTR_DATE_DELIVRANCE_PERMIS": null,
  "CTR_DATE_DEBUT_VALIDITE_PERMIS": null,
  "CTR_DATE_FIN_VALIDITE_PERMIS": null,
  "CTR_SALAIRE_BRUT": null,
  "CTR_SALAIRE_NET": null,
  "CTR_AFFILIATION_SS": null,
  "CTR_NUMERO_EMPLOYEUR": null,
  "CTR_DATE_SIGNATURE": null,
  "CTR_REFERENCE_DOMICILIATION": null,
  "CTR_SIGNATURE_TRAVAILLEUR_PRESENTE": null,
  "CTR_SIGNATURE_EMPLOYEUR_PRESENTE": null,
  "CTR_CACHET_EMPLOYEUR_PRESENT": null
}

Libellés et règles :

- CTR_REFERENCE_DOCUMENT :
  référence imprimée dans le coin supérieur gauche,
  par exemple « TR-6166 ».

- CTR_TYPE :
  titre complet du document.

- CTR_EMPLOYEUR :
  valeur après
  « au nom de l'employeur ci-après désigné : ».

- CTR_ACTIVITE_EMPLOYEUR :
  valeur après « Nature de l'activité : ».

- CTR_DUREE_MOIS :
  valeur après « pour une durée de : »
  et avant « à compter du ».

- CTR_DATE_DEBUT_CONTRAT :
  valeur après « à compter du : ».

- CTR_POSTE :
  valeur après « En qualité de : ».

- CTR_NOM_PRENOM_TRAVAILLEUR :
  valeur après « A (Mr/Mme) : ».

- CTR_PERE_NOM_PRENOM :
  valeur après « Fils de : »
  et avant « et de : ».

- CTR_MERE_NOM_PRENOM :
  valeur après « et de : ».

- CTR_NATIONALITE :
  valeur après « Nationalité : ».

- CTR_DATE_NAISSANCE :
  valeur après « Né(e) le : »
  et avant « à ».

- CTR_LIEU_PAYS_NAISSANCE :
  valeur après « à » sur la ligne de naissance.

- CTR_ADRESSE_ALGERIE :
  valeur après « Adresse en Algérie : ».

- CTR_QUALIFICATION :
  valeur après « Qualification professionnelle : ».

- CTR_NUMERO_PERMIS_TRAVAIL :
  valeur après « permis de travail N° ».
  Recopier la référence complète, y compris la partie après « / ».

- CTR_DATE_DELIVRANCE_PERMIS :
  valeur après « Délivré le : ».

- CTR_DATE_DEBUT_VALIDITE_PERMIS :
  première date après « Valable du ».

- CTR_DATE_FIN_VALIDITE_PERMIS :
  date après « au » sur la même ligne.

- CTR_SALAIRE_BRUT :
  valeur après « Montant du salaire mensuel brut : ».

- CTR_SALAIRE_NET :
  valeur après « Montant du salaire mensuel net : ».

- CTR_AFFILIATION_SS :
  valeur après « Affiliation à la sécurité sociale : ».

- CTR_NUMERO_EMPLOYEUR :
  valeur après « Employeur : ».

- CTR_DATE_SIGNATURE :
  date après « Fait à : Bethioua, le ».

- CTR_REFERENCE_DOMICILIATION :
  dans le cachet « DOMICILIATION IMPORT »,
  recopier les cinq cases dans l'ordre
  et les séparer par « | ».
  Exemple : 271901|2026.1|40|00119|DZD
  null si ce cachet est absent de la page.

Contrôles visuels :
- CTR_SIGNATURE_TRAVAILLEUR_PRESENTE :
  true si un tracé manuscrit est visible directement sous
  « Signature du Travailleur Etranger », sinon false.

- CTR_SIGNATURE_EMPLOYEUR_PRESENTE :
  true si un tracé manuscrit est visible directement sous
  « Signature de l'Employeur », sinon false.

- CTR_CACHET_EMPLOYEUR_PRESENT :
  true si une empreinte de cachet est visible dans la zone
  « Signature de l'Employeur », sinon false.
"""

PROMPT_CONTRAT_SPECIFIQUE = COMMON_RAW_RULES + """
TYPE ATTENDU : CONTRAT_SPECIFIQUE

Extrais exactement les clés suivantes :

{
  "CTS_REFERENCE_DOCUMENT": null,
  "CTS_SAP_ID": null,
  "CTS_EMPLOYEUR": null,
  "CTS_ACTIVITE_EMPLOYEUR": null,
  "CTS_DUREE_MOIS": null,
  "CTS_DATE_DEBUT_CONTRAT": null,
  "CTS_POSTE": null,
  "CTS_NOM_PRENOM_TRAVAILLEUR": null,
  "CTS_PERE_NOM_PRENOM": null,
  "CTS_MERE_NOM_PRENOM": null,
  "CTS_NATIONALITE": null,
  "CTS_DATE_NAISSANCE": null,
  "CTS_LIEU_PAYS_NAISSANCE": null,
  "CTS_ADRESSE_ALGERIE": null,
  "CTS_QUALIFICATION": null,
  "CTS_NUMERO_PERMIS_TRAVAIL": null,
  "CTS_DATE_DELIVRANCE_PERMIS": null,
  "CTS_DATE_DEBUT_VALIDITE_PERMIS": null,
  "CTS_DATE_FIN_VALIDITE_PERMIS": null,
  "CTS_LIGNE_SALAIRE_BRUTE": null,
  "CTS_SALAIRE_NET": null,
  "CTS_SALAIRE_NET_ANCIEN": null,
  "CTS_MENTION_AU_LIEU_DE_PRESENTE": null,
  "CTS_PART_TRANSFERABLE": null,
  "CTS_PART_PAYABLE_DZD": null,
  "CTS_NUMERO_SS_PAYS_ORIGINE": null,
  "CTS_NUMERO_SS_ALGERIE": null,
  "CTS_DATE_DOCUMENT": null,
  "CTS_SIGNATURE_TRAVAILLEUR_PRESENTE": null,
  "CTS_SIGNATURE_EMPLOYEUR_PRESENTE": null,
  "CTS_CACHET_EMPLOYEUR_PRESENT": null,
  "CTS_VISA_INSPECTION_TRAVAIL_PRESENT": null
}

Libellés et règles :

- CTS_REFERENCE_DOCUMENT :
  référence imprimée dans le coin supérieur gauche, par exemple « TR-6166 ».

- CTS_SAP_ID :
  valeur après « SAP id - » en haut de page.

- CTS_EMPLOYEUR :
  valeur après
  « au nom de l'employeur ci-après désigné : ».

- CTS_ACTIVITE_EMPLOYEUR :
  valeur après « Nature de l'activité : ».

- CTS_DUREE_MOIS :
  valeur après « pour une durée de : ».

- CTS_DATE_DEBUT_CONTRAT :
  valeur après « A compter du : ».

- CTS_POSTE :
  valeur après « en qualité de : ».

- CTS_NOM_PRENOM_TRAVAILLEUR :
  valeur après « A (Mr/Mme) : ».

- CTS_PERE_NOM_PRENOM :
  valeur après « Fils de : »
  et avant « et de : ».

- CTS_MERE_NOM_PRENOM :
  valeur après « et de : ».

- CTS_NATIONALITE :
  valeur après « Nationalité : ».

- CTS_DATE_NAISSANCE :
  valeur après « Né(e) le : »
  et avant « à ».

- CTS_LIEU_PAYS_NAISSANCE :
  valeur après « à » sur la ligne de naissance.

- CTS_ADRESSE_ALGERIE :
  valeur après « Adresse en Algérie : ».

- CTS_QUALIFICATION :
  valeur après « Qualification professionnelle : ».

- CTS_NUMERO_PERMIS_TRAVAIL :
  valeur après « permis de travail N° ».
  Recopier la référence complète, y compris la partie après « / ».

- CTS_DATE_DELIVRANCE_PERMIS :
  valeur après « Délivré le : ».

- CTS_DATE_DEBUT_VALIDITE_PERMIS :
  première date après « valable du ».

- CTS_DATE_FIN_VALIDITE_PERMIS :
  date après « au » sur la même ligne.

--- LIGNE DE SALAIRE : RÈGLE PARTICULIÈRE ---

Cette ligne peut prendre DEUX formes :

  Forme A (salaire inchangé) :
    « Salaire mensuel de base net : 506,471.38 »

  Forme B (augmentation de salaire) :
    « Salaire mensuel de base net : 506,471.38 au lieu de 479,274.29 »

- CTS_LIGNE_SALAIRE_BRUTE :
  recopier la ligne ENTIÈRE telle qu'elle apparaît, du libellé
  « Salaire mensuel de base net » jusqu'à la fin de la ligne,
  sans rien retirer.

- CTS_SALAIRE_NET :
  le PREMIER montant de cette ligne, c'est-à-dire celui situé
  immédiatement après « Salaire mensuel de base net : ».
  En forme B, c'est le NOUVEAU salaire, celui placé AVANT
  « au lieu de ». Ne jamais recopier « au lieu de » ni ce qui suit.

- CTS_SALAIRE_NET_ANCIEN :
  le SECOND montant de cette ligne, celui placé APRÈS
  « au lieu de ». C'est l'ANCIEN salaire.
  null si la mention « au lieu de » est absente de cette ligne.

- CTS_MENTION_AU_LIEU_DE_PRESENTE :
  true si la ligne de salaire contient « au lieu de », sinon false.

--- SUITE ---

- CTS_PART_TRANSFERABLE :
  valeur après « La part transférable : ».

- CTS_PART_PAYABLE_DZD :
  valeur après « La part payable en dinars algérien : ».

- CTS_NUMERO_SS_PAYS_ORIGINE :
  valeur après « Dans le pays d'origine : ».

- CTS_NUMERO_SS_ALGERIE :
  valeur après « En Algérie : ».

- CTS_DATE_DOCUMENT :
  date après « Fait à : Bethioua, le ».

Contrôles visuels :
- CTS_SIGNATURE_TRAVAILLEUR_PRESENTE :
  true si un tracé manuscrit est visible sous
  « Signature du Travailleur Etranger ».

- CTS_SIGNATURE_EMPLOYEUR_PRESENTE :
  true si un tracé manuscrit est visible sous
  « Signature de l'Employeur ».

- CTS_CACHET_EMPLOYEUR_PRESENT :
  true si une empreinte de cachet est visible dans la zone employeur.

- CTS_VISA_INSPECTION_TRAVAIL_PRESENT :
  true si le bas de page comporte un cachet ou une mention manuscrite
  près de « Le présent contrat a été visé par nous ».
"""

PROMPT_TITRE_TRAVAIL = COMMON_RAW_RULES + """
TYPE ATTENDU : TITRE_TRAVAIL

DESCRIPTION DE LA PAGE :
Il s'agit d'un titre de travail algérien, bilingue arabe/français,
photocopié en noir et blanc. La qualité est dégradée, les valeurs sont
souvent inscrites sur des lignes de pointillés, et des cachets ronds
peuvent recouvrir partiellement le texte.

MISE EN PAGE — À LIRE ATTENTIVEMENT :
Le document est organisé en DEUX COLONNES.

  COLONNE DE GAUCHE — le poste et l'employeur.
  Chaque ligne suit le schéma :
      libellé français ..... valeur .....        libellé arabe
  Le libellé arabe est collé au bord DROIT de la colonne gauche.
  La valeur se trouve ENTRE le libellé français et le libellé arabe.
  Libellés : « Durée », « Du », « Au », « Lieu de travail »,
  « Nom de l'organisme employeur », « Adresse de l'organisme employeur »,
  « Fait à », « Le ».

  COLONNE DE DROITE — l'identité du travailleur, à côté de la photo.
  Chaque ligne suit le schéma :
      libellé français ..... valeur .....        libellé arabe
  Libellés : « Nom », « Prénom », « Date de naissance »,
  « Lieu de naissance », « Pays », « Nationalité », « Qualification »,
  « Date d'entrée en Algérie ».

RÈGLE CRITIQUE :
Ne JAMAIS confondre un libellé arabe avec une valeur.
La valeur est toujours le texte latin situé sur les pointillés,
entre le libellé français et le libellé arabe.
Si une ligne ne contient que des libellés et des pointillés vides,
retourner null pour ce champ.

Extrais exactement :

{
  "TTR_NUMERO_PERMIS": null,
  "TTR_NUMERO_MANUSCRIT": null,
  "TTR_POSTE": null,
  "TTR_DUREE": null,
  "TTR_DATE_DEBUT": null,
  "TTR_DATE_FIN": null,
  "TTR_LIEU_TRAVAIL": null,
  "TTR_EMPLOYEUR": null,
  "TTR_ADRESSE_EMPLOYEUR": null,
  "TTR_FAIT_A": null,
  "TTR_DATE_DELIVRANCE": null,
  "TTR_NOM": null,
  "TTR_PRENOM": null,
  "TTR_DATE_NAISSANCE": null,
  "TTR_LIEU_NAISSANCE": null,
  "TTR_PAYS": null,
  "TTR_NATIONALITE": null,
  "TTR_QUALIFICATION": null,
  "TTR_DATE_ENTREE_ALGERIE": null,
  "TTR_PHOTO_PRESENTE": null,
  "TTR_CACHET_PRESENT": null
}

Libellés et règles :

- TTR_NUMERO_PERMIS :
  référence encadrée en haut à gauche, de la forme
  « ( R ) 21-00002974 / 31-25-001448 ».
  Recopier la référence complète, y compris les deux parties
  séparées par « / ». Ignorer les parenthèses et la lettre isolée.

- TTR_NUMERO_MANUSCRIT :
  nombre manuscrit inscrit juste sous la référence encadrée,
  par exemple « 6466 ». null si absent.

- TTR_POSTE :
  texte situé sous la phrase
  « Le titulaire du présent permis de travail est autorisé à occuper
  le poste de travail de ».
  Ce texte peut s'étendre sur deux ou trois lignes de pointillés :
  recopier l'ensemble, en séparant les fragments par un espace.

- TTR_DUREE :
  valeur après « Durée », par exemple « 2 ANS, 0 JOURS ».

- TTR_DATE_DEBUT :
  valeur après « Du », sur la ligne portant le libellé arabe
  « إبتداء من ».

- TTR_DATE_FIN :
  valeur après « Au », sur la ligne portant le libellé arabe « إلى ».
  Attention : ce libellé est « Au », pas « Fin de travail ».

- TTR_LIEU_TRAVAIL :
  valeur après « Lieu de travail ».
  Peut s'étendre sur deux lignes : recopier l'ensemble.

- TTR_EMPLOYEUR :
  valeur après « Nom de l'organisme employeur ».
  Peut s'étendre sur deux lignes : recopier l'ensemble.

- TTR_ADRESSE_EMPLOYEUR :
  valeur après « Adresse de l'organisme employeur ».
  Peut s'étendre sur deux lignes : recopier l'ensemble.

- TTR_FAIT_A :
  valeur après « Fait à ».

- TTR_DATE_DELIVRANCE :
  valeur après « Le », sous « Fait à ».
  Cette date est fréquemment recouverte par un cachet rond :
  si elle reste illisible, retourner null plutôt que de deviner.

- TTR_NOM :
  valeur après « Nom », ligne portant le libellé arabe « اللقب ».
  C'est le nom de famille seul.

- TTR_PRENOM :
  valeur après « Prénom », ligne portant le libellé arabe « الإسم ».

- TTR_DATE_NAISSANCE :
  valeur après « Date de naissance »,
  ligne portant le libellé arabe « تاريخ الإزدياد ».

- TTR_LIEU_NAISSANCE :
  valeur après « Lieu de naissance »,
  ligne portant le libellé arabe « مكان الإزدياد ».
  La valeur peut associer une ville et un pays séparés par « / » :
  recopier l'ensemble.

- TTR_PAYS :
  valeur après « Pays », ligne portant le libellé arabe « البلد ».

- TTR_NATIONALITE :
  valeur après « Nationalité »,
  ligne portant le libellé arabe « الجنسية ».

- TTR_QUALIFICATION :
  valeur après « Qualification »,
  ligne portant le libellé arabe « التأهيل ».
  La valeur peut être coupée en fin de ligne et se poursuivre sur la
  ligne suivante : recopier l'ensemble sans ajouter d'espace au point
  de coupure si le mot est manifestement scindé.

- TTR_DATE_ENTREE_ALGERIE :
  valeur après « Date d'entrée en Algérie »,
  ligne portant le libellé arabe « تاريخ الدخول إلى الجزائر ».

Contrôles visuels :
- TTR_PHOTO_PRESENTE :
  true si une photographie d'identité est visible en haut à droite.

- TTR_CACHET_PRESENT :
  true si au moins un cachet rond est visible sur le document.
"""

PROMPT_PERMIS_COUVERTURE = COMMON_RAW_RULES + """
TYPE ATTENDU : PERMIS_TRAVAIL_COUVERTURE

Cette image correspond à la couverture du permis de travail :
titre « جواز العمل / Permis de Travail », extraits de loi et cachet
de la Direction de l'Emploi de la Wilaya.

Extrais exactement :

{
  "PTR_NUMERO_SERIE": null,
  "PTR_WILAYA": null,
  "PTR_CACHET_DIRECTION_EMPLOI_PRESENT": null
}

- PTR_NUMERO_SERIE :
  numéro de série imprimé en bas de la couverture,
  après « N° de Série » ou isolé en bas à gauche.
  null si illisible.

- PTR_WILAYA :
  valeur après « Direction de l'Emploi de la Wilaya de : ».
  null si la ligne est vide.

- PTR_CACHET_DIRECTION_EMPLOI_PRESENT :
  true si un cachet officiel est visible sur cette zone.

Ne pas extraire le contenu des extraits de loi imprimés à droite.
"""

PROMPTS_EXTRACTION = {
    "ENGAGEMENT_DOMICILIATION": PROMPT_ENGAGEMENT,
    "CONTRAT_TRAVAIL": PROMPT_CONTRAT,
    "CONTRAT_SPECIFIQUE": PROMPT_CONTRAT_SPECIFIQUE,
    "TITRE_TRAVAIL": PROMPT_TITRE_TRAVAIL,
    "PERMIS_TRAVAIL_COUVERTURE": PROMPT_PERMIS_COUVERTURE,
}

TYPES_VALIDES = set(PROMPTS_EXTRACTION) | {"AUTRE"}


def champs_attendus_depuis_prompt(prompt):
    """
    Récupère la liste des clés depuis le squelette JSON du prompt.
    Évite de maintenir une seconde liste qui divergerait des prompts.
    """
    match = re.search(r"\{[^{}]*\}", prompt, flags=re.S)
    if not match:
        return []
    bloc = match.group(0)
    try:
        return list(json.loads(bloc).keys())
    except Exception:
        return re.findall(r'"([A-Z0-9_]+)"\s*:', bloc)


CHAMPS_ATTENDUS = {
    doc_type: champs_attendus_depuis_prompt(prompt)
    for doc_type, prompt in PROMPTS_EXTRACTION.items()
}


# =====================================================================
# V7.3 — Prompts spécialisés par VUE de la planche « permis de travail »
# =====================================================================
# V7.2 envoyait le même PROMPT_TITRE_TRAVAIL (21 champs, ~1 800 tokens)
# sur chaque recadrage, y compris sur une demi-image qui n'en contient
# que huit. Deux effets néfastes :
#   - coût : 1 800 tokens d'entrée et ~350 de sortie par vue ;
#   - qualité : en demandant les champs de la colonne de gauche sur une
#     image qui ne montre que celle de droite, on invite le modèle à
#     combler. C'est le mode d'échec que la V7.1 cherchait à corriger.
# V7.3 donne à chaque vue un prompt réduit à ses propres champs.

PROMPT_VUE_ENTETE = COMMON_RAW_RULES + """
TYPE ATTENDU : TITRE_TRAVAIL — VUE ENTÊTE

Cette image est un recadrage serré du COIN SUPÉRIEUR GAUCHE d'un titre
de travail algérien. Elle contient un cadre rectangulaire portant la
référence du permis, et parfois un nombre manuscrit juste en dessous.

C'est le champ le plus important du document : recopie-le caractère par
caractère, sans rien compléter.

Extrais exactement :

{
  "TTR_NUMERO_PERMIS": null,
  "TTR_NUMERO_MANUSCRIT": null
}

- TTR_NUMERO_PERMIS :
  référence encadrée, de la forme « ( R ) 21-00002974 / 31-25-001448 ».
  Recopier les DEUX parties séparées par « / ».
  Ignorer les parenthèses et la lettre isolée « R ».
  Si une seule des deux parties est lisible, ne recopier que celle-là.

- TTR_NUMERO_MANUSCRIT :
  nombre manuscrit inscrit sous le cadre, par exemple « 6466 ».
  null si absent.
"""

PROMPT_VUE_IDENTITE = COMMON_RAW_RULES + """
TYPE ATTENDU : TITRE_TRAVAIL — VUE COLONNE IDENTITÉ

Cette image est la MOITIÉ DROITE d'un titre de travail algérien
bilingue, photocopié en noir et blanc. Elle contient la photographie
d'identité et le bloc d'état civil du travailleur.

Chaque ligne suit le schéma :
    libellé français ..... valeur .....        libellé arabe
La valeur est le texte LATIN situé sur les pointillés, ENTRE le libellé
français et le libellé arabe.
Ne JAMAIS recopier un libellé arabe comme valeur.
Si une ligne ne contient que des libellés et des pointillés vides,
retourner null.

Extrais exactement :

{
  "TTR_NOM": null,
  "TTR_PRENOM": null,
  "TTR_DATE_NAISSANCE": null,
  "TTR_LIEU_NAISSANCE": null,
  "TTR_PAYS": null,
  "TTR_NATIONALITE": null,
  "TTR_QUALIFICATION": null,
  "TTR_DATE_ENTREE_ALGERIE": null,
  "TTR_PHOTO_PRESENTE": null
}

- TTR_NOM : après « Nom », ligne portant « اللقب ». Nom de famille seul.
- TTR_PRENOM : après « Prénom », ligne portant « الإسم ».
- TTR_DATE_NAISSANCE : après « Date de naissance », « تاريخ الإزدياد ».
- TTR_LIEU_NAISSANCE : après « Lieu de naissance », « مكان الإزدياد ».
  Ville et pays peuvent être séparés par « / » : recopier l'ensemble.
- TTR_PAYS : après « Pays », « البلد ».
- TTR_NATIONALITE : après « Nationalité », « الجنسية ».
- TTR_QUALIFICATION : après « Qualification », « التأهيل ».
  La valeur peut être coupée en fin de ligne et se poursuivre sur la
  ligne suivante : recopier l'ensemble sans ajouter d'espace au point
  de coupure si le mot est manifestement scindé.
- TTR_DATE_ENTREE_ALGERIE : après « Date d'entrée en Algérie ».
- TTR_PHOTO_PRESENTE : true si une photographie d'identité est visible.
"""

PROMPT_VUE_POSTE = COMMON_RAW_RULES + """
TYPE ATTENDU : TITRE_TRAVAIL — VUE COLONNE POSTE

Cette image est la MOITIÉ GAUCHE d'un titre de travail algérien
bilingue. Elle décrit le poste occupé, la durée de validité, le lieu de
travail et l'employeur.

Chaque ligne suit le schéma :
    libellé français ..... valeur .....        libellé arabe
Le libellé arabe est collé au bord DROIT de l'image.
La valeur est le texte LATIN situé sur les pointillés, entre les deux.
Ne JAMAIS recopier un libellé arabe comme valeur.

Extrais exactement :

{
  "TTR_POSTE": null,
  "TTR_DUREE": null,
  "TTR_DATE_DEBUT": null,
  "TTR_DATE_FIN": null,
  "TTR_LIEU_TRAVAIL": null,
  "TTR_EMPLOYEUR": null,
  "TTR_ADRESSE_EMPLOYEUR": null,
  "TTR_FAIT_A": null,
  "TTR_DATE_DELIVRANCE": null,
  "TTR_CACHET_PRESENT": null
}

- TTR_POSTE :
  texte sous « Le titulaire du présent permis de travail est autorisé à
  occuper le poste de travail de ». Peut s'étendre sur deux ou trois
  lignes de pointillés : recopier l'ensemble, fragments séparés par un
  espace.
- TTR_DUREE : après « Durée », par exemple « 2 ANS, 0 JOURS ».
- TTR_DATE_DEBUT : après « Du », ligne portant « إبتداء من ».
- TTR_DATE_FIN : après « Au », ligne portant « إلى ».
  Attention : le libellé est « Au », pas « Fin de travail ».
- TTR_LIEU_TRAVAIL : après « Lieu de travail », éventuellement sur deux lignes.
- TTR_EMPLOYEUR : après « Nom de l'organisme employeur ».
- TTR_ADRESSE_EMPLOYEUR : après « Adresse de l'organisme employeur ».
- TTR_FAIT_A : après « Fait à ».
- TTR_DATE_DELIVRANCE :
  après « Le », sous « Fait à ». Cette date est fréquemment recouverte
  par un cachet rond : si elle reste illisible, retourner null plutôt
  que de deviner.
- TTR_CACHET_PRESENT : true si au moins un cachet rond est visible.
"""

# ---------------------------------------------------------------------
# V7.3 — Prompts de relecture ciblée
# ---------------------------------------------------------------------
# Déclenchés uniquement lorsqu'un contrôle arithmétique échoue. Ils ne
# demandent que les champs mis en cause, sur un recadrage serré du bloc
# concerné : le modèle relit quelques nombres, pas une page entière.

PROMPT_RELECTURE_MONTANTS_DOM = COMMON_RAW_RULES + """
TYPE ATTENDU : ENGAGEMENT_DOMICILIATION — RELECTURE DES MONTANTS

Cette image est un recadrage du bloc de montants d'un engagement de
domiciliation. Un contrôle arithmétique a échoué sur une première
lecture : relis ces valeurs chiffre par chiffre.

Recopie chaque montant EXACTEMENT comme il est écrit, avec ses
séparateurs de milliers et sa virgule ou son point décimal.

{
  "DOM_SALAIRE_NET_MENSUEL": null,
  "DOM_PART_TRANSFERABLE": null,
  "DOM_TAUX_TRANSFERABLE": null,
  "DOM_MONTANT_TOTAL_DOMICILIE": null
}

- DOM_SALAIRE_NET_MENSUEL : après « Salaire net mensuel ».
- DOM_PART_TRANSFERABLE : après « Montant de la part transférable ».
- DOM_TAUX_TRANSFERABLE : après « Pourcentage en regard du salaire net mensuel ».
- DOM_MONTANT_TOTAL_DOMICILIE : après « Montant domicilié en DZD ».
  Ce champ est souvent laissé vide : retourner null dans ce cas.
"""

PROMPT_RELECTURE_MONTANTS_CTS = COMMON_RAW_RULES + """
TYPE ATTENDU : CONTRAT_SPECIFIQUE — RELECTURE DES MONTANTS

Cette image est un recadrage du bloc de montants d'un contrat spécifique
à la main d'œuvre étrangère. Un contrôle arithmétique a échoué :
relis ces valeurs chiffre par chiffre.

{
  "CTS_SALAIRE_NET": null,
  "CTS_SALAIRE_NET_ANCIEN": null,
  "CTS_PART_TRANSFERABLE": null,
  "CTS_PART_PAYABLE_DZD": null,
  "CTS_LIGNE_SALAIRE_BRUTE": null
}

- CTS_SALAIRE_NET : premier montant après « Salaire mensuel de base net ».
- CTS_SALAIRE_NET_ANCIEN : montant situé après « au lieu de », s'il existe.
- CTS_PART_TRANSFERABLE : après « La part transférable ».
- CTS_PART_PAYABLE_DZD : après « La part payable en dinars algérien ».
- CTS_LIGNE_SALAIRE_BRUTE : la ligne complète « Salaire mensuel de base
  net : ... », recopiée telle quelle, y compris la mention « au lieu de »
  si elle est présente.
"""

PROMPT_RELECTURE_DATES_DOM = COMMON_RAW_RULES + """
TYPE ATTENDU : ENGAGEMENT_DOMICILIATION — RELECTURE DES DATES

Cette image est un recadrage du bloc d'identification de l'opération.
Un contrôle de cohérence de dates a échoué : relis ces valeurs.

{
  "DOM_NUMERO_CONTRAT": null,
  "DOM_DUREE_CONTRAT_MOIS": null,
  "DOM_DATE_DEBUT_CONTRAT": null,
  "DOM_DATE_FIN_CONTRAT": null
}

Recopier les dates exactement comme écrites, sans les reformater.
"""

PROMPT_RELECTURE_IDENTIFICATION = COMMON_RAW_RULES + """
TYPE ATTENDU : ENGAGEMENT_DOMICILIATION — RELECTURE DE L'IDENTIFICATION

Cette image est un recadrage du cadre « Identification du client ».
Le numéro de compte a échoué au contrôle de clé : relis-le chiffre par
chiffre, sans en omettre ni en ajouter.

{
  "DOM_NOM_RAISON_SOCIAL_CLIENT": null,
  "DOM_COMPTE_LOCAL": null,
  "DOM_AGENCE_DOMICILIATAIRE": null
}

- DOM_COMPTE_LOCAL : après « N de compte » ou « N° de compte ».
  Recopier tous les chiffres, y compris les espaces de séparation.
"""

# ---------------------------------------------------------------------
# V7.3 — Vues de la planche permis
# ---------------------------------------------------------------------
# Ces vues ne sont PAS une cascade : elles sont toutes envoyées, en un
# seul appel à generate. Une cascade avec arrêt anticipé pouvait ne
# jamais exécuter la vue qui portait précisément le champ manquant —
# exactement le défaut qui empêchait les versions < 7.2 de lire le
# permis de travail.

VUES_PLANCHE = {
    "TITRE_TRAVAIL": [
        {"nom": "VUE_ENTETE", "crop": "entete", "prompt": PROMPT_VUE_ENTETE,
         "max_tokens": 200},
        {"nom": "VUE_IDENTITE", "crop": "colonne_identite", "prompt": PROMPT_VUE_IDENTITE,
         "max_tokens": 400},
        {"nom": "VUE_POSTE", "crop": "colonne_poste", "prompt": PROMPT_VUE_POSTE,
         "max_tokens": 450},
        {"nom": "VUE_PLANCHE_ENTIERE", "crop": "titre", "prompt": PROMPT_TITRE_TRAVAIL,
         "max_tokens": 700},
    ],
    "PERMIS_TRAVAIL_COUVERTURE": [
        {"nom": "VUE_COUVERTURE", "crop": "couverture",
         "prompt": PROMPT_PERMIS_COUVERTURE, "max_tokens": 200},
    ],
}

# Ordre de préférence par champ : quelle vue fait foi quand deux vues
# donnent une valeur. La vue serrée prime sur la vue d'ensemble, qui ne
# sert que de filet.
PRIORITE_VUES = {
    "TTR_NUMERO_PERMIS": ["VUE_ENTETE", "VUE_PLANCHE_ENTIERE"],
    "TTR_NUMERO_MANUSCRIT": ["VUE_ENTETE", "VUE_PLANCHE_ENTIERE"],
    "TTR_NOM": ["VUE_IDENTITE", "VUE_PLANCHE_ENTIERE"],
    "TTR_PRENOM": ["VUE_IDENTITE", "VUE_PLANCHE_ENTIERE"],
    "TTR_DATE_NAISSANCE": ["VUE_IDENTITE", "VUE_PLANCHE_ENTIERE"],
    "TTR_LIEU_NAISSANCE": ["VUE_IDENTITE", "VUE_PLANCHE_ENTIERE"],
    "TTR_PAYS": ["VUE_IDENTITE", "VUE_PLANCHE_ENTIERE"],
    "TTR_NATIONALITE": ["VUE_IDENTITE", "VUE_PLANCHE_ENTIERE"],
    "TTR_QUALIFICATION": ["VUE_IDENTITE", "VUE_PLANCHE_ENTIERE"],
    "TTR_DATE_ENTREE_ALGERIE": ["VUE_IDENTITE", "VUE_PLANCHE_ENTIERE"],
    "TTR_PHOTO_PRESENTE": ["VUE_IDENTITE", "VUE_PLANCHE_ENTIERE"],
    "TTR_POSTE": ["VUE_POSTE", "VUE_PLANCHE_ENTIERE"],
    "TTR_DUREE": ["VUE_POSTE", "VUE_PLANCHE_ENTIERE"],
    "TTR_DATE_DEBUT": ["VUE_POSTE", "VUE_PLANCHE_ENTIERE"],
    "TTR_DATE_FIN": ["VUE_POSTE", "VUE_PLANCHE_ENTIERE"],
    "TTR_LIEU_TRAVAIL": ["VUE_POSTE", "VUE_PLANCHE_ENTIERE"],
    "TTR_EMPLOYEUR": ["VUE_POSTE", "VUE_PLANCHE_ENTIERE"],
    "TTR_ADRESSE_EMPLOYEUR": ["VUE_POSTE", "VUE_PLANCHE_ENTIERE"],
    "TTR_FAIT_A": ["VUE_POSTE", "VUE_PLANCHE_ENTIERE"],
    "TTR_DATE_DELIVRANCE": ["VUE_POSTE", "VUE_PLANCHE_ENTIERE"],
    "TTR_CACHET_PRESENT": ["VUE_POSTE", "VUE_PLANCHE_ENTIERE"],
}

# ---------------------------------------------------------------------
# Stratégies d'extraction des pages dactylographiées
# ---------------------------------------------------------------------
# Inchangé sur le principe : une passe standard suffit. La seconde passe
# du contrat spécifique n'est déclenchée que si le remplissage reste bas,
# et elle est désormais groupée en batch avec les autres pages à relire.
STRATEGIES_EXTRACTION = {
    "ENGAGEMENT_DOMICILIATION": [{"nom": "STANDARD"}],
    "CONTRAT_TRAVAIL": [{"nom": "STANDARD"}],
    "CONTRAT_SPECIFIQUE": [
        {"nom": "STANDARD"},
        {"nom": "HD_BLOC_CENTRAL", "crop": (0.05, 0.70, 0.00, 1.00)},
    ],
}

# ---------------------------------------------------------------------
# Recadrages de relecture ciblée
# ---------------------------------------------------------------------
# Fractions calibrées sur la mise en page observée des formulaires BNPPA.
# À ajuster si le formulaire change : ce sont les seules constantes
# géométriques codées en dur du pipeline.
RECADRAGES_RELECTURE = {
    ("ENGAGEMENT_DOMICILIATION", "bloc_montants"): {
        "crop": (0.48, 0.80, 0.00, 1.00),
        "prompt": PROMPT_RELECTURE_MONTANTS_DOM,
    },
    ("ENGAGEMENT_DOMICILIATION", "bloc_dates"): {
        "crop": (0.30, 0.60, 0.00, 1.00),
        "prompt": PROMPT_RELECTURE_DATES_DOM,
    },
    ("ENGAGEMENT_DOMICILIATION", "bloc_identification"): {
        "crop": (0.14, 0.38, 0.00, 1.00),
        "prompt": PROMPT_RELECTURE_IDENTIFICATION,
    },
    ("CONTRAT_SPECIFIQUE", "bloc_montants"): {
        "crop": (0.28, 0.66, 0.00, 1.00),
        "prompt": PROMPT_RELECTURE_MONTANTS_CTS,
    },
}

print("✅ Prompts V7.3 chargés")
print("   Types gérés        :", ", ".join(sorted(PROMPTS_EXTRACTION)))
print("   Vues planche       :", ", ".join(v["nom"] for v in VUES_PLANCHE["TITRE_TRAVAIL"]))
print("   Prompts relecture  :", len(RECADRAGES_RELECTURE))


## 8. Normalisation technique

### 8.1 Typage, validation et contrôles croisés (V7.3)

Toute valeur extraite est typée, bornée et confrontée aux autres
documents du dossier. Le pipeline produit un verdict par contrôle ;
les décisions réglementaires restent dans Alteryx.


In [ ]:
# =====================================================================
# V7.3 — Couche de typage et de validation des valeurs
# =====================================================================
# Principe : une valeur extraite n'est jamais écrasée. Pour chaque champ
# le pipeline produit trois informations :
#   <CHAMP>_RAW     la chaîne exactement telle que lue (auditabilité)
#   <CHAMP>         la valeur typée (float, date ISO, entier, texte)
#   <CHAMP>_STATUT  OK / AMBIGU / ILLISIBLE / HORS_BORNES / ABSENT
# Alteryx et le contrôleur humain peuvent ainsi trancher sur les cas
# douteux sans revenir au PDF.

STATUT_OK = "OK"
STATUT_ABSENT = "ABSENT"
STATUT_AMBIGU = "AMBIGU"
STATUT_ILLISIBLE = "ILLISIBLE"
STATUT_HORS_BORNES = "HORS_BORNES"


def _texte_source(value):
    if value is None or isinstance(value, bool):
        return None
    texte = str(value).replace("\u00a0", " ").strip()
    return texte or None


# ---------------------------------------------------------------------
# Montants
# ---------------------------------------------------------------------
# Les documents mélangent les conventions : « 506,471.38 » (anglo-saxon),
# « 506 471,38 » (français), « 466300.88 », « 25 323,57 ».
# V7.2 devinait à partir de la position relative de la dernière virgule
# et du dernier point. V7.3 raisonne sur le nombre de chiffres qui SUIT
# le dernier séparateur : 2 chiffres = décimales, 3 chiffres = milliers.
# Les cas réellement indécidables ne sont plus tranchés au hasard, ils
# sont marqués AMBIGU.

def normaliser_montant(value):
    brut = _texte_source(value)
    if brut is None:
        return {"valeur": None, "raw": None, "statut": STATUT_ABSENT}

    texte = re.sub(r"[^0-9,.\-]", "", brut)
    if not texte or not re.search(r"\d", texte):
        return {"valeur": None, "raw": brut, "statut": STATUT_ILLISIBLE}

    negatif = texte.lstrip().startswith("-")
    texte = texte.replace("-", "")
    statut = STATUT_OK

    separateurs = [(i, c) for i, c in enumerate(texte) if c in ",."]
    if separateurs:
        pos, _ = separateurs[-1]
        decimales = len(texte) - pos - 1
        if decimales == 2 or (decimales == 1 and len(separateurs) == 1):
            entier = re.sub(r"[.,]", "", texte[:pos])
            texte = f"{entier}.{texte[pos + 1:]}"
        elif decimales == 3:
            # Séparateur de milliers : « 1.234 » ou « 25 323 ».
            # Indécidable si c'est l'unique séparateur et qu'aucun autre
            # indice n'existe : on retient la lecture « milliers », qui
            # est la convention des documents, mais on lève le drapeau.
            if len(separateurs) == 1:
                statut = STATUT_AMBIGU
            texte = re.sub(r"[.,]", "", texte)
        else:
            entier = re.sub(r"[.,]", "", texte[:pos])
            texte = f"{entier}.{texte[pos + 1:]}"

    try:
        valeur = round(float(texte), 2)
    except Exception:
        return {"valeur": None, "raw": brut, "statut": STATUT_ILLISIBLE}

    if negatif:
        valeur = -valeur
    if not (MONTANT_MIN_PLAUSIBLE <= abs(valeur) <= MONTANT_MAX_PLAUSIBLE):
        return {"valeur": valeur, "raw": brut, "statut": STATUT_HORS_BORNES}

    return {"valeur": valeur, "raw": brut, "statut": statut}


def normalize_amount(value):
    """Compatibilité V7.2 : renvoie uniquement la valeur typée."""
    return normaliser_montant(value)["valeur"]


# ---------------------------------------------------------------------
# Dates
# ---------------------------------------------------------------------
# pd.to_datetime(dayfirst=True, errors="coerce") accepte beaucoup trop :
# « 2026.1 » devient une date valide, ce qui a déjà produit des dates de
# contrat fantaisistes. V7.3 n'accepte que des motifs explicites et
# contrôle l'année.

MOTIFS_DATE = [
    (r"^(\d{1,2})[/\-. ](\d{1,2})[/\-. ](\d{4})$", ("j", "m", "a")),
    (r"^(\d{4})[/\-.](\d{1,2})[/\-.](\d{1,2})$", ("a", "m", "j")),
    (r"^(\d{1,2})[/\-. ](\d{1,2})[/\-. ](\d{2})$", ("j", "m", "aa")),
    (r"^(\d{2})(\d{2})(\d{4})$", ("j", "m", "a")),
]


def normaliser_date(value):
    if isinstance(value, (pd.Timestamp, datetime)):
        d = value.date() if not isinstance(value, date) or isinstance(value, datetime) else value
        return {"valeur": d, "iso": d.strftime(FORMAT_DATE_SORTIE),
                "raw": str(value), "statut": STATUT_OK}
    if isinstance(value, date):
        return {"valeur": value, "iso": value.strftime(FORMAT_DATE_SORTIE),
                "raw": str(value), "statut": STATUT_OK}

    brut = _texte_source(value)
    if brut is None:
        return {"valeur": None, "iso": None, "raw": None, "statut": STATUT_ABSENT}

    texte = re.sub(r"\s+", " ", brut).strip().strip(".")

    # Dates en toutes lettres rencontrées sur les mentions manuscrites.
    mois_lettres = {
        "JANVIER": 1, "FEVRIER": 2, "FÉVRIER": 2, "MARS": 3, "AVRIL": 4,
        "MAI": 5, "JUIN": 6, "JUILLET": 7, "AOUT": 8, "AOÛT": 8,
        "SEPTEMBRE": 9, "OCTOBRE": 10, "NOVEMBRE": 11, "DECEMBRE": 12,
        "DÉCEMBRE": 12,
    }
    m_lettres = re.match(
        r"^(\d{1,2})\s+([A-Za-zÀ-ÿ]+)\s+(\d{4})$", texte
    )
    if m_lettres and m_lettres.group(2).upper() in mois_lettres:
        try:
            d = date(int(m_lettres.group(3)),
                     mois_lettres[m_lettres.group(2).upper()],
                     int(m_lettres.group(1)))
            return {"valeur": d, "iso": d.strftime(FORMAT_DATE_SORTIE),
                    "raw": brut, "statut": STATUT_OK}
        except ValueError:
            pass

    for motif, ordre in MOTIFS_DATE:
        m = re.match(motif, texte)
        if not m:
            continue
        parts = dict(zip(ordre, m.groups()))
        annee = int(parts.get("a") or (2000 + int(parts["aa"])))
        mois = int(parts["m"])
        jour = int(parts["j"])
        if not (1 <= mois <= 12 and 1 <= jour <= 31):
            return {"valeur": None, "iso": None, "raw": brut, "statut": STATUT_ILLISIBLE}
        if not (ANNEE_MIN_PLAUSIBLE <= annee <= ANNEE_MAX_PLAUSIBLE):
            return {"valeur": None, "iso": None, "raw": brut, "statut": STATUT_HORS_BORNES}
        try:
            d = date(annee, mois, jour)
        except ValueError:
            return {"valeur": None, "iso": None, "raw": brut, "statut": STATUT_ILLISIBLE}
        return {"valeur": d, "iso": d.strftime(FORMAT_DATE_SORTIE),
                "raw": brut, "statut": STATUT_OK}

    return {"valeur": None, "iso": None, "raw": brut, "statut": STATUT_ILLISIBLE}


def parse_date(value):
    """Compatibilité V7.2 : renvoie un objet date ou None."""
    return normaliser_date(value)["valeur"]


# ---------------------------------------------------------------------
# Numéros de compte et clé RIB
# ---------------------------------------------------------------------
# C'est le contrôle le plus rentable du pipeline : la clé attrape une
# erreur sur UN SEUL chiffre, ce qu'aucun contrôle de longueur ne fait.

def normaliser_compte(value):
    brut = _texte_source(value)
    if brut is None:
        return {"valeur": None, "raw": None, "statut": STATUT_ABSENT,
                "cle_valide": None, "anomalie": None}

    compact = re.sub(r"\D", "", brut)
    if not compact:
        return {"valeur": None, "raw": brut, "statut": STATUT_ILLISIBLE,
                "cle_valide": None, "anomalie": "AUCUN_CHIFFRE"}

    anomalies = []
    statut = STATUT_OK
    if len(compact) != LONGUEUR_COMPTE_ATTENDUE:
        statut = STATUT_AMBIGU
        anomalies.append(f"LONGUEUR_{len(compact)}_ATTENDU_{LONGUEUR_COMPTE_ATTENDUE}")
    if not compact.startswith(CODE_BANQUE_BNPPA):
        anomalies.append("CODE_BANQUE_INATTENDU")

    cle_valide = None
    if CONTROLE_CLE_RIB and len(compact) == LONGUEUR_COMPTE_ATTENDUE:
        cle_valide = cle_rib_est_valide(compact)
        if cle_valide is False:
            statut = STATUT_AMBIGU
            anomalies.append("CLE_RIB_INVALIDE")

    return {"valeur": compact, "raw": brut, "statut": statut,
            "cle_valide": cle_valide,
            "anomalie": ";".join(anomalies) or None}


def cle_rib_attendue(compte, constante=None):
    """
    Clé de contrôle d'un compte à 20 chiffres :
        3 chiffres banque + 5 agence + 10 compte + 2 clé.

    constante = 98 a été calibrée sur les comptes BNPPA observés.
    Vérifier avec calibrer_cle_rib() avant de traiter un lot.
    """
    constante = CLE_RIB_CONSTANTE if constante is None else constante
    compact = re.sub(r"\D", "", str(compte or ""))
    if len(compact) != 20:
        return None
    banque, agence, numero = int(compact[:3]), int(compact[3:8]), int(compact[8:18])
    reste = (89 * banque + 15 * agence + 3 * numero) % 97
    cle = constante - reste
    return cle % 97 if cle > 97 or cle < 0 else cle


def cle_rib_est_valide(compte, constante=None):
    compact = re.sub(r"\D", "", str(compte or ""))
    attendue = cle_rib_attendue(compact, constante=constante)
    if attendue is None:
        return None
    return int(compact[18:]) == int(attendue)


def calibrer_cle_rib(comptes_certains, constantes=(97, 98)):
    """
    À lancer une fois sur une liste de comptes dont on est sûr (export
    du référentiel client). Si aucune constante n'atteint 100 %, laisser
    CONTROLE_CLE_RIB = False : mieux vaut pas de contrôle qu'un contrôle
    qui crie au loup.
    """
    comptes = [re.sub(r"\D", "", str(c or "")) for c in comptes_certains]
    comptes = [c for c in comptes if len(c) == 20]
    resultats = {}
    for k in constantes:
        ok = sum(1 for c in comptes if cle_rib_est_valide(c, constante=k))
        resultats[k] = {"testes": len(comptes), "valides": ok,
                        "taux": round(100 * ok / max(len(comptes), 1), 1)}
        print(f"  constante {k} : {ok}/{len(comptes)} comptes valides "
              f"({resultats[k]['taux']} %)")
    return resultats


# ---------------------------------------------------------------------
# Taux, durées, numéros de permis, noms
# ---------------------------------------------------------------------

def normaliser_taux(value):
    """« 95% », « 95 % », « 0,95 » -> 95.0 (pourcentage)."""
    brut = _texte_source(value)
    if brut is None:
        return {"valeur": None, "raw": None, "statut": STATUT_ABSENT}
    m = re.search(r"(\d+(?:[.,]\d+)?)", brut.replace("\u00a0", ""))
    if not m:
        return {"valeur": None, "raw": brut, "statut": STATUT_ILLISIBLE}
    valeur = float(m.group(1).replace(",", "."))
    if valeur <= 1.0 and "%" not in brut:
        valeur *= 100.0            # forme décimale « 0,95 »
    valeur = round(valeur, 4)
    statut = STATUT_OK if 0 < valeur <= 100 else STATUT_HORS_BORNES
    return {"valeur": valeur, "raw": brut, "statut": statut}


def normaliser_duree_mois(value):
    """« 24 », « 24 Mois », « 2 ANS, 0 JOURS » -> 24."""
    brut = _texte_source(value)
    if brut is None:
        return {"valeur": None, "raw": None, "statut": STATUT_ABSENT}
    texte = brut.upper()
    mois = 0
    trouve = False
    m_ans = re.search(r"(\d+)\s*AN", texte)
    if m_ans:
        mois += int(m_ans.group(1)) * 12
        trouve = True
    m_mois = re.search(r"(\d+)\s*MOIS", texte)
    if m_mois:
        mois += int(m_mois.group(1))
        trouve = True
    if not trouve:
        m_nu = re.search(r"(\d+)", texte)
        if not m_nu:
            return {"valeur": None, "raw": brut, "statut": STATUT_ILLISIBLE}
        mois = int(m_nu.group(1))
    statut = STATUT_OK if 0 < mois <= 120 else STATUT_HORS_BORNES
    return {"valeur": mois, "raw": brut, "statut": statut}


MOTIF_PERMIS = r"\d{2}-\d{8}\s*/\s*\d{2}-\d{2}-\d{6}"


def normaliser_permis(value):
    """Format attendu : 21-00002974/31-25-001448."""
    brut = _texte_source(value)
    if brut is None:
        return {"valeur": None, "raw": None, "statut": STATUT_ABSENT}
    texte = re.sub(r"[()\s]", "", brut).upper()
    texte = re.sub(r"^[A-Z]+", "", texte)
    m = re.search(r"(\d{2}-\d{8})/?(\d{2}-\d{2}-\d{6})?", texte)
    if not m:
        chiffres = re.sub(r"\D", "", brut)
        return {"valeur": chiffres or None, "raw": brut,
                "statut": STATUT_AMBIGU if chiffres else STATUT_ILLISIBLE}
    valeur = m.group(1) + (f"/{m.group(2)}" if m.group(2) else "")
    statut = STATUT_OK if m.group(2) else STATUT_AMBIGU
    return {"valeur": valeur, "raw": brut, "statut": statut}


def normaliser_nom(value):
    brut = _texte_source(value)
    if brut is None:
        return {"valeur": None, "raw": None, "statut": STATUT_ABSENT}
    valeur = re.sub(r"\s+", " ", brut.upper()).strip(" .,;:-")
    statut = STATUT_OK if len(valeur) >= 2 else STATUT_AMBIGU
    return {"valeur": valeur, "raw": brut, "statut": statut}


def normaliser_booleen(value):
    if isinstance(value, bool):
        return {"valeur": value, "raw": value, "statut": STATUT_OK}
    brut = _texte_source(value)
    if brut is None:
        return {"valeur": None, "raw": None, "statut": STATUT_ABSENT}
    if brut.upper() in {"TRUE", "VRAI", "OUI", "1", "YES"}:
        return {"valeur": True, "raw": brut, "statut": STATUT_OK}
    if brut.upper() in {"FALSE", "FAUX", "NON", "0", "NO"}:
        return {"valeur": False, "raw": brut, "statut": STATUT_OK}
    return {"valeur": None, "raw": brut, "statut": STATUT_ILLISIBLE}


def normaliser_texte(value):
    brut = _texte_source(value)
    if brut is None:
        return {"valeur": None, "raw": None, "statut": STATUT_ABSENT}
    return {"valeur": re.sub(r"\s+", " ", brut).strip(), "raw": brut,
            "statut": STATUT_OK}


NORMALISEURS = {
    "montant": normaliser_montant,
    "date": normaliser_date,
    "compte": normaliser_compte,
    "taux": normaliser_taux,
    "duree_mois": normaliser_duree_mois,
    "permis": normaliser_permis,
    "nom": normaliser_nom,
    "booleen": normaliser_booleen,
    "texte": normaliser_texte,
}

# ---------------------------------------------------------------------
# Schéma : un type par champ extrait
# ---------------------------------------------------------------------
SCHEMA_CHAMPS = {
    # Engagement de domiciliation — 15 champs
    "DOM_NOM_RAISON_SOCIAL_CLIENT": "nom",
    "DOM_COMPTE_LOCAL": "compte",
    "DOM_ADRESSE_CLIENT": "texte",
    "DOM_AGENCE_DOMICILIATAIRE": "texte",
    "DOM_NUMERO_CONTRAT": "texte",
    "DOM_DUREE_CONTRAT_MOIS": "duree_mois",
    "DOM_DATE_DEBUT_CONTRAT": "date",
    "DOM_DATE_FIN_CONTRAT": "date",
    "DOM_NOM_RAISON_SOCIAL_EMPLOYEUR": "texte",
    "DOM_ADRESSE_EMPLOYEUR": "texte",
    "DOM_SALAIRE_NET_MENSUEL": "montant",
    "DOM_PART_TRANSFERABLE": "montant",
    "DOM_TAUX_TRANSFERABLE": "taux",
    "DOM_MONTANT_TOTAL_DOMICILIE": "montant",
    "DOM_DATE_SIGNATURE": "date",
    # Contrat de travail à durée déterminée — 28 champs
    "CTR_REFERENCE_DOCUMENT": "texte",
    "CTR_TYPE": "texte",
    "CTR_EMPLOYEUR": "texte",
    "CTR_ACTIVITE_EMPLOYEUR": "texte",
    "CTR_DUREE_MOIS": "duree_mois",
    "CTR_DATE_DEBUT_CONTRAT": "date",
    "CTR_POSTE": "texte",
    "CTR_NOM_PRENOM_TRAVAILLEUR": "nom",
    "CTR_PERE_NOM_PRENOM": "nom",
    "CTR_MERE_NOM_PRENOM": "nom",
    "CTR_NATIONALITE": "texte",
    "CTR_DATE_NAISSANCE": "date",
    "CTR_LIEU_PAYS_NAISSANCE": "texte",
    "CTR_ADRESSE_ALGERIE": "texte",
    "CTR_QUALIFICATION": "texte",
    "CTR_NUMERO_PERMIS_TRAVAIL": "permis",
    "CTR_DATE_DELIVRANCE_PERMIS": "date",
    "CTR_DATE_DEBUT_VALIDITE_PERMIS": "date",
    "CTR_DATE_FIN_VALIDITE_PERMIS": "date",
    "CTR_SALAIRE_BRUT": "montant",
    "CTR_SALAIRE_NET": "montant",
    "CTR_AFFILIATION_SS": "texte",
    "CTR_NUMERO_EMPLOYEUR": "texte",
    "CTR_DATE_SIGNATURE": "date",
    "CTR_REFERENCE_DOMICILIATION": "texte",
    "CTR_SIGNATURE_TRAVAILLEUR_PRESENTE": "booleen",
    "CTR_SIGNATURE_EMPLOYEUR_PRESENTE": "booleen",
    "CTR_CACHET_EMPLOYEUR_PRESENT": "booleen",
    # Contrat spécifique à la main d'œuvre étrangère — 32 champs
    "CTS_REFERENCE_DOCUMENT": "texte",
    "CTS_SAP_ID": "texte",
    "CTS_EMPLOYEUR": "texte",
    "CTS_ACTIVITE_EMPLOYEUR": "texte",
    "CTS_DUREE_MOIS": "duree_mois",
    "CTS_DATE_DEBUT_CONTRAT": "date",
    "CTS_POSTE": "texte",
    "CTS_NOM_PRENOM_TRAVAILLEUR": "nom",
    "CTS_PERE_NOM_PRENOM": "nom",
    "CTS_MERE_NOM_PRENOM": "nom",
    "CTS_NATIONALITE": "texte",
    "CTS_DATE_NAISSANCE": "date",
    "CTS_LIEU_PAYS_NAISSANCE": "texte",
    "CTS_ADRESSE_ALGERIE": "texte",
    "CTS_QUALIFICATION": "texte",
    "CTS_NUMERO_PERMIS_TRAVAIL": "permis",
    "CTS_DATE_DELIVRANCE_PERMIS": "date",
    "CTS_DATE_DEBUT_VALIDITE_PERMIS": "date",
    "CTS_DATE_FIN_VALIDITE_PERMIS": "date",
    "CTS_LIGNE_SALAIRE_BRUTE": "texte",
    "CTS_SALAIRE_NET": "montant",
    "CTS_SALAIRE_NET_ANCIEN": "montant",
    "CTS_MENTION_AU_LIEU_DE_PRESENTE": "booleen",
    "CTS_PART_TRANSFERABLE": "montant",
    "CTS_PART_PAYABLE_DZD": "montant",
    "CTS_NUMERO_SS_PAYS_ORIGINE": "texte",
    "CTS_NUMERO_SS_ALGERIE": "texte",
    "CTS_DATE_DOCUMENT": "date",
    "CTS_SIGNATURE_TRAVAILLEUR_PRESENTE": "booleen",
    "CTS_SIGNATURE_EMPLOYEUR_PRESENTE": "booleen",
    "CTS_CACHET_EMPLOYEUR_PRESENT": "booleen",
    "CTS_VISA_INSPECTION_TRAVAIL_PRESENT": "booleen",
    # Titre de travail (planche permis, bloc haut) — 21 champs
    "TTR_NUMERO_PERMIS": "permis",
    "TTR_NUMERO_MANUSCRIT": "texte",
    "TTR_POSTE": "texte",
    "TTR_DUREE": "duree_mois",
    "TTR_DATE_DEBUT": "date",
    "TTR_DATE_FIN": "date",
    "TTR_LIEU_TRAVAIL": "texte",
    "TTR_EMPLOYEUR": "texte",
    "TTR_ADRESSE_EMPLOYEUR": "texte",
    "TTR_FAIT_A": "texte",
    "TTR_DATE_DELIVRANCE": "date",
    "TTR_NOM": "nom",
    "TTR_PRENOM": "nom",
    "TTR_DATE_NAISSANCE": "date",
    "TTR_LIEU_NAISSANCE": "texte",
    "TTR_PAYS": "texte",
    "TTR_NATIONALITE": "texte",
    "TTR_QUALIFICATION": "texte",
    "TTR_DATE_ENTREE_ALGERIE": "date",
    "TTR_PHOTO_PRESENTE": "booleen",
    "TTR_CACHET_PRESENT": "booleen",
    # Couverture du permis (planche permis, bloc bas) — 3 champs
    "PTR_NUMERO_SERIE": "texte",
    "PTR_WILAYA": "texte",
    "PTR_CACHET_DIRECTION_EMPLOI_PRESENT": "booleen",
}


def typer_valeur(champ, valeur):
    normaliseur = NORMALISEURS.get(SCHEMA_CHAMPS.get(champ, "texte"), normaliser_texte)
    resultat = normaliseur(valeur)
    resultat["champ"] = champ
    resultat["type"] = SCHEMA_CHAMPS.get(champ, "texte")
    return resultat


def typer_champs(donnees):
    """
    Applique le schéma à un dictionnaire de valeurs brutes.
    Retourne {champ: {valeur, raw, statut, type, ...}}.
    """
    return {
        champ: typer_valeur(champ, valeur)
        for champ, valeur in (donnees or {}).items()
        if champ in SCHEMA_CHAMPS
    }


def valeur_typee(champs, nom):
    entree = (champs or {}).get(nom) or {}
    return entree.get("valeur")


def valeur_iso(champs, nom):
    entree = (champs or {}).get(nom) or {}
    return entree.get("iso") or entree.get("valeur")


print("✅ V7.3 : typage et validation des champs chargés")
print(f"   Champs au schéma : {len(SCHEMA_CHAMPS)}")
print(f"   Types            : {', '.join(sorted(set(SCHEMA_CHAMPS.values())))}")


In [ ]:
# =====================================================================
# V7.3 — Contrôles croisés et arithmétiques
# =====================================================================
# Le dossier de domiciliation est massivement redondant : le salaire net
# figure sur trois documents, les dates sur quatre, le numéro de permis
# sur trois. Cette redondance est la meilleure garantie d'exactitude
# disponible — bien meilleure qu'une relecture supplémentaire par le
# modèle, parce qu'elle est déterministe et démontrable.
#
# Deux familles de contrôles :
#   CONCORDANCE : la même donnée lue sur deux documents doit coïncider.
#   ARITHMETIQUE : les montants du dossier sont liés par des identités
#                  exactes, vérifiées au centime.
#
# Le pipeline ne décide rien : il produit un verdict par contrôle.
# La décision réglementaire reste dans Alteryx.

NIVEAU_BLOQUANT = "BLOQUANT"
NIVEAU_ALERTE = "ALERTE"
NIVEAU_INFO = "INFO"


def _controle(code, libelle, niveau, statut, attendu=None, obtenu=None,
              ecart=None, champs_concernes=None, commentaire=None):
    return {
        "CODE": code,
        "LIBELLE": libelle,
        "NIVEAU": niveau,
        "STATUT": statut,
        "ATTENDU": attendu,
        "OBTENU": obtenu,
        "ECART": ecart,
        "CHAMPS": ",".join(champs_concernes or []),
        "COMMENTAIRE": commentaire,
    }


def _egal_montant(a, b, tolerance=None):
    tolerance = TOLERANCE_MONTANT_DZD if tolerance is None else tolerance
    return abs(float(a) - float(b)) <= tolerance


def consolider_champ(champs, noms, libelle):
    """
    Consolide une donnée présente sur plusieurs documents.

    Retourne la valeur retenue, toutes les valeurs par source, et un
    verdict de concordance. Une valeur confirmée par deux documents
    indépendants est de confiance ELEVEE ; une valeur lue une seule fois
    reste utilisable mais signalée comme SOURCE_UNIQUE.
    """
    sources = {}
    for nom in noms:
        entree = (champs or {}).get(nom) or {}
        if entree.get("valeur") not in (None, ""):
            sources[nom] = entree.get("iso") or entree.get("valeur")

    if not sources:
        return {"libelle": libelle, "valeur": None, "sources": {},
                "concordance": "ABSENT", "confiance": "NULLE"}

    distinctes = {str(v) for v in sources.values()}
    valeur = sources[noms[0]] if noms[0] in sources else list(sources.values())[0]

    if len(sources) == 1:
        concordance, confiance = "SOURCE_UNIQUE", "MOYENNE"
    elif len(distinctes) == 1:
        concordance, confiance = "CONCORDANT", "ELEVEE"
    else:
        concordance, confiance = "DIVERGENCE", "FAIBLE"

    return {"libelle": libelle, "valeur": valeur, "sources": sources,
            "concordance": concordance, "confiance": confiance,
            "nb_sources": len(sources)}


CHAMPS_CONSOLIDES = {
    "SALAIRE_NET": (["DOM_SALAIRE_NET_MENSUEL", "CTS_SALAIRE_NET", "CTR_SALAIRE_NET"],
                    "Salaire net mensuel"),
    "PART_TRANSFERABLE": (["DOM_PART_TRANSFERABLE", "CTS_PART_TRANSFERABLE"],
                          "Part transférable"),
    "DATE_DEBUT": (["DOM_DATE_DEBUT_CONTRAT", "CTS_DATE_DEBUT_CONTRAT",
                    "CTR_DATE_DEBUT_CONTRAT", "TTR_DATE_DEBUT"],
                   "Date de début de contrat"),
    # Les contrats ne portent pas de date de fin explicite : elle se
    # déduit de la durée. Les sources sont l'engagement et le titre.
    "DATE_FIN": (["DOM_DATE_FIN_CONTRAT", "TTR_DATE_FIN"],
                 "Date de fin de contrat"),
    "VALIDITE_PERMIS_DEBUT": (["CTS_DATE_DEBUT_VALIDITE_PERMIS",
                               "CTR_DATE_DEBUT_VALIDITE_PERMIS", "TTR_DATE_DEBUT"],
                              "Début de validité du permis"),
    "VALIDITE_PERMIS_FIN": (["CTS_DATE_FIN_VALIDITE_PERMIS",
                             "CTR_DATE_FIN_VALIDITE_PERMIS", "TTR_DATE_FIN"],
                            "Fin de validité du permis"),
    "NOM_TRAVAILLEUR": (["CTS_NOM_PRENOM_TRAVAILLEUR",
                         "CTR_NOM_PRENOM_TRAVAILLEUR"], "Nom du travailleur"),
    "DUREE_MOIS": (["DOM_DUREE_CONTRAT_MOIS", "CTS_DUREE_MOIS",
                    "CTR_DUREE_MOIS"], "Durée du contrat (mois)"),
    "NUMERO_PERMIS": (["CTS_NUMERO_PERMIS_TRAVAIL", "CTR_NUMERO_PERMIS_TRAVAIL",
                       "TTR_NUMERO_PERMIS"], "Numéro de permis de travail"),
    "DATE_NAISSANCE": (["CTS_DATE_NAISSANCE", "CTR_DATE_NAISSANCE",
                        "TTR_DATE_NAISSANCE"], "Date de naissance"),
    "NATIONALITE": (["CTS_NATIONALITE", "CTR_NATIONALITE", "TTR_NATIONALITE"],
                    "Nationalité"),
}


def consolider_dossier_champs(champs):
    return {
        cle: consolider_champ(champs, noms, libelle)
        for cle, (noms, libelle) in CHAMPS_CONSOLIDES.items()
    }


def controles_dossier(champs, consolides=None):
    """
    Produit la liste des contrôles pour un dossier.

    Chaque contrôle vaut OK, ANOMALIE ou NON_APPLICABLE (donnée absente).
    """
    consolides = consolides or consolider_dossier_champs(champs)
    controles = []
    v = lambda nom: valeur_typee(champs, nom)

    net = v("DOM_SALAIRE_NET_MENSUEL") or v("CTS_SALAIRE_NET") or v("CTR_SALAIRE_NET")
    transferable = v("DOM_PART_TRANSFERABLE") or v("CTS_PART_TRANSFERABLE")
    payable = v("CTS_PART_PAYABLE_DZD")
    taux = v("DOM_TAUX_TRANSFERABLE")
    domicilie = v("DOM_MONTANT_TOTAL_DOMICILIE")
    duree = v("DOM_DUREE_CONTRAT_MOIS") or v("CTS_DUREE_MOIS") or v("CTR_DUREE_MOIS")
    brut = v("CTR_SALAIRE_BRUT")
    debut = v("DOM_DATE_DEBUT_CONTRAT") or v("CTS_DATE_DEBUT_CONTRAT") or v("CTR_DATE_DEBUT_CONTRAT")
    fin = v("DOM_DATE_FIN_CONTRAT") or v("TTR_DATE_FIN")

    # --- C01 : part transférable = salaire net x taux ------------------
    # Vérifie trois champs d'un coup. Une erreur d'un seul chiffre sur
    # l'un des trois casse l'égalité.
    if net is not None and taux is not None and transferable is not None:
        attendu = round(net * taux / 100.0, 2)
        ok = _egal_montant(attendu, transferable, tolerance=0.02)
        controles.append(_controle(
            "C01", "Part transférable = salaire net × taux",
            NIVEAU_BLOQUANT, "OK" if ok else "ANOMALIE",
            attendu, transferable, round(transferable - attendu, 2),
            ["DOM_SALAIRE_NET_MENSUEL", "DOM_TAUX_TRANSFERABLE", "DOM_PART_TRANSFERABLE"],
        ))
    else:
        controles.append(_controle(
            "C01", "Part transférable = salaire net × taux",
            NIVEAU_BLOQUANT, "NON_APPLICABLE",
            commentaire="Salaire net, taux ou part transférable absent"))

    # --- C02 : part payable en DZD = net - transférable -----------------
    if net is not None and transferable is not None and payable is not None:
        attendu = round(net - transferable, 2)
        ok = _egal_montant(attendu, payable)
        controles.append(_controle(
            "C02", "Part payable en dinars = net − part transférable",
            NIVEAU_BLOQUANT, "OK" if ok else "ANOMALIE",
            attendu, payable, round(payable - attendu, 2),
            ["CTS_PART_PAYABLE_DZD", "CTS_SALAIRE_NET", "CTS_PART_TRANSFERABLE"],
        ))

    # --- C03 : montant domicilié = part transférable x durée ------------
    if transferable is not None and duree:
        attendu = round(transferable * duree, 2)
        if domicilie is None:
            controles.append(_controle(
                "C03", "Montant domicilié = part transférable × durée",
                NIVEAU_INFO, "CALCULE", attendu, None, None,
                ["DOM_MONTANT_TOTAL_DOMICILIE"],
                "Champ laissé vide sur l'engagement : valeur recalculée"))
        else:
            ok = _egal_montant(attendu, domicilie, tolerance=1.0)
            controles.append(_controle(
                "C03", "Montant domicilié = part transférable × durée",
                NIVEAU_BLOQUANT, "OK" if ok else "ANOMALIE",
                attendu, domicilie, round(domicilie - attendu, 2),
                ["DOM_MONTANT_TOTAL_DOMICILIE"]))

    # --- C04 : salaire brut > salaire net -------------------------------
    if brut is not None and net is not None:
        ok = brut > net
        controles.append(_controle(
            "C04", "Salaire brut supérieur au salaire net",
            NIVEAU_ALERTE, "OK" if ok else "ANOMALIE",
            f"> {net}", brut, round(brut - net, 2),
            ["CTR_SALAIRE_BRUT", "CTR_SALAIRE_NET"]))

    # --- C05 : date de fin = date de début + durée - 1 jour -------------
    if debut and fin and duree:
        attendu = _ajouter_mois(debut, int(duree)) - timedelta(days=1)
        ok = attendu == fin
        controles.append(_controle(
            "C05", "Date de fin cohérente avec début + durée",
            NIVEAU_ALERTE, "OK" if ok else "ANOMALIE",
            attendu.strftime(FORMAT_DATE_SORTIE), fin.strftime(FORMAT_DATE_SORTIE),
            (fin - attendu).days,
            ["DOM_DATE_DEBUT_CONTRAT", "DOM_DATE_FIN_CONTRAT", "DOM_DUREE_CONTRAT_MOIS"]))

    # --- C06 : validité du permis couvre la période du contrat ----------
    permis_du = (v("CTS_DATE_DEBUT_VALIDITE_PERMIS")
                 or v("CTR_DATE_DEBUT_VALIDITE_PERMIS") or v("TTR_DATE_DEBUT"))
    permis_au = (v("CTS_DATE_FIN_VALIDITE_PERMIS")
                 or v("CTR_DATE_FIN_VALIDITE_PERMIS") or v("TTR_DATE_FIN"))
    if debut and fin and permis_du and permis_au:
        ok = permis_du <= debut and permis_au >= fin
        controles.append(_controle(
            "C06", "Permis de travail valide sur toute la période du contrat",
            NIVEAU_BLOQUANT, "OK" if ok else "ANOMALIE",
            f"{debut} → {fin}", f"{permis_du} → {permis_au}", None,
            ["CTS_DATE_DEBUT_VALIDITE_PERMIS", "CTS_DATE_FIN_VALIDITE_PERMIS"]))

    # --- C07 : clé de contrôle du compte domiciliataire -----------------
    compte = (champs or {}).get("DOM_COMPTE_LOCAL") or {}
    if compte.get("valeur"):
        if compte.get("cle_valide") is None:
            controles.append(_controle(
                "C07", "Clé de contrôle du compte", NIVEAU_INFO, "NON_APPLICABLE",
                commentaire="Contrôle désactivé ou longueur inattendue"))
        else:
            controles.append(_controle(
                "C07", "Clé de contrôle du compte",
                NIVEAU_BLOQUANT, "OK" if compte["cle_valide"] else "ANOMALIE",
                cle_rib_attendue(compte["valeur"]), compte["valeur"][18:], None,
                ["DOM_COMPTE_LOCAL"], compte.get("anomalie")))

    # --- C08 à C12 : concordances inter-documents -----------------------
    for i, (cle, bloc) in enumerate(sorted(consolides.items()), start=8):
        if bloc["concordance"] == "DIVERGENCE":
            controles.append(_controle(
                f"C{i:02d}", f"Concordance inter-documents : {bloc['libelle']}",
                NIVEAU_BLOQUANT if cle in {"SALAIRE_NET", "PART_TRANSFERABLE",
                                           "DATE_DEBUT", "DATE_FIN",
                                           "NOM_TRAVAILLEUR"} else NIVEAU_ALERTE,
                "ANOMALIE", "valeurs identiques",
                json.dumps(bloc["sources"], ensure_ascii=False, default=str),
                None, list(bloc["sources"].keys())))
        elif bloc["concordance"] == "CONCORDANT":
            controles.append(_controle(
                f"C{i:02d}", f"Concordance inter-documents : {bloc['libelle']}",
                NIVEAU_INFO, "OK", None, bloc["valeur"], None,
                list(bloc["sources"].keys()),
                f"{bloc['nb_sources']} source(s) concordante(s)"))

    # --- Champs au statut douteux ---------------------------------------
    for nom, entree in (champs or {}).items():
        if entree.get("statut") in (STATUT_AMBIGU, STATUT_HORS_BORNES, STATUT_ILLISIBLE):
            controles.append(_controle(
                "C99", f"Valeur douteuse : {nom}",
                NIVEAU_ALERTE if entree["statut"] == STATUT_AMBIGU else NIVEAU_BLOQUANT,
                "ANOMALIE", None, entree.get("raw"), None, [nom],
                f"Statut de typage : {entree['statut']}"))

    return controles


def _ajouter_mois(d, mois):
    """Ajoute n mois à une date en bornant au dernier jour du mois."""
    total = d.month - 1 + int(mois)
    annee = d.year + total // 12
    mois_cible = total % 12 + 1
    jour = min(d.day, calendar.monthrange(annee, mois_cible)[1])
    return date(annee, mois_cible, jour)


def synthese_qualite(controles, champs):
    """Résumé exploitable en une ligne d'Excel."""
    anomalies = [c for c in controles if c["STATUT"] == "ANOMALIE"]
    bloquantes = [c for c in anomalies if c["NIVEAU"] == NIVEAU_BLOQUANT]
    renseignes = sum(1 for e in (champs or {}).values()
                     if e.get("valeur") not in (None, ""))
    return {
        "NB_CHAMPS_RENSEIGNES": renseignes,
        "NB_CHAMPS_ATTENDUS": len(champs or {}),
        "NB_CONTROLES": len(controles),
        "NB_CONTROLES_OK": sum(1 for c in controles if c["STATUT"] == "OK"),
        "NB_ANOMALIES": len(anomalies),
        "NB_ANOMALIES_BLOQUANTES": len(bloquantes),
        "CODES_ANOMALIES": ",".join(sorted({c["CODE"] for c in anomalies})) or None,
        "QUALITE_DOSSIER": (
            "A_CONTROLER" if bloquantes
            else ("A_VERIFIER" if anomalies else "CONFORME")
        ),
    }


# ---------------------------------------------------------------------
# Relecture ciblée : quelle zone relire quand un contrôle échoue
# ---------------------------------------------------------------------
# On ne relance pas le modèle sur toute la page « au cas où ». On relit
# le bloc précis mis en cause par le contrôle, avec un prompt réduit à
# ces seuls champs. Escalade motivée, pas escalade aveugle.

ZONES_RELECTURE = {
    "C01": ("ENGAGEMENT_DOMICILIATION", "bloc_montants"),
    "C02": ("CONTRAT_SPECIFIQUE", "bloc_montants"),
    "C03": ("ENGAGEMENT_DOMICILIATION", "bloc_montants"),
    "C05": ("ENGAGEMENT_DOMICILIATION", "bloc_dates"),
    "C07": ("ENGAGEMENT_DOMICILIATION", "bloc_identification"),
}


def zones_a_relire(controles):
    zones = []
    for c in controles:
        if c["STATUT"] != "ANOMALIE":
            continue
        cible = ZONES_RELECTURE.get(c["CODE"])
        if cible and cible not in zones:
            zones.append(cible)
    return zones[:MAX_RELECTURES_PAR_DOSSIER]


print("✅ V7.3 : contrôles croisés et arithmétiques chargés")
print(f"   Champs consolidés multi-sources : {len(CHAMPS_CONSOLIDES)}")


In [ ]:

NULL_VALUES = {"", "NULL", "NONE", "N/A", "NA", "NEANT", "NÉANT", "ILLISIBLE"}

def clean_raw_value(value):
    if value is None or isinstance(value, bool):
        return value
    text = str(value).strip()
    return None if text.upper() in NULL_VALUES else text

def clean_raw_dict(data):
    return {k: clean_raw_value(v) for k, v in data.items()} if isinstance(data, dict) else {}

# normalize_amount et parse_date sont définis dans la cellule
# « Typage et validation » ci-dessus (versions strictes V7.3).

def normalize_text(value):
    if value is None:
        return ""
    text = re.sub(r"[^\w\s]", " ", str(value).upper())
    return re.sub(r"\s+", " ", text).strip()

def name_similarity(a, b):
    from difflib import SequenceMatcher
    a, b = normalize_text(a), normalize_text(b)
    return SequenceMatcher(None, a, b).ratio() if a and b else 0.0


def normalize_dom_reference(value, date_domiciliation=None):
    """
    Format exact : 271901AAAAT40NNNNNDZD
    """
    if value is None:
        return None

    raw = str(value).strip().upper()
    compact = re.sub(r"[^A-Z0-9]", "", raw)

    if re.fullmatch(r"271901\d{4}[1-4]40\d{5}[A-Z]{3}", compact):
        return compact

    parts = [p.strip().upper() for p in re.split(r"[|;/\\]+", raw) if p.strip()]
    if len(parts) >= 5:
        prefix = re.sub(r"\D", "", parts[0])
        fixed_code = re.sub(r"\D", "", parts[2])
        sequence = re.sub(r"\D", "", parts[3]).zfill(5)
        currency = re.sub(r"[^A-Z]", "", parts[4]) or "DZD"
        match_yq = re.search(r"(\d{4})\D*([1-4])", parts[1])
        if prefix == "271901" and fixed_code == "40" and match_yq:
            return f"271901{match_yq.group(1)}{match_yq.group(2)}40{sequence[:5]}{currency[:3]}"

    match_short = re.search(
        r"(\d{4})\D*([1-4])\D*40\D*(\d{1,5})(?:\D*([A-Z]{3}))?",
        raw
    )
    if match_short:
        year = match_short.group(1)
        quarter = match_short.group(2)
        sequence = match_short.group(3).zfill(5)
        currency = match_short.group(4) or "DZD"
        return f"271901{year}{quarter}40{sequence}{currency}"

    return compact or None


def dom_reference_is_valid(value):
    return bool(value and re.fullmatch(r"271901\d{4}[1-4]40\d{5}[A-Z]{3}", str(value)))

def first_not_null(*values):
    return next((v for v in values if v not in (None, "", "NULL")), None)

def safe_read_excel(path):
    path = Path(path)
    if not path.exists():
        return None
    try:
        frames = []
        xls = pd.ExcelFile(path)
        for sheet in xls.sheet_names:
            df = pd.read_excel(path, sheet_name=sheet)
            if not df.empty:
                df["_SOURCE_SHEET"] = sheet
                frames.append(df)
        return pd.concat(frames, ignore_index=True) if frames else None
    except Exception as exc:
        print(f"⚠️ Lecture impossible {path.name}: {exc}")
        return None

def resolve_column(df, aliases):
    if df is None:
        return None
    normalized = {normalize_text(c): c for c in df.columns}
    for alias in aliases:
        if normalize_text(alias) in normalized:
            return normalized[normalize_text(alias)]
    for alias in aliases:
        target = normalize_text(alias)
        for key, original in normalized.items():
            if target in key or key in target:
                return original
    return None

ALIASES = {
    "numero_dom": ["Numéro de domiciliation", "Numero de domiciliation", "N° domiciliation"],
    "date_dom": ["Date domiciliation", "Date de domiciliation", "Date demande", "Request Decision Date"],
    "nom_client": ["Nom complet/Raison social", "Nom complet/Raison sociale", "Nom du Fournisseur/Client", "Name", "Nom"],
    "numero_client": ["Identifiant client", "Numero client", "N° client", "Code client"],
    "date_debut": ["Date début du contrat", "Date debut du contrat"],
    "date_fin": ["Date fin de contrat", "Date de fin du contrat"],
    "reference": ["Référence", "Reference"],
}

def prepare_reference(df, source):
    if df is None or df.empty:
        return None
    out = pd.DataFrame()
    out["SOURCE_MATCH"] = source
    out["SOURCE_SHEET"] = df.get("_SOURCE_SHEET")
    for key, aliases in ALIASES.items():
        col = resolve_column(df, aliases)
        out[key] = df[col] if col else None
    out["numero_dom_normalise"] = out.apply(lambda r: normalize_dom_reference(r.get("numero_dom"), r.get("date_dom")), axis=1)
    out["date_debut_parse"] = out["date_debut"].apply(parse_date)
    out["date_fin_parse"] = out["date_fin"].apply(parse_date)
    return out


def _first_non_empty(series):
    values = [
        v for v in series.tolist()
        if v is not None and not (isinstance(v, float) and pd.isna(v)) and str(v).strip() != ""
    ]
    return values[0] if values else None


def build_reference_table():
    """
    DOM est le référentiel principal.
    PREDOM est joint à gauche uniquement pour enrichir DOM avec :
    - date début du contrat ;
    - date fin du contrat ;
    - référence PREDOM.

    Une ligne DOM + une ligne PREDOM portant le même numéro DOM
    représentent un seul dossier, et non deux candidats.
    """
    dom = prepare_reference(safe_read_excel(DOM_REFERENCE_EXCEL), "DOM")
    predom = prepare_reference(safe_read_excel(PREDOM_REFERENCE_EXCEL), "PREDOM")

    if dom is None or dom.empty:
        return None

    # Une seule ligne principale par numéro DOM.
    # Les doublons DOM restent signalés pour éviter une attribution automatique risquée.
    dom = dom.copy()
    dom["DOM_MATCH_COUNT"] = dom.groupby("numero_dom_normalise")["numero_dom_normalise"].transform("size")

    if predom is None or predom.empty:
        dom["date_debut_predom"] = None
        dom["date_fin_predom"] = None
        dom["reference_predom"] = None
        dom["PREDOM_MATCH_COUNT"] = 0
        return dom

    predom = predom.copy()

    # Consolidation PREDOM par numéro DOM.
    # On ne choisit pas arbitrairement entre plusieurs valeurs différentes :
    # le nombre de lignes est conservé dans PREDOM_MATCH_COUNT.
    predom_grouped = (
        predom.groupby("numero_dom_normalise", dropna=False)
        .agg(
            date_debut_predom=("date_debut", _first_non_empty),
            date_fin_predom=("date_fin", _first_non_empty),
            date_debut_predom_parse=("date_debut_parse", _first_non_empty),
            date_fin_predom_parse=("date_fin_parse", _first_non_empty),
            reference_predom=("reference", _first_non_empty),
            PREDOM_MATCH_COUNT=("numero_dom_normalise", "size"),
        )
        .reset_index()
    )

    reference = dom.merge(
        predom_grouped,
        on="numero_dom_normalise",
        how="left",
        validate="many_to_one",
    )

    reference["PREDOM_MATCH_COUNT"] = (
        reference["PREDOM_MATCH_COUNT"].fillna(0).astype(int)
    )

    # Pour le matching par période, les dates PREDOM sont prioritaires.
    # Si elles sont absentes, on utilise les dates disponibles dans DOM.
    reference["date_debut_match"] = reference["date_debut_predom_parse"].where(
        reference["date_debut_predom_parse"].notna(),
        reference["date_debut_parse"],
    )
    reference["date_fin_match"] = reference["date_fin_predom_parse"].where(
        reference["date_fin_predom_parse"].notna(),
        reference["date_fin_parse"],
    )

    return reference

print("✅ Helpers référentiel chargés : DOM principal + enrichissement PREDOM")


# ---------------------------------------------------------------------
# V7.1 : relecture de la ligne de salaire du contrat spécifique
# ---------------------------------------------------------------------

# Un montant algérien peut s'écrire 506,471.38 / 506 471,38 / 506471.38
MOTIF_MONTANT = r"\d[\d\s\u00a0.,]*\d|\d"

# Variantes rencontrées : « au lieu de », « au lieu du », « en lieu de »
MOTIF_AU_LIEU_DE = r"\ben\s+lieu\s+de\b|\bau\s+lieu\s+d[eu]\b"


def split_ligne_salaire(ligne):
    """
    Décompose « Salaire mensuel de base net : X au lieu de Y ».

    Retourne les montants normalisés :
      - nouveau : montant avant « au lieu de » (salaire applicable) ;
      - ancien  : montant après « au lieu de » (salaire précédent).

    Sert de filet de sécurité lorsque le modèle a recopié la ligne
    complète sans isoler correctement les deux montants.
    """
    vide = {"nouveau": None, "ancien": None, "mention_presente": False}

    if not ligne:
        return vide

    texte = str(ligne).replace("\xa0", " ").strip()
    separateur = re.search(MOTIF_AU_LIEU_DE, texte, flags=re.I)

    if not separateur:
        montants = re.findall(MOTIF_MONTANT, texte)
        return {
            "nouveau": normalize_amount(montants[0]) if montants else None,
            "ancien": None,
            "mention_presente": False,
        }

    avant = texte[: separateur.start()]
    apres = texte[separateur.end():]

    montants_avant = re.findall(MOTIF_MONTANT, avant)
    montants_apres = re.findall(MOTIF_MONTANT, apres)

    return {
        # Dernier montant avant la mention : le libellé peut contenir des
        # chiffres parasites en début de ligne.
        "nouveau": normalize_amount(montants_avant[-1]) if montants_avant else None,
        "ancien": normalize_amount(montants_apres[0]) if montants_apres else None,
        "mention_presente": True,
    }


print("✅ Relecture de la ligne de salaire (augmentation) OK")


## 9. Consolidation des documents et informations KYC

In [ ]:

def build_page_row(pdf_name, page_record):
    row = {
        "FICHIER": pdf_name,
        "PAGE": page_record.get("page_num"),
        "TYPE_DOCUMENT": page_record.get("doc_type"),
        "TITRE_DETECTE": page_record.get("titre_detecte"),
        "CONFIANCE_CLASSIFICATION": page_record.get("classification_confidence"),
        "CLASSIFICATION_REQUALIFIEE": page_record.get("classification_requalifiee"),
        "BLOC_IDENTITE_PRESENT": page_record.get("bloc_identite_present"),
        "STATUT_EXTRACTION": page_record.get("extraction_status"),
        "TAUX_REMPLISSAGE": page_record.get("extraction_taux_remplissage"),
        "STRATEGIES_UTILISEES": " > ".join(
            s.get("nom", "")
            for s in (page_record.get("extraction_strategies") or [])
        ) or None,
        "ERREUR_EXTRACTION": page_record.get("extraction_error"),
    }
    row.update(page_record.get("raw_data") or {})
    return row

def consolidate_dossier(pdf_name, records):
    row = {
        "FICHIER": pdf_name,
        "NB_PAGES": len(records),
        "TYPES_DOCUMENTS": " | ".join(str(r.get("doc_type")) for r in records),
    }
    pages_by_type = defaultdict(list)
    for record in records:
        pages_by_type[record.get("doc_type")].append(str(record.get("page_num")))
        for key, value in (record.get("raw_data") or {}).items():
            if row.get(key) in (None, ""):
                row[key] = value
    for doc_type, pages in pages_by_type.items():
        row[f"PAGES_{doc_type}"] = ",".join(pages)

    for field in AMOUNT_FIELDS:
        if field in row:
            row[field + "_RAW"] = row[field]
            row[field] = normalize_amount(row[field])

    raw_ref = first_not_null(row.get("CTR_REFERENCE_DOMICILIATION"))
    row["REFERENCE_DOM_EXTRAITE_RAW"] = raw_ref
    row["REFERENCE_DOM_EXTRAITE_NORMALISEE"] = normalize_dom_reference(raw_ref)
    row["REFERENCE_DOM_FORMAT_VALIDE"] = dom_reference_is_valid(row["REFERENCE_DOM_EXTRAITE_NORMALISEE"])
    row["NOM_CLIENT_REFERENCE"] = first_not_null(
        row.get("DOM_NOM_RAISON_SOCIAL_CLIENT"),
        row.get("CTR_NOM_PRENOM_TRAVAILLEUR"),
        row.get("CTS_NOM_PRENOM_TRAVAILLEUR"),
        " ".join(x for x in [str(row.get("TTR_NOM") or "").strip(), str(row.get("TTR_PRENOM") or "").strip()] if x) or None,
    )
    row["NUMERO_CONTRAT_REFERENCE"] = first_not_null(row.get("DOM_NUMERO_CONTRAT"), row.get("CTR_REFERENCE_DOCUMENT"), row.get("CTS_REFERENCE_DOCUMENT"))
    row["DATE_DEBUT_CONTRAT_REFERENCE"] = first_not_null(row.get("DOM_DATE_DEBUT_CONTRAT"), row.get("CTR_DATE_DEBUT_CONTRAT"), row.get("CTS_DATE_DEBUT_CONTRAT"))
    row["DATE_FIN_CONTRAT_REFERENCE"] = first_not_null(row.get("DOM_DATE_FIN_CONTRAT"), row.get("CTR_DATE_FIN_CONTRAT"), row.get("CTS_DATE_FIN_CONTRAT"))
    row["NUMERO_PERMIS_REFERENCE"] = first_not_null(
        row.get("TTR_NUMERO_PERMIS"),
        row.get("CTR_NUMERO_PERMIS_TRAVAIL"),
        row.get("CTS_NUMERO_PERMIS_TRAVAIL"),
        row.get("PTR_NUMERO_SERIE"),
    )

    # -----------------------------------------------------------------
    # V7.1 : augmentation de salaire
    # Ligne source du contrat spécifique :
    #   « Salaire mensuel de base net : 506,471.38 au lieu de 479,274.29 »
    #   -> nouveau salaire = 506 471,38 ; ancien salaire = 479 274,29
    # -----------------------------------------------------------------
    salaire_nouveau = first_not_null(
        row.get("CTS_SALAIRE_NET"),
        row.get("CTR_SALAIRE_NET"),
        row.get("DOM_SALAIRE_NET_MENSUEL"),
    )
    salaire_ancien = row.get("CTS_SALAIRE_NET_ANCIEN")

    # Filet de sécurité : si le modèle n'a pas isolé les deux montants
    # mais a bien recopié la ligne complète, on relit la ligne brute.
    if salaire_ancien is None:
        secours = split_ligne_salaire(row.get("CTS_LIGNE_SALAIRE_BRUTE"))
        if secours["ancien"] is not None:
            salaire_ancien = secours["ancien"]
            row["CTS_SALAIRE_NET_ANCIEN"] = salaire_ancien
            row["SOURCE_AUGMENTATION"] = "RELECTURE_LIGNE_BRUTE"
            if salaire_nouveau is None and secours["nouveau"] is not None:
                salaire_nouveau = secours["nouveau"]
    elif salaire_ancien is not None:
        row["SOURCE_AUGMENTATION"] = "CHAMPS_MODELE"

    row["SALAIRE_NOUVEAU_AUGMENTE"] = salaire_nouveau
    row["SALAIRE_ANCIEN"] = salaire_ancien
    row["AUGMENTATION_DETECTEE"] = bool(
        salaire_ancien is not None
        and salaire_nouveau is not None
        and salaire_nouveau != salaire_ancien
    )

    if row["AUGMENTATION_DETECTEE"]:
        ecart = round(float(salaire_nouveau) - float(salaire_ancien), 2)
        row["MONTANT_AUGMENTATION"] = ecart
        row["SENS_VARIATION_SALAIRE"] = "AUGMENTATION" if ecart > 0 else "DIMINUTION"
        row["TAUX_AUGMENTATION_PCT"] = (
            round(ecart / float(salaire_ancien) * 100, 2)
            if float(salaire_ancien)
            else None
        )
    else:
        row["MONTANT_AUGMENTATION"] = None
        row["SENS_VARIATION_SALAIRE"] = None
        row["TAUX_AUGMENTATION_PCT"] = None
        row.setdefault("SOURCE_AUGMENTATION", None)

    # Cohérence entre l'engagement de domiciliation et le contrat
    # spécifique : le salaire domicilié doit être le NOUVEAU salaire.
    salaire_dom = row.get("DOM_SALAIRE_NET_MENSUEL")
    if salaire_dom is not None and salaire_nouveau is not None:
        row["ECART_SALAIRE_DOM_CONTRAT"] = round(
            float(salaire_dom) - float(salaire_nouveau), 2
        )
        row["COHERENCE_SALAIRE_DOM_CONTRAT"] = (
            abs(row["ECART_SALAIRE_DOM_CONTRAT"]) < 0.01
        )
    else:
        row["ECART_SALAIRE_DOM_CONTRAT"] = None
        row["COHERENCE_SALAIRE_DOM_CONTRAT"] = None

    # Alerte : le salaire domicilié correspond à l'ANCIEN salaire alors
    # qu'une augmentation est actée dans le contrat spécifique.
    row["ALERTE_DOM_SUR_ANCIEN_SALAIRE"] = bool(
        row["AUGMENTATION_DETECTEE"]
        and salaire_dom is not None
        and salaire_ancien is not None
        and abs(float(salaire_dom) - float(salaire_ancien)) < 0.01
    )

    # -----------------------------------------------------------------
    # V7.1 : identité consolidée, enrichie par le titre de travail
    # -----------------------------------------------------------------
    row["NOM_TRAVAILLEUR_REFERENCE"] = first_not_null(
        row.get("CTR_NOM_PRENOM_TRAVAILLEUR"),
        row.get("CTS_NOM_PRENOM_TRAVAILLEUR"),
        " ".join(
            x for x in [
                str(row.get("TTR_NOM") or "").strip(),
                str(row.get("TTR_PRENOM") or "").strip(),
            ] if x
        ) or None,
    )
    row["DATE_NAISSANCE_REFERENCE"] = first_not_null(
        row.get("CTR_DATE_NAISSANCE"),
        row.get("CTS_DATE_NAISSANCE"),
        row.get("TTR_DATE_NAISSANCE"),
    )
    row["NATIONALITE_REFERENCE"] = first_not_null(
        row.get("CTR_NATIONALITE"),
        row.get("CTS_NATIONALITE"),
        row.get("TTR_NATIONALITE"),
    )
    row["DATE_ENTREE_ALGERIE"] = row.get("TTR_DATE_ENTREE_ALGERIE")
    row["PERMIS_TRAVAIL_LU"] = bool(
        row.get("TTR_NOM") or row.get("TTR_NUMERO_PERMIS")
    )

    return row


def match_dossier(row, ref):
    """
    Matching sécurisé.

    1. Si le numéro DOM est extrait du PDF :
       correspondance exacte uniquement.

    2. Si aucun numéro DOM n'est extrait :
       recherche uniquement sur la période exacte du contrat
       (date début + date fin).

    3. Le numéro DOM n'est retenu que lorsqu'un seul candidat DOM
       est incontestable. Sinon, NUMERO_DOM_RETENU reste vide.
    """
    result = {
        "MATCH_SOURCE": None,
        "MATCH_METHOD": None,
        "MATCH_SCORE": 0.00,
        "MATCH_STATUS": "AUCUN_MATCH",
        "MATCH_CANDIDATES_COUNT": 0,
        "NUMERO_CLIENT_RETENU": None,
        "NUMERO_DOM_RETENU": None,
        "DATE_DOM_RETENUE": None,
        "REFERENCE_EXTERNE_RETENUE": None,
        "REFERENCE_PREDOM_RETENUE": None,
        "DATE_DEBUT_CONTRAT_PREDOM": None,
        "DATE_FIN_CONTRAT_PREDOM": None,
        "PREDOM_TROUVEE": False,
    }

    if ref is None or ref.empty:
        result["MATCH_STATUS"] = "REFERENTIEL_ABSENT"
        return result

    dom_ref = row.get("REFERENCE_DOM_EXTRAITE_NORMALISEE")

    # ---------------------------------------------------------
    # 1) Numéro DOM extrait : matching exact uniquement
    # ---------------------------------------------------------
    if dom_ref:
        exact = ref[ref["numero_dom_normalise"] == dom_ref].copy()
        result["MATCH_CANDIDATES_COUNT"] = int(len(exact))
        result["MATCH_METHOD"] = "NUMERO_DOM_EXACT"

        if len(exact) == 0:
            result["MATCH_STATUS"] = "NUMERO_DOM_NON_TROUVE"
            return result

        if len(exact) > 1 or int(exact.iloc[0].get("DOM_MATCH_COUNT") or 1) > 1:
            result["MATCH_STATUS"] = "PLUSIEURS_CANDIDATS_DOM"
            return result

        best = exact.iloc[0]

    # ---------------------------------------------------------
    # 2) Numéro DOM absent : période exacte et candidat unique
    # ---------------------------------------------------------
    else:
        start_date = parse_date(row.get("DATE_DEBUT_CONTRAT_REFERENCE"))
        end_date = parse_date(row.get("DATE_FIN_CONTRAT_REFERENCE"))

        if not start_date or not end_date:
            result["MATCH_STATUS"] = "DATES_CONTRAT_INSUFFISANTES"
            result["MATCH_METHOD"] = "PERIODE_CONTRAT_EXACTE"
            return result

        period_matches = ref[
            (ref["date_debut_match"] == start_date)
            & (ref["date_fin_match"] == end_date)
        ].copy()

        result["MATCH_CANDIDATES_COUNT"] = int(len(period_matches))
        result["MATCH_METHOD"] = "PERIODE_CONTRAT_EXACTE"

        if len(period_matches) == 0:
            result["MATCH_STATUS"] = "AUCUN_MATCH_PERIODE"
            return result

        if len(period_matches) > 1:
            result["MATCH_STATUS"] = "PLUSIEURS_CANDIDATS_PERIODE"
            return result

        best = period_matches.iloc[0]

        # Sécurité supplémentaire : pas d'attribution si le DOM principal
        # contient lui-même plusieurs lignes pour ce numéro.
        if int(best.get("DOM_MATCH_COUNT") or 1) > 1:
            result["MATCH_STATUS"] = "PLUSIEURS_CANDIDATS_DOM"
            return result

    # ---------------------------------------------------------
    # 3) Attribution seulement après match unique et sûr
    # ---------------------------------------------------------
    predom_count = int(best.get("PREDOM_MATCH_COUNT") or 0)

    result.update({
        "MATCH_SOURCE": "DOM",
        "MATCH_SCORE": 100.00,
        "MATCH_STATUS": (
            "DOMICILIATION_TROUVEE"
            if dom_ref
            else "MATCH_PERIODE_EXACTE"
        ),
        "NUMERO_CLIENT_RETENU": best.get("numero_client"),
        "NUMERO_DOM_RETENU": best.get("numero_dom_normalise") or best.get("numero_dom"),
        "DATE_DOM_RETENUE": best.get("date_dom"),
        "REFERENCE_EXTERNE_RETENUE": best.get("reference"),
        "REFERENCE_PREDOM_RETENUE": best.get("reference_predom"),
        "DATE_DEBUT_CONTRAT_PREDOM": best.get("date_debut_predom"),
        "DATE_FIN_CONTRAT_PREDOM": best.get("date_fin_predom"),
        "PREDOM_TROUVEE": predom_count >= 1,
    })

    # Plusieurs lignes PREDOM ne créent pas plusieurs candidats DOM,
    # mais sont signalées pour contrôle des données d'enrichissement.
    if predom_count > 1:
        result["MATCH_STATUS"] = "MATCH_DOM_TROUVE_PREDOM_MULTIPLE"

    return result

def month_segment_rows(row):
    start = parse_date(row.get("DATE_DEBUT_CONTRAT_REFERENCE"))
    end = parse_date(row.get("DATE_FIN_CONTRAT_REFERENCE"))
    if not start or not end or end < start:
        return []
    plafond = row.get("DOM_PART_TRANSFERABLE")
    rows, cursor = [], date(start.year, start.month, 1)
    while cursor <= end:
        days_month = calendar.monthrange(cursor.year, cursor.month)[1]
        month_end = date(cursor.year, cursor.month, days_month)
        seg_start, seg_end = max(start, cursor), min(end, month_end)
        if seg_start <= seg_end:
            # Règle métier :
            # - mois entièrement couvert : AAAA-MM, sans P1/P2 ;
            # - mois partiel commençant le 1er : P1 ;
            # - mois partiel commençant après le 1er : P2.
            mois_complet = (
                seg_start == cursor
                and seg_end == month_end
            )

            if mois_complet:
                part = None
                period = f"{cursor.year:04d}-{cursor.month:02d}"
            elif seg_start.day == 1:
                part = "P1"
                period = f"{cursor.year:04d}-{cursor.month:02d}P1"
            else:
                part = "P2"
                period = f"{cursor.year:04d}-{cursor.month:02d}P2"
            nb_days = (seg_end - seg_start).days + 1
            coef = round(nb_days / days_month, 8)
            rows.append({
                "FICHIER": row.get("FICHIER"),
                "NUMERO_DOMICILIATION": row.get("NUMERO_DOM_RETENU"),
                "DATE_DOMICILIATION": row.get("DATE_DOM_RETENUE"),
                "DATE_DOMICILIATION_SOURCE": "FICHIER_DOM" if row.get("DATE_DOM_RETENUE") else None,
                "NUMERO_CLIENT": row.get("NUMERO_CLIENT_RETENU"),
                "NOM_CLIENT": row.get("NOM_CLIENT_REFERENCE"),
                "NUMERO_CONTRAT": row.get("NUMERO_CONTRAT_REFERENCE"),
                "DATE_DEBUT_CONTRAT": start.isoformat(),
                "DATE_FIN_CONTRAT": end.isoformat(),
                "NUMERO_PERMIS_TRAVAIL": row.get("NUMERO_PERMIS_REFERENCE"),
                "PERIODE_TL": period, "MOIS_BASE": f"{cursor.year:04d}-{cursor.month:02d}",
                "PARTIE": part, "DATE_DEBUT_SEGMENT": seg_start.isoformat(),
                "DATE_FIN_SEGMENT": seg_end.isoformat(),
                "NB_JOURS_SEGMENT": round(float(nb_days), 2), "NB_JOURS_MOIS": round(float(days_month), 2),
                "COEFFICIENT_PRORATA": round(float(coef), 2),
                "SALAIRE_NET_REFERENCE": row.get("DOM_SALAIRE_NET_MENSUEL"),
                "TAUX_TRANSFERABLE_REFERENCE_RAW": row.get("DOM_TAUX_TRANSFERABLE"),
                "PLAFOND_MENSUEL_REFERENCE": plafond,
                "MONTANT_MAX_THEORIQUE": round(plafond * coef, 2) if plafond is not None else None,
                "MONTANT_AUTORISE_SAISI": None, "MONTANT_TRANSFERE": None,
                "SOLDE_RESTANT": None,
                "MATCH_STATUS": row.get("MATCH_STATUS"),
                "MATCH_METHOD": row.get("MATCH_METHOD"),
                "MATCH_SCORE": row.get("MATCH_SCORE"),
                "MATCH_CANDIDATES_COUNT": row.get("MATCH_CANDIDATES_COUNT"),
                "REFERENCE_PREDOM": row.get("REFERENCE_PREDOM_RETENUE"),
                # V7.1 : contexte augmentation, répété sur chaque période
                # pour que les contrôles mensuels Alteryx sachent sur quelle
                # base salariale le plafond a été calculé.
                "SALAIRE_ANCIEN": row.get("SALAIRE_ANCIEN"),
                "SALAIRE_NOUVEAU_AUGMENTE": row.get("SALAIRE_NOUVEAU_AUGMENTE"),
                "AUGMENTATION_DETECTEE": row.get("AUGMENTATION_DETECTEE"),
                "MONTANT_AUGMENTATION": row.get("MONTANT_AUGMENTATION"),
                "TAUX_AUGMENTATION_PCT": row.get("TAUX_AUGMENTATION_PCT"),
                "ALERTE_DOM_SUR_ANCIEN_SALAIRE": row.get("ALERTE_DOM_SUR_ANCIEN_SALAIRE"),
            })
        cursor = date(cursor.year + 1, 1, 1) if cursor.month == 12 else date(cursor.year, cursor.month + 1, 1)
    return rows

print("✅ Consolidation, matching sécurisé et planning V6.3 chargés")


In [ ]:

# Test technique P1/P2 de la V6
_demo = {
    "FICHIER": "demo.pdf",
    "DATE_DEBUT_CONTRAT_REFERENCE": "12/06/2025",
    "DATE_FIN_CONTRAT_REFERENCE": "11/06/2026",
    "DOM_PART_TRANSFERABLE": 442985.84,
    "DOM_SALAIRE_NET_MENSUEL": 466300.88,
    "DOM_TAUX_TRANSFERABLE": "95%",
    "NUMERO_CONTRAT_REFERENCE": "DEMO",
    "NOM_CLIENT_REFERENCE": "CLIENT DEMO",
    "NUMERO_PERMIS_REFERENCE": "PERMIS-DEMO",
    "NUMERO_DOM_RETENU": "DOM-DEMO",
    "DATE_DOM_RETENUE": "01/01/2025",
    "NUMERO_CLIENT_RETENU": "CLIENT-001",
    "MATCH_STATUS": "TEST",
}

_demo_rows = month_segment_rows(_demo)

assert len(_demo_rows) == 13
assert _demo_rows[0]["PERIODE_TL"] == "2025-06P2"
assert _demo_rows[-1]["PERIODE_TL"] == "2026-06P1"
assert _demo_rows[0]["NB_JOURS_MOIS"] == 30
assert _demo_rows[0]["NB_JOURS_SEGMENT"] == 19
assert _demo_rows[-1]["NB_JOURS_SEGMENT"] == 11

print("✅ Test planning P1/P2 V6 réussi")
print(
    _demo_rows[0]["PERIODE_TL"],
    _demo_rows[0]["MONTANT_MAX_THEORIQUE"],
)
print(
    _demo_rows[-1]["PERIODE_TL"],
    _demo_rows[-1]["MONTANT_MAX_THEORIQUE"],
)


## 10. Gestion des mois scindés P1 / P2 et du planning TL

In [ ]:

assert normalize_dom_reference("271901|2026.1|40|00119|DZD") == "271901202614000119DZD"
assert normalize_dom_reference("2026.1.40-00119") == "271901202614000119DZD"
assert dom_reference_is_valid("271901202614000119DZD")
print("✅ Test format DOM réussi")


### Test de la règle P1 / P2 demandée

In [ ]:

print("Le planning est généré uniquement depuis la page ENGAGEMENT_DOMICILIATION.")


## 11. Classification et extraction d’un dossier

In [ ]:
# =====================================================================
# V7.3 — Classification
# =====================================================================

def _classer_lot(images, pages_source):
    """Un seul generate pour un lot d'images de classification."""
    records = []
    for start in range(0, len(images), GPU_BATCH_SIZE_CLASSIFICATION):
        lot_img = images[start:start + GPU_BATCH_SIZE_CLASSIFICATION]
        lot_page = pages_source[start:start + GPU_BATCH_SIZE_CLASSIFICATION]
        sorties = ask_batch(PROMPT_CLASSIFICATION, lot_img,
                            MAX_NEW_TOKENS_CLASSIFICATION)
        for page, sortie in zip(lot_page, sorties):
            records.append((page, sortie))
    return records


def _interpreter_classification(sortie):
    parsed = parse_json_response(sortie["text"])
    doc_type = parsed.get("type_document") or parsed.get("type") or "AUTRE"
    try:
        confidence = float(parsed.get("confidence") or 0)
    except Exception:
        confidence = 0.0
    bloc_identite = bool(parsed.get("bloc_identite_present"))

    if doc_type not in TYPES_VALIDES:
        doc_type = "AUTRE"

    # La planche « permis de travail » porte deux documents. Si le modèle
    # signale un bloc d'identité tout en classant la page en couverture,
    # on force TITRE_TRAVAIL : c'est le bloc identité qui porte les
    # données exploitables.
    requalifie = False
    if bloc_identite and doc_type in ("PERMIS_TRAVAIL_COUVERTURE", "AUTRE"):
        doc_type = "TITRE_TRAVAIL"
        confidence = max(confidence, CLASSIFICATION_THRESHOLD)
        requalifie = True

    if confidence < CLASSIFICATION_THRESHOLD and not requalifie:
        doc_type = "AUTRE"

    return {
        "doc_type": doc_type,
        "titre_detecte": parsed.get("titre_detecte"),
        "bloc_identite_present": bloc_identite,
        "classification_requalifiee": requalifie,
        "classification_confidence": confidence,
    }


def classify_pages(pages):
    """
    V7.3 :
      - les pages blanches ne partent plus au GPU ;
      - la classification tourne en basse résolution (~4x moins de tokens) ;
      - toute page classée AUTRE ou PERMIS_TRAVAIL_COUVERTURE est
        reclassée en pleine résolution. On n'économise jamais sur la
        planche permis, qui est la page la plus difficile du dossier.
    """
    records = []
    a_classer = [p for p in pages if not p.get("blanche")]

    for page in pages:
        if page.get("blanche"):
            records.append(_nouveau_record(page, {
                "doc_type": "AUTRE", "titre_detecte": None,
                "bloc_identite_present": False,
                "classification_requalifiee": False,
                "classification_confidence": 1.0,
            }, {"text": "", "tokens_in": 0, "tokens_out": 0, "elapsed_s": 0.0},
                statut="PAGE_BLANCHE"))

    if a_classer:
        images = [p.get("image_classification") or p["image"] for p in a_classer]
        for page, sortie in _classer_lot(images, a_classer):
            interpretation = _interpreter_classification(sortie)
            records.append(_nouveau_record(page, interpretation, sortie))

    # --- Reclassement pleine résolution des cas douteux ---------------
    if RECLASSEMENT_PLEINE_RESOLUTION and CLASSIFICATION_BASSE_RESOLUTION:
        douteux = [
            r for r in records
            if r["extraction_status"] != "PAGE_BLANCHE"
            and r["doc_type"] in ("AUTRE", "PERMIS_TRAVAIL_COUVERTURE")
        ]
        if douteux:
            pages_par_num = {p["page_num"]: p for p in pages}
            images = [pages_par_num[r["page_num"]]["image"] for r in douteux]
            sorties = ask_batch(PROMPT_CLASSIFICATION, images,
                                MAX_NEW_TOKENS_CLASSIFICATION)
            for record, sortie in zip(douteux, sorties):
                interpretation = _interpreter_classification(sortie)
                record.update(interpretation)
                record["classification_reclassee"] = True
                record["classification_tokens_in"] += sortie["tokens_in"]
                record["classification_tokens_out"] += sortie["tokens_out"]
                record["classification_elapsed_s"] = round(
                    record["classification_elapsed_s"] + sortie["elapsed_s"], 3)

    records.sort(key=lambda r: r["page_num"])
    return records


def _nouveau_record(page, interpretation, sortie, statut="NON_LANCEE"):
    record = {
        "page_num": page["page_num"],
        "image": page["image"],
        "width": page["width"],
        "height": page["height"],
        "white_ratio": page["white_ratio"],
        "page_blanche": bool(page.get("blanche")),
        "classification_raw_text": sortie.get("text"),
        "classification_tokens_in": sortie.get("tokens_in", 0),
        "classification_tokens_out": sortie.get("tokens_out", 0),
        "classification_elapsed_s": sortie.get("elapsed_s", 0.0),
        "classification_reclassee": False,
        "raw_data": {},
        "extraction_status": statut,
        "extraction_error": None,
        "extraction_raw_text": None,
        "extraction_strategies": None,
        "extraction_vues": None,
        "extraction_conflits": None,
        "extraction_taux_remplissage": 0.0,
        "extraction_tokens_in": 0,
        "extraction_tokens_out": 0,
        "extraction_elapsed_s": 0.0,
    }
    record.update(interpretation)
    return record


# =====================================================================
# V7.3 — Extraction des pages dactylographiées, en batch
# =====================================================================

def _appliquer_sortie(record, sortie, nom_strategie, crop=None):
    parsed, statut_json = parse_json_response(sortie["text"], tracer=True)
    parsed = clean_raw_dict(parsed)
    nouveaux = 0
    for cle, valeur in parsed.items():
        if record["raw_data"].get(cle) in (None, "") and valeur not in (None, ""):
            record["raw_data"][cle] = valeur
            nouveaux += 1

    record["extraction_tokens_in"] += sortie["tokens_in"]
    record["extraction_tokens_out"] += sortie["tokens_out"]
    record["extraction_elapsed_s"] = round(
        record["extraction_elapsed_s"] + sortie["elapsed_s"], 3)
    record["extraction_raw_text"] = "\n\n".join(filter(None, [
        record.get("extraction_raw_text"),
        f"[{nom_strategie}] {sortie['text']}",
    ]))
    strategies = record.get("extraction_strategies") or []
    strategies.append({
        "nom": nom_strategie,
        "crop": crop,
        "statut_json": statut_json,
        "champs_ajoutes": nouveaux,
        "taux_apres": taux_remplissage(
            record["raw_data"], CHAMPS_ATTENDUS.get(record["doc_type"]) or []),
    })
    record["extraction_strategies"] = strategies
    return nouveaux


def _finaliser_record(record):
    champs = CHAMPS_ATTENDUS.get(record["doc_type"]) or []
    for cle in champs:
        record["raw_data"].setdefault(cle, None)
    record["extraction_taux_remplissage"] = taux_remplissage(record["raw_data"], champs)
    if record["extraction_status"] in ("NON_LANCEE", "OK", "PARTIELLE", "JSON_VIDE"):
        record["extraction_status"] = (
            "OK" if record["extraction_taux_remplissage"] >= SEUIL_REMPLISSAGE_OK
            else ("PARTIELLE" if any(
                v not in (None, "") for v in record["raw_data"].values())
                else "JSON_VIDE")
        )


def extraire_pages_standard(records, pdf_path):
    """
    Toutes les pages dactylographiées du dossier en UN SEUL generate.

    C'est le principal gain de temps de la V7.3 : V7.2 appelait
    ask_single page par page, soit 3 à 4 générations séquentielles là où
    une seule suffit.
    """
    cibles = [
        r for r in records
        if r["doc_type"] in STRATEGIES_EXTRACTION and not r.get("page_blanche")
    ]
    if not cibles:
        return records

    for start in range(0, len(cibles), GPU_BATCH_SIZE_EXTRACTION):
        lot = cibles[start:start + GPU_BATCH_SIZE_EXTRACTION]
        prompts = [PROMPTS_EXTRACTION[r["doc_type"]] for r in lot]
        images = [r["image"] for r in lot]
        sorties = ask_batch_multi(prompts, images, MAX_NEW_TOKENS_EXTRACTION)
        for record, sortie in zip(lot, sorties):
            _appliquer_sortie(record, sortie, "STANDARD")

    # --- Seconde passe, uniquement pour les pages restées incomplètes --
    a_reprendre = []
    for record in cibles:
        strategies = STRATEGIES_EXTRACTION.get(record["doc_type"]) or []
        if len(strategies) < 2:
            continue
        champs = CHAMPS_ATTENDUS.get(record["doc_type"]) or []
        if taux_remplissage(record["raw_data"], champs) >= SEUIL_REMPLISSAGE_OK:
            continue
        a_reprendre.append((record, strategies[1]))

    if a_reprendre:
        for start in range(0, len(a_reprendre), GPU_BATCH_SIZE_EXTRACTION):
            lot = a_reprendre[start:start + GPU_BATCH_SIZE_EXTRACTION]
            prompts = [PROMPTS_EXTRACTION[r["doc_type"]] for r, _ in lot]
            images = [
                render_page_region(pdf_path, r["page_num"] - 1,
                                   crop=strategie.get("crop"))
                for r, strategie in lot
            ]
            sorties = ask_batch_multi(prompts, images, MAX_NEW_TOKENS_EXTRACTION)
            for (record, strategie), sortie in zip(lot, sorties):
                _appliquer_sortie(record, sortie, strategie["nom"],
                                  crop=strategie.get("crop"))

    for record in cibles:
        _finaliser_record(record)
    return records


# =====================================================================
# V7.3 — Lecture multi-vues de la planche « permis de travail »
# =====================================================================

def extraire_planche(record, pdf_path):
    """
    Les vues de la planche sont TOUTES lues, en un seul generate, avec
    le processor haute définition.

    La fusion est priorisée et tracée : pour chaque champ on sait de
    quelle vue vient la valeur retenue, et si une autre vue proposait
    une valeur différente (conflit signalé, jamais masqué).
    """
    vues = VUES_PLANCHE.get(record["doc_type"]) or []
    if not vues:
        return record

    crops = crops_planche_permis(record["image"])
    record["frontiere_planche"] = crops["frontiere"]

    images, prompts, noms, max_tokens = [], [], [], []
    for vue in vues:
        images.append(render_page_region(
            pdf_path, record["page_num"] - 1,
            zoom=PDF_ZOOM_HAUTE_DEF,
            max_side=IMAGE_MAX_SIZE_HAUTE_DEF,
            crop=crops[vue["crop"]],
        ))
        prompts.append(vue["prompt"])
        noms.append(vue["nom"])
        max_tokens.append(vue["max_tokens"])

    sorties = []
    for start in range(0, len(images), GPU_BATCH_SIZE_VUES):
        lot_img = images[start:start + GPU_BATCH_SIZE_VUES]
        lot_prompt = prompts[start:start + GPU_BATCH_SIZE_VUES]
        lot_max = max(max_tokens[start:start + GPU_BATCH_SIZE_VUES])
        sorties.extend(ask_batch_multi(lot_prompt, lot_img, lot_max,
                                       proc=processor_hd))

    # --- Résultats par vue --------------------------------------------
    par_vue, detail_vues = {}, []
    for nom, vue, sortie, image in zip(noms, vues, sorties, images):
        parsed, statut_json = parse_json_response(sortie["text"], tracer=True)
        parsed = clean_raw_dict(parsed)
        par_vue[nom] = parsed
        detail_vues.append({
            "nom": nom,
            "crop": crops[vue["crop"]],
            "dimensions": f"{image.width}x{image.height}",
            "statut_json": statut_json,
            "champs_lus": sum(1 for v in parsed.values() if v not in (None, "")),
            "tokens_in": sortie["tokens_in"],
            "tokens_out": sortie["tokens_out"],
        })
        record["extraction_tokens_in"] += sortie["tokens_in"]
        record["extraction_tokens_out"] += sortie["tokens_out"]
        record["extraction_elapsed_s"] = round(
            record["extraction_elapsed_s"] + sortie["elapsed_s"], 3)
        record["extraction_raw_text"] = "\n\n".join(filter(None, [
            record.get("extraction_raw_text"), f"[{nom}] {sortie['text']}"]))

    # --- Fusion priorisée et tracée ------------------------------------
    fusion, provenance, conflits = {}, {}, []
    champs = CHAMPS_ATTENDUS.get(record["doc_type"]) or []
    for champ in champs:
        ordre = PRIORITE_VUES.get(champ, noms)
        valeurs = {
            nom: par_vue[nom][champ]
            for nom in ordre
            if nom in par_vue and par_vue[nom].get(champ) not in (None, "")
        }
        if not valeurs:
            fusion[champ] = None
            continue
        premiere_vue = next(iter(valeurs))
        fusion[champ] = valeurs[premiere_vue]
        provenance[champ] = premiere_vue
        distinctes = {str(v).strip().upper() for v in valeurs.values()}
        if len(distinctes) > 1:
            conflits.append({
                "champ": champ,
                "retenu": valeurs[premiere_vue],
                "vue_retenue": premiere_vue,
                "autres": {k: v for k, v in valeurs.items() if k != premiere_vue},
            })

    record["raw_data"] = fusion
    record["extraction_vues"] = detail_vues
    record["extraction_conflits"] = conflits or None
    record["extraction_strategies"] = [
        {"nom": v["nom"], "crop": v["crop"], "statut_json": v["statut_json"],
         "champs_ajoutes": v["champs_lus"], "taux_apres": None}
        for v in detail_vues
    ]
    record["provenance_champs"] = provenance
    _finaliser_record(record)
    return record


def extract_classified_pages(records, pdf_path):
    extraire_pages_standard(records, pdf_path)
    for record in records:
        if record.get("page_blanche"):
            record["extraction_status"] = "PAGE_BLANCHE"
            continue
        if record["doc_type"] in VUES_PLANCHE:
            try:
                extraire_planche(record, pdf_path)
            except Exception as exc:
                record["extraction_status"] = "ERREUR"
                record["extraction_error"] = repr(exc)
        elif record["doc_type"] not in PROMPTS_EXTRACTION:
            record["extraction_status"] = "NON_APPLICABLE"
    return records


# =====================================================================
# V7.3 — Relecture ciblée déclenchée par un contrôle en échec
# =====================================================================

def relecture_ciblee(records, controles, pdf_path):
    """
    Relit UNIQUEMENT les blocs mis en cause par un contrôle arithmétique
    en échec, avec un prompt réduit à ces champs.

    Les valeurs relues ne remplacent les valeurs initiales que si elles
    rétablissent la cohérence : on ne substitue jamais une lecture à une
    autre sans justification. L'ancienne valeur est conservée dans
    RELECTURE_VALEUR_INITIALE.
    """
    zones = zones_a_relire(controles)
    if not zones:
        return []

    par_type = {}
    for record in records:
        par_type.setdefault(record["doc_type"], record)

    demandes = []
    for doc_type, zone in zones:
        record = par_type.get(doc_type)
        recadrage = RECADRAGES_RELECTURE.get((doc_type, zone))
        if record is None or recadrage is None:
            continue
        demandes.append((record, doc_type, zone, recadrage))

    if not demandes:
        return []

    images = [
        render_page_region(pdf_path, r["page_num"] - 1,
                           zoom=PDF_ZOOM_HAUTE_DEF,
                           max_side=IMAGE_MAX_SIZE_HAUTE_DEF,
                           crop=rec["crop"])
        for r, _, _, rec in demandes
    ]
    prompts = [rec["prompt"] for _, _, _, rec in demandes]
    sorties = ask_batch_multi(prompts, images, MAX_NEW_TOKENS_RELECTURE,
                              proc=processor_hd)

    relectures = []
    for (record, doc_type, zone, _), sortie in zip(demandes, sorties):
        parsed = clean_raw_dict(parse_json_response(sortie["text"]))
        record["extraction_tokens_in"] += sortie["tokens_in"]
        record["extraction_tokens_out"] += sortie["tokens_out"]
        for champ, valeur in parsed.items():
            if valeur in (None, ""):
                continue
            ancienne = record["raw_data"].get(champ)
            relectures.append({
                "TYPE_DOCUMENT": doc_type,
                "ZONE": zone,
                "CHAMP": champ,
                "VALEUR_INITIALE": ancienne,
                "VALEUR_RELUE": valeur,
                "IDENTIQUE": str(ancienne or "").strip() == str(valeur).strip(),
            })
    return relectures


def _appliquer_relectures(row, champs, relectures):
    """
    Teste la substitution : si remplacer la valeur initiale par la valeur
    relue fait passer les contrôles, on garde la relecture. Sinon on
    conserve la lecture initiale et on laisse l'anomalie visible.
    """
    divergentes = [r for r in relectures if not r["IDENTIQUE"]]
    if not divergentes:
        return row, champs, False

    candidat = dict(row)
    for r in divergentes:
        candidat[r["CHAMP"]] = r["VALEUR_RELUE"]

    champs_candidat = typer_champs(candidat)
    ctrl_avant = synthese_qualite(controles_dossier(champs), champs)
    ctrl_apres = synthese_qualite(controles_dossier(champs_candidat), champs_candidat)

    if ctrl_apres["NB_ANOMALIES_BLOQUANTES"] < ctrl_avant["NB_ANOMALIES_BLOQUANTES"]:
        for r in divergentes:
            candidat[r["CHAMP"] + "_VALEUR_INITIALE"] = r["VALEUR_INITIALE"]
        candidat["RELECTURE_APPLIQUEE"] = True
        return candidat, champs_candidat, True

    row["RELECTURE_APPLIQUEE"] = False
    return row, champs, False


# =====================================================================
# V7.3 — Typage du dossier consolidé
# =====================================================================

def appliquer_typage_row(row, champs):
    """
    Réécrit la ligne dossier avec les valeurs typées.

      <CHAMP>        valeur typée (float, date ISO AAAA-MM-JJ, entier)
      <CHAMP>_RAW    chaîne exactement telle que lue sur le document
      <CHAMP>_STATUT OK / AMBIGU / ILLISIBLE / HORS_BORNES / ABSENT

    Les dates passent en ISO : c'est le seul format non ambigu pour
    Alteryx et pour Excel, et il rend « 01/02/2026 » interprétable sans
    convention implicite.
    """
    for nom, entree in (champs or {}).items():
        row[nom + "_RAW"] = entree.get("raw")
        row[nom] = entree.get("iso") if entree.get("type") == "date" else entree.get("valeur")
        row[nom + "_STATUT"] = entree.get("statut")
        if entree.get("type") == "compte" and entree.get("cle_valide") is not None:
            row[nom + "_CLE_VALIDE"] = entree["cle_valide"]
    return row


def appliquer_consolidation_row(row, consolides):
    for cle, bloc in (consolides or {}).items():
        row[f"CONSO_{cle}"] = bloc["valeur"]
        row[f"CONSO_{cle}_CONCORDANCE"] = bloc["concordance"]
        row[f"CONSO_{cle}_CONFIANCE"] = bloc["confiance"]
        row[f"CONSO_{cle}_SOURCES"] = json.dumps(
            bloc["sources"], ensure_ascii=False, default=str) or None
    return row


# =====================================================================
# V7.3 — Traitement complet d'un dossier
# =====================================================================

def process_pdf(pdf_path, verbose=True):
    t0 = time.time()
    if verbose:
        log(f"📁 {pdf_path.name}")

    try:
        pages = pdf_to_pages(pdf_path)
        if not pages:
            raise ValueError(f"Aucune page détectée dans {pdf_path.name}")

        records = classify_pages(pages)
        records = extract_classified_pages(records, pdf_path)

        page_rows = [build_page_row(pdf_path.name, record) for record in records]
        dossier_row = consolidate_dossier(pdf_path.name, records)

        # --- Typage, consolidation multi-sources, contrôles ------------
        champs = typer_champs(dossier_row)
        consolides = consolider_dossier_champs(champs)
        controles = controles_dossier(champs, consolides)

        # --- Relecture ciblée si un contrôle bloquant a échoué ---------
        relectures = []
        if RELECTURE_CIBLEE_ACTIVE and any(
            c["STATUT"] == "ANOMALIE" and c["NIVEAU"] == NIVEAU_BLOQUANT
            for c in controles
        ):
            relectures = relecture_ciblee(records, controles, pdf_path)
            dossier_row, champs, applique = _appliquer_relectures(
                dossier_row, champs, relectures)
            if applique:
                consolides = consolider_dossier_champs(champs)
                controles = controles_dossier(champs, consolides)

        qualite = synthese_qualite(controles, champs)
        dossier_row = appliquer_typage_row(dossier_row, champs)
        dossier_row = appliquer_consolidation_row(dossier_row, consolides)
        dossier_row.update(qualite)

        planning = []
        tokens_in = sum(r.get("classification_tokens_in", 0)
                        + r.get("extraction_tokens_in", 0) for r in records)
        tokens_out = sum(r.get("classification_tokens_out", 0)
                         + r.get("extraction_tokens_out", 0) for r in records)
        elapsed_total = round(time.time() - t0, 3)

        dossier_row.update({
            "STATUT_TRAITEMENT_PIPELINE": "TRAITE_NOUVEAU",
            "TEMPS_ECOULE_DOSSIER_S": round(elapsed_total, 2),
            "TOKENS_IN_DOSSIER": int(tokens_in),
            "TOKENS_OUT_DOSSIER": int(tokens_out),
            "TOKENS_TOTAL_DOSSIER": int(tokens_in + tokens_out),
            "NB_RELECTURES_CIBLEES": len(relectures),
        })

        dossier = {
            "source_file": pdf_path.name,
            "source_sha256": sha256_file(pdf_path),
            "pipeline_version": PIPELINE_VERSION,
            "stats": {
                "pages": len(pages),
                "pages_blanches": sum(1 for p in pages if p.get("blanche")),
                "tokens_in": tokens_in,
                "tokens_out": tokens_out,
                "tokens_total": tokens_in + tokens_out,
                "elapsed_s": elapsed_total,
                "relectures_ciblees": len(relectures),
            },
            # Valeurs typées, avec brut et statut : c'est la sortie de
            # référence pour toute reprise automatisée.
            "champs_normalises": {
                nom: {
                    "valeur": e.get("iso") if e.get("type") == "date" else e.get("valeur"),
                    "raw": e.get("raw"),
                    "type": e.get("type"),
                    "statut": e.get("statut"),
                }
                for nom, e in champs.items()
            },
            "consolidation": consolides,
            "controles": controles,
            "qualite": qualite,
            "relectures": relectures,
            "page_records": [
                {k: v for k, v in record.items() if k != "image"}
                for record in records
            ],
            "page_rows": page_rows,
            "dossier_row": dossier_row,
            "planning_tl": planning,
        }

        checkpoint = canonical_checkpoint_path(pdf_path)
        with open(checkpoint, "w", encoding="utf-8") as f:
            json.dump(dossier, f, ensure_ascii=False, indent=2, default=str)

        if verbose:
            log(f"   {qualite['QUALITE_DOSSIER']} | "
                f"{qualite['NB_CONTROLES_OK']}/{qualite['NB_CONTROLES']} contrôles OK | "
                f"{tokens_in + tokens_out} tokens | {elapsed_total:.1f}s")

        return dossier
    finally:
        # Le cache de rendu haute définition est propre à un dossier.
        vider_cache_hd()


print("✅ Classification V7.3 (basse résolution + reclassement ciblé) OK")
print("✅ Extraction V7.3 (batch multi-prompts + lecture multi-vues de la planche) OK")
print("✅ Relecture ciblée pilotée par les contrôles OK")


## 12. Export Excel compatible Alteryx

In [ ]:

def ordered_columns(rows):
    preferred = [
        "FICHIER", "NB_PAGES", "TYPES_DOCUMENTS",
        "STATUT_TRAITEMENT_PIPELINE", "TEMPS_ECOULE_DOSSIER_S",
        "TOKENS_IN_DOSSIER", "TOKENS_OUT_DOSSIER", "TOKENS_TOTAL_DOSSIER",
        "REFERENCE_DOM_EXTRAITE_RAW", "REFERENCE_DOM_EXTRAITE_NORMALISEE",
        "NUMERO_DOM_RETENU", "DATE_DOM_RETENUE",
            "REFERENCE_PREDOM_RETENUE", "DATE_DEBUT_CONTRAT_PREDOM",
            "DATE_FIN_CONTRAT_PREDOM", "PREDOM_TROUVEE", "NUMERO_CLIENT_RETENU",
        "NOM_CLIENT_REFERENCE", "NUMERO_CONTRAT_REFERENCE",
        "DATE_DEBUT_CONTRAT_REFERENCE", "DATE_FIN_CONTRAT_REFERENCE",
        "NUMERO_PERMIS_REFERENCE", "MATCH_SOURCE", "MATCH_METHOD",
        "MATCH_SCORE", "MATCH_STATUS", "MATCH_CANDIDATES_COUNT",
        "REFERENCE_PREDOM_RETENUE", "DATE_DEBUT_CONTRAT_PREDOM",
        "DATE_FIN_CONTRAT_PREDOM", "PREDOM_TROUVEE",
        # --- V7.1 : augmentation de salaire ---
        "SALAIRE_ANCIEN", "SALAIRE_NOUVEAU_AUGMENTE",
        "AUGMENTATION_DETECTEE", "SENS_VARIATION_SALAIRE",
        "MONTANT_AUGMENTATION", "TAUX_AUGMENTATION_PCT",
        "SOURCE_AUGMENTATION", "CTS_LIGNE_SALAIRE_BRUTE",
        "ECART_SALAIRE_DOM_CONTRAT", "COHERENCE_SALAIRE_DOM_CONTRAT",
        "ALERTE_DOM_SUR_ANCIEN_SALAIRE",
        # --- V7.1 : identité et permis de travail ---
        "NOM_TRAVAILLEUR_REFERENCE", "DATE_NAISSANCE_REFERENCE",
        "NATIONALITE_REFERENCE", "DATE_ENTREE_ALGERIE",
        "PERMIS_TRAVAIL_LU",
        # --- V7.3 : qualité et contrôles ---
        "QUALITE_DOSSIER", "NB_ANOMALIES", "NB_ANOMALIES_BLOQUANTES",
        "CODES_ANOMALIES", "NB_CONTROLES", "NB_CONTROLES_OK",
        "NB_CHAMPS_RENSEIGNES", "NB_CHAMPS_ATTENDUS",
        "NB_RELECTURES_CIBLEES",
    ]
    cols = set().union(*(r.keys() for r in rows)) if rows else set()
    return [c for c in preferred if c in cols] + sorted(cols - set(preferred))

def sheet_from_rows(wb, title, rows, columns=None, amount_columns=None):
    ws = wb.create_sheet(title)
    columns = columns or ordered_columns(rows)
    if not columns:
        ws["A1"] = "Aucune donnée"
        return
    amount_columns = set(amount_columns or [])
    fill = PatternFill("solid", fgColor="1F4E78")
    font = Font(color="FFFFFF", bold=True, name="Arial", size=9)
    for j, name in enumerate(columns, 1):
        c = ws.cell(1, j, name); c.fill = fill; c.font = font
        c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    for i, row in enumerate(rows, 2):
        for j, name in enumerate(columns, 1):
            value = row.get(name)
            if isinstance(value, (dict, list)):
                value = json.dumps(value, ensure_ascii=False)
            c = ws.cell(i, j, value)
            if name in amount_columns and isinstance(value, (int, float)):
                c.number_format = EXCEL_AMOUNT_FORMAT
    ws.freeze_panes = "A2"; ws.auto_filter.ref = ws.dimensions
    for j, name in enumerate(columns, 1):
        ws.column_dimensions[get_column_letter(j)].width = 38 if "ADRESSE" in name else min(34, max(14, len(name) + 2))

def build_long_raw_rows(dossiers):
    rows = []
    for d in dossiers:
        for page in d.get("page_records", []):
            for field, value in (page.get("raw_data") or {}).items():
                rows.append({
                    "FICHIER": d.get("source_file"), "PAGE": page.get("page_num"),
                    "TYPE_DOCUMENT": page.get("doc_type"), "CHAMP": field,
                    "VALEUR_BRUTE": value,
                    "CONFIANCE_CLASSIFICATION": page.get("classification_confidence"),
                    "STATUT_EXTRACTION": page.get("extraction_status"),
                })
    return rows

COLONNES_MONTANT_V71 = {
    "SALAIRE_ANCIEN", "SALAIRE_NOUVEAU_AUGMENTE",
    "MONTANT_AUGMENTATION", "ECART_SALAIRE_DOM_CONTRAT",
}


def create_excel(excel_path, dossiers, errors, reference_df=None):
    pages, dossier_rows, planning, matching = [], [], [], []
    for d in dossiers:
        pages.extend(d.get("page_rows", []))
        row = d.get("dossier_row") or {}
        dossier_rows.append(row)
        planning.extend(month_segment_rows(row))
        matching.append({k: row.get(k) for k in [
            "FICHIER", "STATUT_TRAITEMENT_PIPELINE",
            "TEMPS_ECOULE_DOSSIER_S", "TOKENS_IN_DOSSIER",
            "TOKENS_OUT_DOSSIER", "TOKENS_TOTAL_DOSSIER",
            "NOM_CLIENT_REFERENCE", "NUMERO_CONTRAT_REFERENCE",
            "DATE_DEBUT_CONTRAT_REFERENCE", "DATE_FIN_CONTRAT_REFERENCE",
            "REFERENCE_DOM_EXTRAITE_RAW", "REFERENCE_DOM_EXTRAITE_NORMALISEE",
            "MATCH_SOURCE", "MATCH_METHOD", "MATCH_SCORE", "MATCH_STATUS",
            "MATCH_CANDIDATES_COUNT", "NUMERO_CLIENT_RETENU",
            "NUMERO_DOM_RETENU", "DATE_DOM_RETENUE",
            "REFERENCE_PREDOM_RETENUE", "DATE_DEBUT_CONTRAT_PREDOM",
            "DATE_FIN_CONTRAT_PREDOM", "PREDOM_TROUVEE",
            "AUGMENTATION_DETECTEE", "ALERTE_DOM_SUR_ANCIEN_SALAIRE",
        ]})
    raw = build_long_raw_rows(dossiers)
    suivi = []
    for d in dossiers:
        stats = d.get("stats") or {}
        row = d.get("dossier_row") or {}
        suivi.append({
            "FICHIER": d.get("source_file"),
            "STATUT_TRAITEMENT_PIPELINE": row.get("STATUT_TRAITEMENT_PIPELINE"),
            "NB_PAGES": stats.get("pages"),
            "TEMPS_ECOULE_DOSSIER_S": row.get("TEMPS_ECOULE_DOSSIER_S"),
            "TOKENS_IN_DOSSIER": row.get("TOKENS_IN_DOSSIER"),
            "TOKENS_OUT_DOSSIER": row.get("TOKENS_OUT_DOSSIER"),
            "TOKENS_TOTAL_DOSSIER": row.get("TOKENS_TOTAL_DOSSIER"),
            "PIPELINE_VERSION_JSON": d.get("pipeline_version"),
            "SHA256": d.get("source_sha256"),
        })

    wb = Workbook(); wb.remove(wb.active)
    sheet_from_rows(wb, "DOSSIERS_DOMICILIATION", dossier_rows,
                    amount_columns=AMOUNT_FIELDS | COLONNES_MONTANT_V71)
    sheet_from_rows(wb, "SUIVI_TRAITEMENT", suivi, [
        "FICHIER", "STATUT_TRAITEMENT_PIPELINE", "NB_PAGES",
        "TEMPS_ECOULE_DOSSIER_S", "TOKENS_IN_DOSSIER",
        "TOKENS_OUT_DOSSIER", "TOKENS_TOTAL_DOSSIER",
        "PIPELINE_VERSION_JSON", "SHA256",
    ], amount_columns={"TEMPS_ECOULE_DOSSIER_S"})
    sheet_from_rows(wb, "PLANNING_TL", planning, amount_columns={
        "SALAIRE_NET_REFERENCE", "PLAFOND_MENSUEL_REFERENCE",
        "MONTANT_MAX_THEORIQUE", "MONTANT_AUTORISE_SAISI",
        "MONTANT_TRANSFERE", "SOLDE_RESTANT", "NB_JOURS_SEGMENT", "NB_JOURS_MOIS", "COEFFICIENT_PRORATA",
        "SALAIRE_ANCIEN", "SALAIRE_NOUVEAU_AUGMENTE", "MONTANT_AUGMENTATION",
    })
    sheet_from_rows(wb, "PAGES_DOCUMENTS", pages)
    sheet_from_rows(wb, "EXTRACTION_BRUTE", raw, [
        "FICHIER", "PAGE", "TYPE_DOCUMENT", "CHAMP", "VALEUR_BRUTE",
        "CONFIANCE_CLASSIFICATION", "STATUT_EXTRACTION",
    ])
    sheet_from_rows(wb, "MATCHING_DOM", matching)
    # --- V7.3 : onglets qualité -------------------------------------
    controles_rows, relectures_rows = [], []
    for d in dossiers:
        for c in (d.get("controles") or []):
            ligne = {"FICHIER": d.get("source_file")}
            ligne.update(c)
            controles_rows.append(ligne)
        for r in (d.get("relectures") or []):
            ligne = {"FICHIER": d.get("source_file")}
            ligne.update(r)
            relectures_rows.append(ligne)

    sheet_from_rows(wb, "CONTROLES", controles_rows, [
        "FICHIER", "CODE", "LIBELLE", "NIVEAU", "STATUT",
        "ATTENDU", "OBTENU", "ECART", "CHAMPS", "COMMENTAIRE",
    ])
    sheet_from_rows(wb, "ANOMALIES", [
        c for c in controles_rows if c.get("STATUT") == "ANOMALIE"
    ], [
        "FICHIER", "CODE", "LIBELLE", "NIVEAU",
        "ATTENDU", "OBTENU", "ECART", "CHAMPS", "COMMENTAIRE",
    ])
    if relectures_rows:
        sheet_from_rows(wb, "RELECTURES_CIBLEES", relectures_rows, [
            "FICHIER", "TYPE_DOCUMENT", "ZONE", "CHAMP",
            "VALEUR_INITIALE", "VALEUR_RELUE", "IDENTIQUE",
        ])
    sheet_from_rows(wb, "ERREURS", errors)
    if reference_df is not None:
        sheet_from_rows(wb, "REFERENTIEL_DOM_PREDOM", reference_df.where(pd.notna(reference_df), None).to_dict("records"))
    wb.save(excel_path)
    print(f"✅ Excel créé : {excel_path} | dossiers={len(dossier_rows)} | planning={len(planning)}")

print("✅ Export V7.3 chargé : onglets CONTROLES, ANOMALIES, RELECTURES_CIBLEES")


In [ ]:

print(f"Nombre de PDF détectés : {len(pdfs)}")
if not pdfs:
    raise RuntimeError(
        f"Aucun PDF trouvé dans {INPUT_DIR}. "
        "Déposer les dossiers de domiciliation dans ce répertoire."
    )

diagnostic_errors = []
total_pages = 0

for p in pdfs:
    try:
        with fitz.open(str(p)) as doc:
            page_count = int(doc.page_count)
        total_pages += page_count
        print(
            f"{p.name} -> pages={page_count}, "
            f"taille={p.stat().st_size:,} octets"
        )
        if page_count <= 0:
            diagnostic_errors.append(f"{p.name}: 0 page")
    except Exception as exc:
        diagnostic_errors.append(f"{p.name}: {exc!r}")

if diagnostic_errors:
    raise RuntimeError(
        "Diagnostic PDF en erreur :\n- " + "\n- ".join(diagnostic_errors)
    )

print(f"✅ Diagnostic PDF : {len(pdfs)} fichier(s), {total_pages} page(s)")

# Nettoyage des checkpoints invalides.
# Les anciens checkpoints valides suffixés par hash seront migrés
# au moment de leur chargement vers un seul JSON canonique par PDF.
removed = 0
for json_file in JSON_DIR.glob("*.json"):
    try:
        dossier = json.loads(json_file.read_text(encoding="utf-8"))
        pages_checkpoint = int(dossier.get("stats", {}).get("pages", 0) or 0)
        records_checkpoint = dossier.get("page_records") or []
        if pages_checkpoint <= 0 or not records_checkpoint:
            json_file.unlink()
            removed += 1
            print(f"Checkpoint vide supprimé : {json_file.name}")
    except Exception:
        json_file.unlink()
        removed += 1
        print(f"Checkpoint illisible supprimé : {json_file.name}")

print(f"Checkpoints invalides supprimés : {removed}")


In [ ]:

# Test technique sur le premier PDF avant le traitement complet.
_test_pdf = pdfs[0]
_test_pages = pdf_to_pages(_test_pdf)

print(
    f"Test conversion : {_test_pdf.name} -> "
    f"{len(_test_pages)} page(s)"
)
print(
    "Dimensions première page :",
    _test_pages[0]["width"],
    "x",
    _test_pages[0]["height"],
)
print(
    "Ratio blanc première page :",
    _test_pages[0]["white_ratio"],
)

if len(_test_pages) == 0:
    raise RuntimeError("Le test de conversion PDF a retourné zéro page.")

# Libération immédiate des images du test.
del _test_pages
gc.collect()
print("✅ Test de conversion PDF réussi")


## 13. Exécution complète avec reprise automatique

In [ ]:

# Tests techniques du matching sécurisé V6.3
_test_ref = pd.DataFrame([
    {
        "numero_dom": "271901202614000119DZD",
        "numero_dom_normalise": "271901202614000119DZD",
        "date_dom": "15/01/2026",
        "numero_client": "CL001",
        "reference": "DOM-001",
        "reference_predom": "PREDOM-001",
        "date_debut_predom": "12/06/2025",
        "date_fin_predom": "11/06/2026",
        "date_debut_match": date(2025, 6, 12),
        "date_fin_match": date(2026, 6, 11),
        "DOM_MATCH_COUNT": 1,
        "PREDOM_MATCH_COUNT": 1,
    }
])

# Numéro DOM absent + période exacte unique : attribution autorisée
_test_row_unique = {
    "REFERENCE_DOM_EXTRAITE_NORMALISEE": None,
    "DATE_DEBUT_CONTRAT_REFERENCE": "12/06/2025",
    "DATE_FIN_CONTRAT_REFERENCE": "11/06/2026",
}
_test_match_unique = match_dossier(_test_row_unique, _test_ref)
assert _test_match_unique["MATCH_STATUS"] == "MATCH_PERIODE_EXACTE"
assert _test_match_unique["NUMERO_DOM_RETENU"] == "271901202614000119DZD"

# Aucun match : aucun numéro DOM ne doit être renseigné
_test_row_none = {
    "REFERENCE_DOM_EXTRAITE_NORMALISEE": None,
    "DATE_DEBUT_CONTRAT_REFERENCE": "01/01/2030",
    "DATE_FIN_CONTRAT_REFERENCE": "31/12/2030",
}
_test_match_none = match_dossier(_test_row_none, _test_ref)
assert _test_match_none["MATCH_STATUS"] == "AUCUN_MATCH_PERIODE"
assert _test_match_none["NUMERO_DOM_RETENU"] is None

# Plusieurs candidats : aucun numéro DOM ne doit être renseigné
_test_ref_multiple = pd.concat([_test_ref, _test_ref.assign(
    numero_dom="271901202624000120DZD",
    numero_dom_normalise="271901202624000120DZD",
)], ignore_index=True)
_test_match_multiple = match_dossier(_test_row_unique, _test_ref_multiple)
assert _test_match_multiple["MATCH_STATUS"] == "PLUSIEURS_CANDIDATS_PERIODE"
assert _test_match_multiple["NUMERO_DOM_RETENU"] is None

print("✅ Tests matching sécurisé V6.3 réussis")


In [ ]:

# Tests techniques V6.4 : mois complet sans P1/P2 et date DOM dans Planning_TL
_test_full_month = {
    "FICHIER": "demo.pdf",
    "NUMERO_DOM_RETENU": "271901202614000119DZD",
    "DATE_DOM_RETENUE": "15/01/2026",
    "NUMERO_CLIENT_RETENU": "CL001",
    "NOM_CLIENT_REFERENCE": "CLIENT TEST",
    "NUMERO_CONTRAT_REFERENCE": "CTR001",
    "DATE_DEBUT_CONTRAT_REFERENCE": "01/05/2026",
    "DATE_FIN_CONTRAT_REFERENCE": "31/05/2026",
    "NUMERO_PERMIS_REFERENCE": "PT001",
    "DOM_PART_TRANSFERABLE": 100000.00,
    "DOM_SALAIRE_NET_MENSUEL": 120000.00,
    "DOM_TAUX_TRANSFERABLE": "80%",
    "MATCH_STATUS": "DOMICILIATION_TROUVEE",
    "MATCH_METHOD": "NUMERO_DOM_EXACT",
    "MATCH_SCORE": 100.00,
    "MATCH_CANDIDATES_COUNT": 1,
    "REFERENCE_PREDOM_RETENUE": "PREDOM001",
}

_test_rows = month_segment_rows(_test_full_month)
assert len(_test_rows) == 1
assert _test_rows[0]["PERIODE_TL"] == "2026-05"
assert _test_rows[0]["PARTIE"] is None
assert _test_rows[0]["COEFFICIENT_PRORATA"] == 1.00
assert _test_rows[0]["DATE_DOMICILIATION"] == "15/01/2026"
assert _test_rows[0]["DATE_DOMICILIATION_SOURCE"] == "FICHIER_DOM"

print("✅ Tests V6.4 réussis : mois complet sans P1/P2 + date DOM dans Planning_TL")


In [ ]:

# Tests techniques V6.5 : un seul JSON par PDF et statistiques dossier
assert canonical_checkpoint_path(Path("07000-670909-202605-DOM.pdf")).name == \
       "07000-670909-202605-DOM.json"

_test_stats_dossier = {
    "dossier_row": {"FICHIER": "demo.pdf"},
    "stats": {
        "elapsed_s": 12.345,
        "tokens_in": 1000,
        "tokens_out": 250,
        "tokens_total": 1250,
    },
}
_test_stats_dossier = enrich_dossier_row_with_stats(
    _test_stats_dossier,
    "REPRIS_JSON_EXISTANT",
)
_test_row = _test_stats_dossier["dossier_row"]

assert _test_row["TEMPS_ECOULE_DOSSIER_S"] == 12.35
assert _test_row["TOKENS_IN_DOSSIER"] == 1000
assert _test_row["TOKENS_OUT_DOSSIER"] == 250
assert _test_row["TOKENS_TOTAL_DOSSIER"] == 1250
assert _test_row["STATUT_TRAITEMENT_PIPELINE"] == "REPRIS_JSON_EXISTANT"

print("✅ Tests V6.5 réussis : reprise JSON + temps + tokens")


In [ ]:

# Tests techniques V6.7 : suivi compact Domino
assert format_duration(65) == "01:05"
assert format_duration(3661) == "01:01:01"

print("✅ Tests V6.7 réussis : suivi compact Domino")


In [ ]:

# =====================================================================
# Tests techniques V7.1 (adaptés V7.3)
# =====================================================================

# --- 1. Décomposition de la ligne de salaire ------------------------

_cas = split_ligne_salaire(
    "Salaire mensuel de base net :  506,471.38  au lieu de  479,274.29"
)
assert _cas["nouveau"] == 506471.38, _cas
assert _cas["ancien"] == 479274.29, _cas
assert _cas["mention_presente"] is True

# Variante avec espaces insécables et virgule décimale
_cas_fr = split_ligne_salaire(
    "Salaire mensuel de base net : 506 471,38 au lieu de 479 274,29"
)
assert _cas_fr["nouveau"] == 506471.38, _cas_fr
assert _cas_fr["ancien"] == 479274.29, _cas_fr

# Sans augmentation : un seul montant, aucun ancien salaire
_cas_simple = split_ligne_salaire("Salaire mensuel de base net : 466,300.88")
assert _cas_simple["nouveau"] == 466300.88, _cas_simple
assert _cas_simple["ancien"] is None
assert _cas_simple["mention_presente"] is False

# Ligne absente
assert split_ligne_salaire(None)["ancien"] is None

print("✅ Décomposition de la ligne de salaire OK")

# --- 2. Consolidation de l'augmentation -----------------------------

_rec_augmentation = [{
    "page_num": 1,
    "doc_type": "CONTRAT_SPECIFIQUE",
    "raw_data": {
        "CTS_SALAIRE_NET": "506,471.38",
        "CTS_SALAIRE_NET_ANCIEN": "479,274.29",
        "CTS_LIGNE_SALAIRE_BRUTE":
            "Salaire mensuel de base net : 506,471.38 au lieu de 479,274.29",
        "CTS_NOM_PRENOM_TRAVAILLEUR": "YILDIRIM IBRAHIM",
    },
}, {
    "page_num": 2,
    "doc_type": "ENGAGEMENT_DOMICILIATION",
    "raw_data": {
        "DOM_SALAIRE_NET_MENSUEL": "506471,38",
        "DOM_PART_TRANSFERABLE": "481 147.81",
    },
}]

_row = consolidate_dossier("test_augmentation.pdf", _rec_augmentation)

assert _row["SALAIRE_NOUVEAU_AUGMENTE"] == 506471.38, _row["SALAIRE_NOUVEAU_AUGMENTE"]
assert _row["SALAIRE_ANCIEN"] == 479274.29, _row["SALAIRE_ANCIEN"]
assert _row["AUGMENTATION_DETECTEE"] is True
assert _row["SENS_VARIATION_SALAIRE"] == "AUGMENTATION"
assert _row["MONTANT_AUGMENTATION"] == 27197.09, _row["MONTANT_AUGMENTATION"]
assert _row["TAUX_AUGMENTATION_PCT"] == 5.67, _row["TAUX_AUGMENTATION_PCT"]
assert _row["COHERENCE_SALAIRE_DOM_CONTRAT"] is True
assert _row["ALERTE_DOM_SUR_ANCIEN_SALAIRE"] is False

print(
    "✅ Augmentation détectée : "
    f"{_row['SALAIRE_ANCIEN']} -> {_row['SALAIRE_NOUVEAU_AUGMENTE']} "
    f"(+{_row['MONTANT_AUGMENTATION']} / +{_row['TAUX_AUGMENTATION_PCT']}%)"
)

# --- 3. Alerte : domiciliation restée sur l'ancien salaire -----------

_rec_alerte = [{
    "page_num": 1,
    "doc_type": "CONTRAT_SPECIFIQUE",
    "raw_data": {
        "CTS_SALAIRE_NET": "506,471.38",
        "CTS_SALAIRE_NET_ANCIEN": "479,274.29",
    },
}, {
    "page_num": 2,
    "doc_type": "ENGAGEMENT_DOMICILIATION",
    "raw_data": {"DOM_SALAIRE_NET_MENSUEL": "479 274,29"},
}]

_row_alerte = consolidate_dossier("test_alerte.pdf", _rec_alerte)
assert _row_alerte["ALERTE_DOM_SUR_ANCIEN_SALAIRE"] is True
assert _row_alerte["COHERENCE_SALAIRE_DOM_CONTRAT"] is False

print("✅ Alerte domiciliation sur ancien salaire OK")

# --- 4. Relecture de secours quand le modèle n'isole pas les montants -

_rec_secours = [{
    "page_num": 1,
    "doc_type": "CONTRAT_SPECIFIQUE",
    "raw_data": {
        "CTS_SALAIRE_NET": "506,471.38",
        "CTS_SALAIRE_NET_ANCIEN": None,
        "CTS_LIGNE_SALAIRE_BRUTE":
            "Salaire mensuel de base net : 506,471.38 au lieu de 479,274.29",
    },
}]

_row_secours = consolidate_dossier("test_secours.pdf", _rec_secours)
assert _row_secours["SALAIRE_ANCIEN"] == 479274.29, _row_secours["SALAIRE_ANCIEN"]
assert _row_secours["SOURCE_AUGMENTATION"] == "RELECTURE_LIGNE_BRUTE"

print("✅ Relecture de secours depuis la ligne brute OK")

# --- 5. Champs attendus et stratégies -------------------------------

assert "CTS_SALAIRE_NET_ANCIEN" in CHAMPS_ATTENDUS["CONTRAT_SPECIFIQUE"]
assert "TTR_NOM" in CHAMPS_ATTENDUS["TITRE_TRAVAIL"]
assert "TTR_DATE_ENTREE_ALGERIE" in CHAMPS_ATTENDUS["TITRE_TRAVAIL"]
assert "TTR_NUMERO_PERMIS" in CHAMPS_ATTENDUS["TITRE_TRAVAIL"]
assert len(VUES_PLANCHE["TITRE_TRAVAIL"]) == 4

# Taux de remplissage
assert taux_remplissage({"A": 1, "B": None}, ["A", "B"]) == 0.5
assert taux_remplissage({}, []) == 1.0

print("✅ Champs attendus et stratégies d'escalade OK")

# --- 6. Recadrage ----------------------------------------------------

_img_test = Image.new("RGB", (1000, 2000), "white")
_haut = crop_region(_img_test, 0.00, 0.58)
assert _haut.size == (1000, 1160), _haut.size

_droite = crop_region(_img_test, 0.00, 0.58, 0.44, 1.00)
assert _droite.size == (560, 1160), _droite.size

print("✅ Recadrage par fractions OK")

# --- 7. Détection automatique de la frontière de planche --------------

# Planche synthétique : bloc haut, bande blanche, bloc bas.
_planche = Image.new("L", (800, 2000), 255)
_px = _planche.load()
for y in list(range(60, 880)) + list(range(1120, 1940)):
    for x in range(40, 760, 3):
        _px[x, y] = 0

_frontiere = detecter_frontiere_documents(_planche.convert("RGB"))
assert 0.44 <= _frontiere <= 0.52, _frontiere

_crops = crops_planche_permis(_planche.convert("RGB"))
assert _crops["titre"][0] == 0.0
assert _crops["titre"][1] > _crops["couverture"][0], "les blocs doivent se chevaucher"
assert _crops["colonne_identite"][2] == 0.44
assert _crops["couverture"][1] == 1.0

print(f"✅ Frontière détectée sur planche synthétique : {_frontiere}")

# Repli sur une image sans bande blanche franche
assert detecter_frontiere_documents(Image.new("RGB", (800, 1200), "white")) == FRONTIERE_DEFAUT
print(f"✅ Repli sur image sans coupure nette : {FRONTIERE_DEFAUT}")

# --- 8. Stratégies dynamiques ----------------------------------------

# V7.3 : les recadrages de la planche sont portés par VUES_PLANCHE,
# qui remplace l'escalade conditionnelle par des vues systématiques.
_cles_dynamiques = {
    v["crop"]
    for v in VUES_PLANCHE["TITRE_TRAVAIL"]
}
assert _cles_dynamiques <= set(_crops), _cles_dynamiques
assert "colonne_identite" in _cles_dynamiques
assert "entete" in _cles_dynamiques
assert VUES_PLANCHE["PERMIS_TRAVAIL_COUVERTURE"][0]["crop"] == "couverture"

print("✅ Stratégies dynamiques cohérentes avec les recadrages")
print("\n✅ Tous les tests V7.1 sont passés")


### Tests techniques V7.3


In [ ]:
# =====================================================================
# Tests techniques V7.3 — à exécuter avant tout traitement de lot
# =====================================================================

# --- 1. Normalisation des montants -----------------------------------
# Les trois conventions rencontrées dans les dossiers réels.
assert normaliser_montant("506,471.38")["valeur"] == 506471.38
assert normaliser_montant("506 471,38")["valeur"] == 506471.38
assert normaliser_montant("466300.88")["valeur"] == 466300.88
assert normaliser_montant("10,631,660.16")["valeur"] == 10631660.16
assert normaliser_montant("481.147,81")["valeur"] == 481147.81
assert normaliser_montant("ILLISIBLE")["statut"] == STATUT_ILLISIBLE
assert normaliser_montant(None)["statut"] == STATUT_ABSENT
print("✅ Montants : les trois conventions d'écriture sont lues correctement")

# --- 2. Dates strictes ------------------------------------------------
assert normaliser_date("13/10/2025")["iso"] == "2025-10-13"
assert normaliser_date("01/01/2026")["iso"] == "2026-01-01"
assert normaliser_date("12 août 2026")["iso"] == "2026-08-12"
# « 2026.1 » est une référence de domiciliation, pas une date.
# pd.to_datetime l'acceptait ; le parseur strict la rejette.
assert normaliser_date("2026.1")["statut"] == STATUT_ILLISIBLE
assert normaliser_date("31/02/2026")["statut"] == STATUT_ILLISIBLE
print("✅ Dates : parseur strict, sortie ISO, faux positifs rejetés")

# --- 3. Comptes et clé RIB -------------------------------------------
_compte = normaliser_compte("0270073101 1032700188")
assert _compte["valeur"] == "02700731011032700188"
assert _compte["cle_valide"] is True
assert normaliser_compte("02700731010835200137")["cle_valide"] is True
_faux = normaliser_compte("02700731011032700189")   # dernier chiffre modifié
assert _faux["cle_valide"] is False
print("✅ Comptes : clé de contrôle validée sur deux comptes réels, "
      "erreur d'un chiffre détectée")

# --- 4. Taux, durées, permis -----------------------------------------
assert normaliser_taux("95%")["valeur"] == 95.0
assert normaliser_taux("0,95")["valeur"] == 95.0
assert normaliser_duree_mois("2 ANS, 0 JOURS")["valeur"] == 24
assert normaliser_duree_mois("24 Mois")["valeur"] == 24
assert normaliser_permis("( R ) 21-00002974 / 31-25-001448")["valeur"] == \
       "21-00002974/31-25-001448"
print("✅ Taux, durées et numéros de permis normalisés")

# --- 5. Lecture JSON robuste -----------------------------------------
# Réponse tronquée par max_new_tokens : V7.2 renvoyait {} et déclenchait
# une escalade inutile.
_tronque = '{"DOM_SALAIRE_NET_MENSUEL": "506471,38", "DOM_PART_TRANSFERABLE": "481147'
_parse, _statut = parse_json_response(_tronque, tracer=True)
assert _parse.get("DOM_SALAIRE_NET_MENSUEL") == "506471,38"
assert _statut in ("REPARE_TRONQUE", "REPARE_VIRGULE")
# Deux objets dans la réponse : la regex gloutonne de V7.2 échouait.
_double = 'Voici : {"A": "1"} et aussi {"B": "2"}'
assert parse_json_response(_double) == {"A": "1", "B": "2"}
print("✅ Lecture JSON : troncature réparée, réponses multi-objets fusionnées")

# --- 6. Contrôles arithmétiques sur un dossier réel -------------------
_dossier_reel = {
    "DOM_COMPTE_LOCAL": "0270073101 1032700188",
    "DOM_DUREE_CONTRAT_MOIS": "24",
    "DOM_DATE_DEBUT_CONTRAT": "13/10/2025",
    "DOM_DATE_FIN_CONTRAT": "12/10/2027",
    "DOM_SALAIRE_NET_MENSUEL": "506471,38",
    "DOM_PART_TRANSFERABLE": "481 147.81",
    "DOM_TAUX_TRANSFERABLE": "95%",
    "CTR_SALAIRE_BRUT": "822 774,95",
    "CTR_SALAIRE_NET": "506 471,38",
    "CTS_SALAIRE_NET": "506,471.38",
    "CTS_PART_TRANSFERABLE": "481,147.81",
    "CTS_PART_PAYABLE_DZD": "25,323.57",
    "CTS_DATE_DEBUT_VALIDITE_PERMIS": "13/10/2025",
    "CTS_DATE_FIN_VALIDITE_PERMIS": "12/10/2027",
}
_champs = typer_champs(_dossier_reel)
_controles = controles_dossier(_champs)
_synthese = synthese_qualite(_controles, _champs)
assert _synthese["NB_ANOMALIES"] == 0, _synthese
print(f"✅ Dossier réel : {_synthese['NB_CONTROLES_OK']}/"
      f"{_synthese['NB_CONTROLES']} contrôles OK → {_synthese['QUALITE_DOSSIER']}")

# --- 7. Détection d'une erreur d'un seul chiffre ----------------------
# C'est le test qui justifie toute la couche de contrôles : une lecture
# erronée d'UN chiffre sur la part transférable est invisible à l'œil nu
# dans un Excel de 1 000 lignes, mais casse trois identités arithmétiques.
_errone = dict(_dossier_reel)
_errone["DOM_PART_TRANSFERABLE"] = "481 148.81"
_champs_err = typer_champs(_errone)
_ctrl_err = controles_dossier(_champs_err)
_codes = {c["CODE"] for c in _ctrl_err if c["STATUT"] == "ANOMALIE"}
assert "C01" in _codes, _codes
assert synthese_qualite(_ctrl_err, _champs_err)["QUALITE_DOSSIER"] == "A_CONTROLER"
print(f"✅ Erreur d'un chiffre détectée par les contrôles {sorted(_codes)}")
print("   Zones programmées pour relecture :", zones_a_relire(_ctrl_err))

# --- 8. Cohérence du schéma et des vues -------------------------------
for _type, _champs_type in CHAMPS_ATTENDUS.items():
    _hors_schema = [c for c in _champs_type if c not in SCHEMA_CHAMPS]
    assert not _hors_schema, f"{_type} : champs sans type déclaré {_hors_schema}"
for _champ in PRIORITE_VUES:
    assert _champ in SCHEMA_CHAMPS, f"{_champ} absent du schéma"
print("✅ Tous les champs extraits ont un type déclaré au schéma")

print("\n✅ Tests V7.3 réussis")


In [ ]:

ram_free = psutil.virtual_memory().available / 1_000_000_000
log(f"RAM libre : {ram_free:.1f} GB")
reference_df = build_reference_table()
log(f"Référentiel externe : {0 if reference_df is None else len(reference_df)} ligne(s)")

all_dossiers, errors = [], []
nb_repris = 0
nb_nouveaux = 0
pipeline_start = time.time()

print_pipeline_header(len(pdfs))

for position, pdf_path in enumerate(pdfs, 1):
    try:
        dossier = load_existing_checkpoint(pdf_path)
        skipped = dossier is not None

        if skipped:
            dossier = enrich_dossier_row_with_stats(
                dossier,
                "REPRIS_JSON_EXISTANT",
            )
            nb_repris += 1
        else:
            dossier = process_pdf(pdf_path, verbose=False)
            dossier = enrich_dossier_row_with_stats(
                dossier,
                "TRAITE_NOUVEAU",
            )
            nb_nouveaux += 1

        all_dossiers.append(dossier)

        stats = dossier.get("stats") or {}
        print_compact_progress(
            position=position,
            total=len(pdfs),
            pdf_name=pdf_path.name,
            pages=stats.get("pages", 0),
            skipped=skipped,
            tokens_in=stats.get("tokens_in", 0),
            tokens_out=stats.get("tokens_out", 0),
            elapsed_s=stats.get("elapsed_s", 0),
            pipeline_start=pipeline_start,
        )

    except Exception as exc:
        errors.append({
            "FICHIER": pdf_path.name,
            "ETAPE": "PROCESS_PDF",
            "ERREUR": repr(exc),
            "DATE": datetime.now().isoformat(timespec="seconds"),
        })

        print(
            f"[{position}/{len(pdfs)}] "
            f"{pdf_path.name} | ERREUR | {exc!r}",
            flush=True,
        )
        log(f"[{position}/{len(pdfs)}] ❌ {pdf_path.name}: {exc}")

    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

for dossier in all_dossiers:
    row = dossier.get("dossier_row") or {}
    row.update(match_dossier(row, reference_df))
    dossier["dossier_row"] = row

    # Mise à jour du JSON canonique avec le matching actuel,
    # sans relancer la classification ni l'extraction VLM.
    source_pdf = INPUT_DIR / dossier.get("source_file", "")
    if source_pdf.exists():
        canonical_checkpoint_path(source_pdf).write_text(
            json.dumps(dossier, ensure_ascii=False, indent=2, default=str),
            encoding="utf-8",
        )

create_excel(EXCEL_PATH, all_dossiers, errors, reference_df)

with open(MASTER_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump({
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "pipeline_version": PIPELINE_VERSION,
        "dossiers": all_dossiers, "errors": errors,
    }, f, ensure_ascii=False, indent=2, default=str)


elapsed_pipeline = time.time() - pipeline_start
total_tokens_in = sum(
    int((d.get("stats") or {}).get("tokens_in", 0) or 0)
    for d in all_dossiers
)
total_tokens_out = sum(
    int((d.get("stats") or {}).get("tokens_out", 0) or 0)
    for d in all_dossiers
)

print(
    f"\nTerminé | total={len(all_dossiers)} "
    f"| traités={nb_nouveaux} "
    f"| skip={nb_repris} "
    f"| erreurs={len(errors)} "
    f"| IN={total_tokens_in:,} "
    f"| OUT={total_tokens_out:,} "
    f"| durée={format_duration(elapsed_pipeline)}",
    flush=True,
)

log(f"✅ Pipeline terminé | dossiers={len(all_dossiers)} | nouveaux={nb_nouveaux} | repris={nb_repris} | erreurs={len(errors)}")


## 14. Lecture des résultats

### Fichiers produits

- `domiciliations_master.json` : référentiel consolidé, liens de
  renouvellement et planning.
- Un JSON par dossier dans `json_dossiers/`, avec quatre blocs V7.3 :
  - `champs_normalises` : pour chaque champ, `valeur` typée, `raw` lue,
    `type` et `statut`. C'est la sortie de référence pour toute reprise
    automatisée — elle ne demande aucune réinterprétation de format.
  - `consolidation` : pour chaque donnée présente sur plusieurs
    documents, la valeur retenue, toutes les sources et le verdict de
    concordance.
  - `controles` : un enregistrement par contrôle, avec valeur attendue,
    valeur obtenue et écart chiffré.
  - `qualite` : la synthèse du dossier.

### Onglets Excel

- `DOSSIERS_DOMICILIATION` : une ligne par dossier. Chaque champ typé est
  accompagné de sa colonne `_RAW` et de sa colonne `_STATUT`.
- `CONTROLES` : tous les contrôles, tous statuts confondus.
- `ANOMALIES` : le sous-ensemble à traiter — c'est l'onglet de travail
  du contrôleur.
- `RELECTURES_CIBLEES` : valeur initiale et valeur relue, côte à côte.
- `PLANNING_TL` : une ligne par période de transfert (mois partiels en
  `P1` / `P2`).
- `PAGES_DOCUMENTS`, `EXTRACTION_BRUTE`, `MATCHING_DOM`,
  `SUIVI_TRAITEMENT`, `ERREURS` : inchangés.

### Comment lire la colonne `QUALITE_DOSSIER`

| Valeur | Signification | Action |
|---|---|---|
| `CONFORME` | tous les contrôles passent | traitement automatique |
| `A_VERIFIER` | anomalies non bloquantes seulement | revue rapide |
| `A_CONTROLER` | au moins une anomalie bloquante | revue obligatoire avant transfert |

`CODES_ANOMALIES` indique quels contrôles ont échoué : un rapprochement
direct avec l'onglet `ANOMALIES` donne la valeur attendue et l'écart.


In [ ]:

if EXCEL_PATH.exists():
    for sheet in ["DOSSIERS_DOMICILIATION", "PLANNING_TL", "MATCHING_DOM"]:
        df = pd.read_excel(EXCEL_PATH, sheet_name=sheet)
        print(f"{sheet}: {len(df)} ligne(s)")
        display(df.head(10))
else:
    print("Le fichier Excel n'a pas encore été généré.")
